# ARC-AGI-2: Qwen3.8-27B-FP8 + Python tools, offline on 4xL4 (v3)

Generated by `kaggle/build_notebook.py --variant v3` from the team repo (the repo is the source of truth; do not
edit this notebook by hand). Experiment notes: `experiments/E021_kaggle_v3/README.md`.

**Same machinery as v2**: mode switch (competition rerun -> hidden test set; otherwise a self-scored local test set
and a commit gate), Qwen3.8-27B-FP8 (hash-verified mirror) on vLLM 0.23.0 from our own offline wheel source, TP=4,
FP8 KV, prefix caching; the tool agent (`agent_qwen.solve`, the same modules as the evals in `evals/arc_harness/`)
with a local child-process executor; deadline forcing; force 11h00m, hard stop 11h30m, emergency writer 11h42m,
guard 11h45m; the last cell validates and repairs `submission.json`.

**v3 options** (section 1, `SETTINGS`; documented in `kaggle/README.md`):
- `ARC_FREE_ATTEMPT2`: attempt_2 from other test grids of the same attempt (failing programs, the session's
  `solve` / `answer`, salvage), ranked by train check (`arc_harness/selection.py`). No model tokens.
- `ARC_ATTEMPT2=gated`: with leftover capacity before the force deadline, a second attempt only for pairs whose
  first answer fails the train check or that have no answer, highest expected value first; never before every
  first attempt has started.
- `ARC_TURN_THINK_CAP` / `ARC_TURN_THINK_GRACE`: per-turn thinking cap with a "test it with code now" nudge.
- `ARC_TOOLS_ANSWER_MODE=program`, `ARC_TASK_SUMMARY`, `ARC_HELPERS`, `ARC_TOOLS_PROMPT`, `ARC_FEEDBACK`.
- Robustness: per-session memory cap (`ARC_SESSION_MEM_MB`, RLIMIT_AS = VmSize + 2 GB; programs 2 GB) and
  concurrency = KV tokens / p90 context (`ARC_KV_PER_SEQ` 48000 -> 29 on vLLM 0.23's 1.40M KV).


## 1. Configuration

The wall clock for every deadline starts here (`ARC_T0`).

In [ ]:
import os, time
os.environ['ARC_T0'] = str(time.time())  # deadline clock starts at the first cell
os.environ['ARC_MODE'] = 'submit' if os.environ.get('KAGGLE_IS_COMPETITION_RERUN') else 'local'
SETTINGS = {
 "ARC_SERVER": "vllm",
 "ARC_MTP": 0,
 "ARC_KV_PER_SEQ": 48000,
 "ARC_RATE_PRIOR": 220,
 "ARC_LOCAL_TASKS": "7b0280bc,db0c5428,d35bdbdc,13e47133,e376de54,7b5033c1,8f215267,2d0172a1,3e6067c3,35ab12c3,64efde09,898e7135,31f7f899,88e364bc,9385bd28,271d71e2,a6f40cea,3dc255db,136b0064,97d7923e,195c6913,de809cff,58f5dbd5,a47bf94d,cbebaa4b,5961cc34,62593bfd,eee78d87,a251c730,4c7dc4dd,20270e3b,332f06d7,65b59efc,6e453dd6,db695cfb",
 "ARC_LOCAL_READY_MIN": 120,
 "ARC_LOCAL_HARD_MIN": 15,
 "ARC_TOOLS_TIER": "xhigh",
 "ARC_TOOLS_CAP": 48000,
 "ARC_TOOLS_SOFT": 24000,
 "ARC_TOOLS_ANSWER_MODE": "program",
 "ARC_TOOLS_PROMPT": "lean2progroles",
 "ARC_TASK_SUMMARY": 1,
 "ARC_HELPERS": 1,
 "ARC_FEEDBACK": "rich",
 "ARC_TURN_THINK_CAP": 8192,
 "ARC_TURN_THINK_GRACE": 2048,
 "ARC_FREE_ATTEMPT2": 1,
 "ARC_ATTEMPT2": "gated",
 "ARC_ATTEMPT2_UNKNOWN": 0,
 "ARC_SESSION_MEM_MB": 2048,
 "ARC_THINK_CALL_RESCUE": 1,
 "ARC_LOCAL_SOURCE": "eval"
}
for k, v in SETTINGS.items():
    os.environ.setdefault(k, str(v))
os.makedirs('/kaggle/working/arc_solver/evals/arc_harness', exist_ok=True)
os.makedirs('/kaggle/working/arc_solver/notes/prompts', exist_ok=True)
print('mode', os.environ['ARC_MODE'], {k: v for k, v in os.environ.items() if k.startswith('ARC_')})

## 2. Modules

The E012 tool harness (`evals/arc_harness/`), the ARC Prize prompt, and the solver (server install/launch, scheduler, deadlines, submission writer) are written to `/kaggle/working/arc_solver/` with their repo layout.

In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/__init__.py
"""ARC-AGI-2 agent harness: prompts, model access, grid checks (more layers to come; see notes/specs)."""


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/grids.py
"""ARC grid validity: the single definition used for answers, submissions and scoring."""


def valid_grid(grid):
    """A rectangular list of 1-30 rows of 1-30 Python ints in 0-9."""
    return (isinstance(grid, list) and 1 <= len(grid) <= 30
            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)
            and len({len(row) for row in grid}) == 1
            and all(type(value) is int and 0 <= value <= 9 for row in grid for value in row))


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/arcprize.py
"""ARC Prize public-eval prompt and answer parser (github.com/arcprize/arc-agi-benchmarking), verbatim."""
import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
TASKS = ROOT / "data/arc-agi-2-github/data/evaluation"
TEMPLATE = (ROOT / "notes/prompts/arcprize_official_system_prompt.txt").read_text()


def build_prompt(train, test_input):
    ex = ""
    for i, pair in enumerate(train):
        ex += f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(pair["input"]) + "\n\n"
        ex += "OUTPUT: \n\n" + json.dumps(pair["output"]) + "\n\n"
    return TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))


def is_grid(x):
    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)


def extract_from_boxed(s):
    m = re.search(r"\\boxed\{(.*?)\}", s, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            return g if is_grid(g) else None
        except json.JSONDecodeError:
            return None
    return None


def backscan_json_parser(s):
    end = max(s.rfind("]"), s.rfind("}"))
    if end < 0:
        return None
    close = s[end]
    open_ = "[" if close == "]" else "{"
    depth = 1
    for i in range(end - 1, -1, -1):
        depth += (s[i] == close) - (s[i] == open_)
        if depth == 0:
            try:
                g = json.loads(s[i:end + 1])
                return g if is_grid(g) else None
            except json.JSONDecodeError:
                return None
    return None


def parse(s):
    return extract_from_boxed(s) or backscan_json_parser(s or "")



In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/prompts.py
"""Everything the model reads: the opening message, tool descriptions and fixed instructions.

Kept byte-identical to harness persistent-v3 so results stay comparable across refactors.
"""
from .arcprize import build_prompt

# Optional suffixes appended to the ARC Prize prompt (--prompt). "plain" keeps the original prompt.
PROMPT_SUFFIXES = {
    "plain": "",
    "lean": ("\n\nYou have a Python tool. Work efficiently: inspect the grids with code instead of reading them "
             "by hand, form a hypothesis, and write code that computes the output. Never construct, copy or count "
             "grids by hand. Check your rule on every training pair with code before answering, and if a pair "
             "fails, fix the rule rather than explaining the mismatch away. Keep your reasoning brief."),
    "lean2": ("\n\nYou have a Python tool. The variables `train` (list of {'input', 'output'} grids) and "
              "`test_input` are already loaded in it: use them directly and never retype or reassign them. "
              "Work efficiently: inspect the grids with code instead of reading them by hand, form a hypothesis, "
              "and write code that computes the output. Never construct, copy or count grids by hand. Check your "
              "rule on every training pair with code before answering, and if a pair fails, fix the rule rather "
              "than explaining the mismatch away. Before answering, check that the rule's assumptions also hold "
              "on test_input. Keep your reasoning brief."),
}
CODE_MODE_SUFFIX = "\n\nUse run_python to investigate; submit self-contained transform(grid) code with submit_solution."
FINALIZE = "Finalize now. Return only your best test output as a JSON grid."
MODES = ("grid", "code")


def initial_messages(train, test_input, prompt, mode):
    content = build_prompt(train, test_input) + PROMPT_SUFFIXES[prompt]
    if mode == "code":
        content += CODE_MODE_SUFFIX
    return [{"role": "user", "content": content}]


def tool_specs(mode, cell_timeout):
    descriptions = {"run_python": (
        "Execute a Python cell in your persistent IPython session. Variables/functions persist. "
        "Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. "
        f"scipy is available. No internet or host files. Timeout {cell_timeout}s. "
        "Use print() to inspect results. Check transformations on all training pairs."
    )}
    if mode == "code":
        descriptions["submit_solution"] = (
            "Submit self-contained Python code defining transform(grid), which returns a rectangular "
            "grid of ints 0-9 (or numpy array). Include imports/helpers; execution uses a fresh namespace. "
            "The harness checks every training example. Failures return feedback so you can revise. "
            "If all pass, it applies transform to test_input and ends this attempt."
        )
    return [{"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}},
                       "required": ["code"], "additionalProperties": False},
    }} for name, description in descriptions.items()]


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/executor.py
"""Code-execution backends for the tool harness, behind one small interface.

An executor is a persistent Python session for one attempt:

    with make_executor(train, test_input) as ex:     # train: [{'input', 'output'}], test_input: grid
        result = ex.execute(code)                    # {'process_status': completed|error|timeout,
                                                     #  'stdout': str, 'stderr': str, ['state_reset': True]}

It preloads `train`, `test_input`, `np` and `json`, keeps variables between cells, and shows the value of a
cell's last expression (IPython-style). Attributes used by the answer helpers: `train`, `test_input`, `max_output`.

Backends:
- `SandboxSession` (evals/sandbox_client.py): the Docker NeMo-Skills sandbox (no network, no host mounts).
- `LocalExecutor` (here): a child Python process per attempt, for Kaggle where Docker is unavailable. It is not a
  security boundary (same user, same filesystem). It adds limits (address space, cumulative CPU, file size, nice,
  oom_score_adj=1000), a per-cell timeout (SIGINT to the process group, then kill and restart) and output truncation.

`salvage_answer` (qwen-tools-3) recovers an answer from the session's own functions when the model left none.
`submit_answer` / `check_answer_on_train` / `tool_output` are kept identical to evals/run_deepseek_tools.py and
evals/sandbox_client.py (copied, so this package runs without those scripts).
"""
import json
import os
import select
import signal
import subprocess
import sys
import time
from uuid import uuid4

from .grids import valid_grid

# ---------------------------------------------------------------- local executor

_CHILD = r'''
import ast, contextlib, io, json, os, resource, signal, sys, traceback
resp = os.fdopen(int(sys.argv[1]), "w")
limit = int(sys.argv[2])
mem, cpu, fsize, nice = (int(x) for x in sys.argv[3:7])
# Self-imposed limits (no preexec_fn: the parent spawns from many threads). The child leads its own session.
if mem:
    resource.setrlimit(resource.RLIMIT_AS, (mem, mem))
if cpu:
    resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu + 5))
if fsize:
    resource.setrlimit(resource.RLIMIT_FSIZE, (fsize, fsize))
    signal.signal(signal.SIGXFSZ, signal.SIG_IGN)  # oversized writes fail with OSError instead of killing us
if nice:
    os.nice(nice)
try:
    with open("/proc/self/oom_score_adj", "w") as f:
        f.write("1000")  # under host memory pressure, kill tool code before the model server
except OSError:
    pass
ns = {"__name__": "__main__"}

def run(code):
    tree = ast.parse(code, "<cell>", "exec")
    last = None
    if tree.body and isinstance(tree.body[-1], ast.Expr):
        last = ast.Expression(tree.body.pop().value)
    exec(compile(tree, "<cell>", "exec"), ns)
    if last is not None:
        value = eval(compile(last, "<cell>", "eval"), ns)
        if value is not None:
            print(repr(value))

def plain_traceback(exc):
    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename != "<string>"]
    text = "".join(traceback.format_list(frames)) if frames else ""
    return "Traceback (most recent call last):\n" + text + "".join(traceback.format_exception_only(type(exc), exc))

for line in sys.stdin:
    req = json.loads(line)
    out, err = io.StringIO(), io.StringIO()
    status = "completed"
    try:
        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
            run(req["code"])
    except KeyboardInterrupt:
        status = "timeout"
        err.write("KeyboardInterrupt: cell timed out (state kept)\n")
    except BaseException as exc:
        status = "error"
        try:
            err.write(plain_traceback(exc))
        except BaseException:
            err.write(f"{type(exc).__name__}: {exc}\n")
    o, e = out.getvalue(), err.getvalue()
    if len(o) > limit:
        o = o[:limit] + "\n[output truncated]"
    if len(e) > limit:
        e = e[:limit] + "\n[output truncated]"
    resp.write(json.dumps({"process_status": status, "stdout": o, "stderr": e}) + "\n")
    resp.flush()
'''


class LocalExecutor:
    """Persistent Python in a child process. On timeout the cell gets SIGINT (state kept, like the patched
    sandbox); if it does not stop within `grace` seconds the process is killed and restarted with only the
    task data (`state_reset`)."""

    def __init__(self, train, test_input, timeout=30, max_output=16000, memory_mb=4096, grace=5,
                 python=sys.executable, cpu_seconds=1200, fsize_mb=64, nice=10, start_timeout=90, prelude_extra=""):
        self.train, self.test_input = train, test_input
        self.timeout, self.max_output, self.memory_mb, self.grace = timeout, max_output, memory_mb, grace
        self.python, self.cpu_seconds, self.fsize_mb, self.nice = python, cpu_seconds, fsize_mb, nice
        self.start_timeout = start_timeout  # the prelude (numpy import) may be slow on a busy 4-CPU box
        self.proc = self.resp = None
        self.closed = False
        data = json.dumps({"train": train, "test_input": test_input})
        self.prelude = (
            "import json, numpy as np\n"
            f"_arc_data = json.loads({data!r})\n"
            "train, test_input = _arc_data['train'], _arc_data['test_input']\n"
            "del _arc_data\n"
        ) + prelude_extra  # E020 --helpers: re-run with the task data after a restart

    def _spawn_once(self, timeout):
        r, w = os.pipe()
        self.proc = subprocess.Popen(
            [self.python, "-I", "-u", "-c", _CHILD, str(w), str(self.max_output), str(self.memory_mb * 2 ** 20),
             str(int(self.cpu_seconds)), str(self.fsize_mb * 2 ** 20), str(self.nice)],
            stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, pass_fds=(w,),
            start_new_session=True, text=True, cwd="/tmp",
            env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
                 "MKL_NUM_THREADS": "1"})
        os.close(w)
        self.resp, self.buf = r, b""
        return self._run(self.prelude, timeout)

    def _spawn(self):
        """Start the child and load the task data; one retry with a longer timeout, else RuntimeError."""
        for timeout in (max(self.timeout, 30), self.start_timeout):
            result = self._spawn_once(timeout)
            if result is not None and result["process_status"] == "completed":
                return
            self._kill()
        detail = "" if result is None else result.get("stderr", "")[:500]
        raise RuntimeError(f"Local executor initialization failed {detail}".strip())

    def _read(self, deadline):
        while b"\n" not in self.buf:
            wait = deadline - time.monotonic()
            if wait <= 0:
                return None
            ready, _, _ = select.select([self.resp], [], [], wait)
            if ready:
                chunk = os.read(self.resp, 1 << 16)
                if not chunk:
                    return "eof"
                self.buf += chunk
        line, self.buf = self.buf.split(b"\n", 1)
        return json.loads(line)

    def _signal_group(self, sig):
        try:
            os.killpg(self.proc.pid, sig)  # the child leads its own session: this also reaches grandchildren
        except (ProcessLookupError, PermissionError, OSError):
            try:
                self.proc.send_signal(sig)
            except (ProcessLookupError, OSError):
                pass

    def _kill(self):
        if self.proc is not None:
            self._signal_group(signal.SIGKILL)
            try:
                self.proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            try:
                self.proc.stdin.close()
            except Exception:
                pass
        if self.resp is not None:
            os.close(self.resp)
        self.proc = self.resp = None

    def _run(self, code, timeout=None):
        """Run one cell; returns the result, or None if the process died or had to be killed."""
        self.last_failure = "error"
        try:
            self.proc.stdin.write(json.dumps({"code": code}) + "\n")
            self.proc.stdin.flush()
        except (BrokenPipeError, OSError):
            return None
        result = self._read(time.monotonic() + (timeout or self.timeout))
        if result is None:  # timed out: interrupt, keep state if the cell stops
            self._signal_group(signal.SIGINT)
            self.last_failure = "timeout"
            result = self._read(time.monotonic() + self.grace)
            if isinstance(result, dict):
                result["process_status"] = "timeout"
                return result
            return None
        return result if isinstance(result, dict) else None

    def initialize(self):
        self._spawn()

    def execute(self, code):
        if self.closed:
            raise RuntimeError("Executor is closed")
        if self.proc is None:
            self._spawn()
        result = self._run(code)
        if result is None:
            # Died (memory or CPU limit, os._exit, segfault) or ignored SIGINT: restart with task data only.
            self._kill()
            try:
                self._spawn()
            except RuntimeError as exc:  # leave proc None: the next execute tries to start again
                return {"process_status": "error", "stdout": "", "state_reset": True,
                        "stderr": f"Python process died and could not be restarted: {exc}"}
            result = {"process_status": self.last_failure, "stdout": "", "state_reset": True,
                      "stderr": ("Cell did not stop after the timeout; process restarted." if self.last_failure == "timeout"
                                 else "Python process died (crash, exit or memory limit); process restarted.")}
        return result

    def close(self):
        if not self.closed:
            self.closed = True
            self._kill()

    def __enter__(self):
        try:
            self.initialize()
        except BaseException:
            self.close()
            raise
        return self

    def __exit__(self, *exc):
        self.close()


def local_factory(timeout=30, max_output=16000, memory_mb=4096, **kwargs):
    """kwargs: grace, cpu_seconds, fsize_mb, nice, start_timeout, python (see LocalExecutor)."""
    return lambda train, test_input: LocalExecutor(train, test_input, timeout, max_output, memory_mb, **kwargs)


# E021 (Kaggle v3): the evals/run_qwen_tools.py --session-mem-mb prelude (same text), importable without the
# runner. Run first in a session (prelude_extra): caps the kernel's address space at its current VmSize + {mb} MB,
# so a runaway cell raises MemoryError instead of exhausting host memory.
SESSION_MEM_CAP = """
def _arc_memcap():
    import resource
    try:
        with open('/proc/self/status') as f:
            vm = next(int(l.split()[1]) * 1024 for l in f if l.startswith('VmSize:'))
        cap = vm + {mb} * 2 ** 20
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        if hard == resource.RLIM_INFINITY or cap < hard:
            resource.setrlimit(resource.RLIMIT_AS, (cap, hard if hard != resource.RLIM_INFINITY else cap))
    except Exception:
        pass
_arc_memcap()
del _arc_memcap
"""


def sandbox_factory(url, timeout=30, max_output=16000, prelude_extra=""):
    """Docker sandbox sessions (evals/sandbox_client.py must be importable, i.e. evals/ on sys.path)."""
    from sandbox_client import SandboxSession
    if prelude_extra:
        return lambda train, test_input: SandboxSession(url, train, test_input, timeout, max_output,
                                                        prelude_extra=prelude_extra)
    return lambda train, test_input: SandboxSession(url, train, test_input, timeout, max_output)


# ---------------------------------------------------------------- helpers shared with run_deepseek_tools.py

def tool_output(result, limit=16000):
    text = result.get("stdout", "")
    if result.get("stderr"):
        text += "\n[stderr]\n" + result["stderr"]
    if result.get("state_reset"):
        text = "[Python state lost. train/test_input/np restored; redefine other variables.]\n" + text
    text = f"[{result['process_status']}]\n" + (text or "[no output; use print()]")
    return text if len(text) <= limit else text[:limit] + "\n[output truncated]"


def submit_answer(session, code):
    """Evaluate the model's answer expression in its own session and return (feedback, grid|None).

    `code` is an expression (e.g. solve(test_input)) or statements that set `answer`."""
    marker = f"ARC_ANSWER_{uuid4().hex}:"
    cell = f'''
def _arc_answer():
    import contextlib, io, json
    _src = {code!r}
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            _expr = compile(_src, '<answer>', 'eval')
        except SyntaxError:
            _expr = None
        if _expr is not None:
            value = eval(_expr, globals())
        else:
            exec(compile(_src, '<answer>', 'exec'), globals())
            if 'answer' not in globals():
                raise NameError('code did not define answer')
            value = globals()['answer']
        if hasattr(value, 'tolist'):
            value = value.tolist()
        text = json.dumps(value, default=int)
    print({marker!r} + text)
try:
    _arc_answer()
finally:
    del _arc_answer
'''
    result = session.execute(cell)
    grid = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    grid = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
        else:
            return "[answer not accepted] no value returned", None
    else:
        return tool_output(result, session.max_output)[:2000] + "\n[answer not accepted]", None
    if not valid_grid(grid):
        shape = (f"{len(grid)} rows, row lengths {sorted({len(r) for r in grid if isinstance(r, list)})}"
                 if isinstance(grid, list) else type(grid).__name__)
        return ("[answer not accepted] must be a rectangular 1-30 by 1-30 grid of ints 0-9; got " + shape), None
    return f"[answer accepted: {len(grid)}x{len(grid[0])} grid]", grid


def check_answer_on_train(session, code):
    """Re-evaluate the submitted expression with `test_input` bound to each training input and compare with the
    host's training outputs. None when there is nothing to check, else (failed_indices, feedback_text)."""
    try:
        expr = compile(code, "<answer>", "eval")
    except SyntaxError:
        return None
    if "test_input" not in expr.co_names:
        return None
    marker = f"ARC_CHECK_{uuid4().hex}:"
    inputs = [pair["input"] for pair in session.train]
    cell = f'''
def _arc_check():
    import contextlib, io, json
    _expr = compile({code!r}, '<answer>', 'eval')
    outs = []
    with contextlib.redirect_stdout(io.StringIO()):
        for _inp in json.loads({json.dumps(inputs)!r}):
            try:
                v = eval(_expr, {{**globals(), 'test_input': _inp}})
                if hasattr(v, 'tolist'):
                    v = v.tolist()
                outs.append(json.loads(json.dumps(v, default=int)))
            except Exception as exc:
                outs.append({{'error': type(exc).__name__ + ': ' + str(exc)[:200]}})
    print({marker!r} + json.dumps(outs))
try:
    _arc_check()
finally:
    del _arc_check
'''
    result = session.execute(cell)
    outs = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    outs = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
    if not isinstance(outs, list) or len(outs) != len(inputs):
        return None
    failed, notes = [], []
    for i, (o, pair) in enumerate(zip(outs, session.train)):
        if o == pair["output"]:
            continue
        failed.append(i)
        if isinstance(o, dict) and "error" in o:
            notes.append(f"pair {i}: {o['error']}")
        elif valid_grid(o):
            exp = pair["output"]
            if (len(o), len(o[0])) != (len(exp), len(exp[0])):
                notes.append(f"pair {i}: shape {len(o)}x{len(o[0])}, expected {len(exp)}x{len(exp[0])}")
            else:
                notes.append(f"pair {i}: {sum(a != b for ra, rb in zip(o, exp) for a, b in zip(ra, rb))} cells differ")
        else:
            notes.append(f"pair {i}: not a valid grid")
    return failed, "; ".join(notes)


def salvage_answer(session, max_functions=8):
    """No answer was submitted or set: score the session's own one-argument functions on the training inputs
    (host-side comparison) and apply the best one to test_input. Returns (info, grid|None); info carries the
    function name and how many training pairs it reproduced. Costs no model tokens."""
    marker = f"ARC_SALVAGE_{uuid4().hex}:"
    inputs = [pair["input"] for pair in session.train] + [session.test_input]
    cell = f'''
def _arc_salvage():
    import contextlib, copy, inspect, io, json
    import numpy as _np
    fns = []
    for _name, _f in list(globals().items())[::-1]:
        if _name.startswith('_') or not inspect.isfunction(_f) or _f.__module__ != '__main__':
            continue
        try:
            _req = [p for p in inspect.signature(_f).parameters.values()
                    if p.default is p.empty and p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)]
        except (TypeError, ValueError):
            continue
        if len(_req) == 1:
            fns.append((_name, _f))
        if len(fns) >= {max_functions}:
            break
    res = {{}}
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        for _name, _f in fns:
            outs = []
            for _inp in json.loads({json.dumps(inputs)!r}):
                v = None
                for arg in (copy.deepcopy(_inp), _np.array(_inp)):
                    try:
                        v = _f(arg)
                        if hasattr(v, 'tolist'):
                            v = v.tolist()
                        v = json.loads(json.dumps(v, default=int))
                        break
                    except Exception:
                        v = None
                outs.append(v)
            res[_name] = outs
    print({marker!r} + json.dumps(res))
try:
    _arc_salvage()
finally:
    del _arc_salvage
'''
    result = session.execute(cell)
    found = None
    if result["process_status"] == "completed":
        for line in reversed(result.get("stdout", "").splitlines()):
            if line.startswith(marker):
                try:
                    found = json.loads(line[len(marker):])
                except json.JSONDecodeError:
                    pass
                break
    if not isinstance(found, dict):
        return {"status": result["process_status"]}, None
    best = None  # (train pairs matched, newest first) -> name, grid
    for rank, (name, outs) in enumerate(found.items()):
        if not isinstance(outs, list) or len(outs) != len(inputs) or not valid_grid(outs[-1]):
            continue
        score = sum(o == pair["output"] for o, pair in zip(outs, session.train))
        if best is None or score > best[0]:
            best = (score, name, outs[-1])
    if best is None:
        return {"functions": list(found)}, None
    return {"function": best[1], "train_matched": best[0], "train_pairs": len(session.train)}, best[2]


# ---------------------------------------------------------------- program answers (qwen-tools-4, answer_mode=program)

_PROGRAM_CHILD = r'''
import contextlib, io, json, linecache, resource, signal, sys, traceback
req = json.loads(sys.stdin.read())
code, timeout = req["code"], req["timeout"]
try:
    if req.get("mem"):
        resource.setrlimit(resource.RLIMIT_AS, (req["mem"], req["mem"]))
except (ValueError, OSError):
    pass

class _ProgramTimeout(BaseException):
    pass

def _alarm(*_):
    raise _ProgramTimeout()

def _plain(o):
    if hasattr(o, "tolist"):
        return o.tolist()
    return int(o)

linecache.cache["<program>"] = (len(code), None, code.splitlines(True), "<program>")
signal.signal(signal.SIGALRM, _alarm)
buf = io.StringIO()
try:
    signal.setitimer(signal.ITIMER_REAL, timeout)
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        import numpy as np
        ns = {"__name__": "__arc_program__", "np": np}
        if req.get("prelude"):
            exec(compile(req["prelude"], "<helpers>", "exec"), ns)
        exec(compile(code, "<program>", "exec"), ns)
        fn = ns.get("solve")
        if not callable(fn):
            raise NameError("the program does not define solve(grid)")
        value = fn(req["grid"])
        value = json.loads(json.dumps(value, default=_plain))
    res = {"out": value}
except _ProgramTimeout:
    res = {"error": f"timeout: more than {timeout:g}s", "kind": "timeout"}
except BaseException as exc:
    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename == "<program>"]
    lines = [f"  line {f.lineno}, in {f.name}: {(f.line or '').strip()[:160]}" for f in frames[-2:]]
    msg = "".join(traceback.format_exception_only(type(exc), exc)).strip()[-400:]
    res = {"error": "\n".join(lines + [msg]), "kind": type(exc).__name__}
finally:
    signal.setitimer(signal.ITIMER_REAL, 0)
sys.stdout.write(req["marker"] + json.dumps(res) + "\n")
'''


def _marker_json(result, marker):
    if result.get("process_status") != "completed":
        return None
    for line in reversed(result.get("stdout", "").splitlines()):
        if line.startswith(marker):
            try:
                return json.loads(line[len(marker):])
            except json.JSONDecodeError:
                return None
    return None


def run_program(session, code, grid, timeout=10, memory_mb=3072, prelude=""):
    """Run self-contained `code` (which must define solve(grid)) on one grid in a FRESH Python process started
    from the session's environment (sandbox or LocalExecutor child), so session state cannot leak in.
    Returns {'out': value} or {'error': text, 'kind': name}; per-grid wall-clock limit `timeout` seconds.
    `prelude` (E020 --helpers) is executed in the program's namespace first (e.g. the arc_helpers source)."""
    marker = f"ARC_PROGRAM_{uuid4().hex}:"
    body = {"code": code, "grid": grid, "timeout": timeout, "marker": marker, "mem": memory_mb * 2 ** 20}
    if prelude:
        body["prelude"] = prelude
    req = json.dumps(body)
    cell = f'''
def _arc_program():
    import os, shutil, subprocess, sys
    exe = sys.executable  # the Docker sandbox runs cells under uwsgi: use the python on PATH there
    if not os.path.basename(exe or "").startswith("python"):
        exe = shutil.which("python3") or shutil.which("python") or exe
    try:
        r = subprocess.run([exe, "-I", "-c", {_PROGRAM_CHILD!r}], input={req!r}, capture_output=True,
                           text=True, timeout={timeout + 10!r}, cwd="/tmp")
        out = r.stdout
    except subprocess.TimeoutExpired:
        out = ""
    for line in out.splitlines()[::-1]:
        if line.startswith({marker!r}):
            print(line)
            return
    print({marker!r} + '{{"error": "timeout or crash (no result; memory limit {memory_mb} MB)", "kind": "crash"}}')
try:
    _arc_program()
finally:
    del _arc_program
'''
    res = _marker_json(session.execute(cell), marker)
    return res if isinstance(res, dict) else {"error": "program runner failed", "kind": "runner"}


def run_session_solve(session, grid, timeout=10):
    """Apply the session's own `solve` to one grid (fallback when no submitted program gives a test output).
    Runs inside the session (it may use session helpers). Returns {'out': value} or {'error': text}; None if the
    session has no callable `solve`."""
    marker = f"ARC_SSOLVE_{uuid4().hex}:"
    cell = f'''
def _arc_ssolve():
    import contextlib, copy, io, json, signal
    if not callable(globals().get("solve")):
        print({marker!r} + "null")
        return
    class _T(BaseException):
        pass
    def _h(*_):
        raise _T()
    def _plain(o):
        return o.tolist() if hasattr(o, "tolist") else int(o)
    old = signal.signal(signal.SIGALRM, _h)
    try:
        signal.setitimer(signal.ITIMER_REAL, {timeout!r})
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            v = globals()["solve"](copy.deepcopy(json.loads({json.dumps(grid)!r})))
            v = json.loads(json.dumps(v, default=_plain))
        res = {{"out": v}}
    except _T:
        res = {{"error": "timeout"}}
    except Exception as exc:
        res = {{"error": type(exc).__name__ + ": " + str(exc)[:200]}}
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)
    print({marker!r} + json.dumps(res))
try:
    _arc_ssolve()
finally:
    del _arc_ssolve
'''
    result = session.execute(cell)
    if result.get("process_status") != "completed":
        return {"error": "session solve did not complete"}
    for line in reversed(result.get("stdout", "").splitlines()):
        if line.startswith(marker):
            try:
                return json.loads(line[len(marker):])
            except json.JSONDecodeError:
                break
    return {"error": "no result"}


def grid_shape(g):
    if isinstance(g, list) and g and all(isinstance(r, list) for r in g):
        widths = sorted({len(r) for r in g})
        return f"{len(g)}x{widths[0]}" if len(widths) == 1 else f"{len(g)} rows of widths {widths[:5]}"
    return type(g).__name__


def score_outputs(outs, train):
    """Per training pair: {'ok', 'error'?, 'shape'?, 'wrong'?} comparing program outputs with the host's outputs."""
    rows = []
    for o, pair in zip(outs, train):
        exp = pair["output"]
        if "error" in o:
            rows.append({"ok": False, "error": o["error"], "kind": o.get("kind")})
            continue
        got = o["out"]
        if got == exp:
            rows.append({"ok": True})
        elif not valid_grid(got):
            rows.append({"ok": False, "error": f"not a valid grid ({grid_shape(got)}; need a rectangular 1-30 x "
                                               "1-30 grid of ints 0-9)", "kind": "invalid"})
        elif (len(got), len(got[0])) != (len(exp), len(exp[0])):
            rows.append({"ok": False, "shape": (grid_shape(got), grid_shape(exp)), "got": got})
        else:
            rows.append({"ok": False, "wrong": [(r, c, exp[r][c], got[r][c]) for r in range(len(exp))
                                                for c in range(len(exp[0])) if exp[r][c] != got[r][c]]})
    return rows


def _digits(g):
    return "\n".join("".join(str(v) for v in row) for row in g)


def rich_detail(row, pair, max_side=20):
    """E020 --feedback rich: extra lines for one failing pair (needs the host's training output)."""
    exp, lines = pair["output"], []
    if "wrong" in row:
        conf = {}
        for _, _, e, g in row["wrong"]:
            conf[(e, g)] = conf.get((e, g), 0) + 1
        top = sorted(conf.items(), key=lambda kv: -kv[1])[:8]
        lines.append("colour confusions (expected->got x count): " + ", ".join(f"{e}->{g} x{n}" for (e, g), n in top))
        if len(exp) <= max_side and len(exp[0]) <= max_side:
            bad = {(r, c): g for r, c, _, g in row["wrong"]}
            got = "\n".join("".join(str(bad[(r, c)]) if (r, c) in bad else "." for c in range(len(exp[0])))
                            for r in range(len(exp)))
            lines.append(f"expected output:\n{_digits(exp)}\nyour output ('.' = correct cell, digit = your wrong "
                         f"value):\n{got}")
    elif "shape" in row and row.get("got") is not None:
        got = row["got"]
        h, w = len(exp), len(exp[0])
        gh, gw = len(got), len(got[0])
        hint = []
        if (gh, gw) == (w, h):
            hint.append("your output is transposed in shape")
        for name, a, b in (("height", gh, h), ("width", gw, w)):
            if a != b and b and a % b == 0:
                hint.append(f"your {name} is {a // b}x the expected")
            elif a != b and a and b % a == 0:
                hint.append(f"the expected {name} is {b // a}x yours")
        if hint:
            lines.append("shape hint: " + "; ".join(hint))
        if max(h, w, gh, gw) <= 12:
            lines.append(f"expected output:\n{_digits(exp)}\nyour output:\n{_digits(got)}")
    return lines


def program_feedback(rows, k, cap, max_cells=12, test_error=None, rich=False, train=None):
    """Short tool result for a rejected program: which pairs fail and why, a capped diff for the first one.
    rich (E020): also colour confusions and a small-grid diff of the first failing pair (needs `train`)."""
    n, bad = len(rows), [i for i, r in enumerate(rows) if not r["ok"]]
    if not bad:
        head = (f"[program rejected (submission {k} of {cap}): it passes all {n} training pairs but failed on "
                f"test_input: {test_error}]")
        return head + "\nFix solve so that it also handles the test input."
    lines = [f"[program rejected (submission {k} of {cap}): {len(bad)} of {n} training pairs fail]"]
    for i, r in enumerate(rows):
        if r["ok"]:
            lines.append(f"pair {i}: OK")
        elif "error" in r:
            err = r["error"].splitlines()
            lines.append(f"pair {i}: error: " + (err[-1] if err else ""))
        elif "shape" in r:
            lines.append(f"pair {i}: wrong shape: got {r['shape'][0]}, expected {r['shape'][1]}")
        else:
            lines.append(f"pair {i}: right shape, {len(r['wrong'])} wrong cells")
    first = rows[bad[0]]
    if "error" in first and "\n" in first["error"]:
        lines.append(f"pair {bad[0]} traceback (last lines):\n{first['error']}")
    elif "wrong" in first:
        cells = "; ".join(f"({r},{c}) expected {e} got {g}" for r, c, e, g in first["wrong"][:max_cells])
        more = len(first["wrong"]) - max_cells
        lines.append(f"pair {bad[0]} first wrong cells (row,col): {cells}" + (f"; ... {more} more" if more > 0 else ""))
    if rich and train is not None:
        lines += rich_detail(first, train[bad[0]])
    return "\n".join(lines)


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/audit.py
"""E020 host-side checks of a train-passing program (no model tokens; never uses the test output).

test_audit(train, test_input, test_output, runtimes): flags from the program's own test output
  - colours in the test output that appear in no training output and not in test_input;
  - an output size that breaks a size rule every training pair follows (same as input, constant, fixed ratio);
  - an output identical to test_input (when no training output equals its input), or near-empty;
  - test-input features outside the training range (the task summary's "Outside train range" line);
  - a test run much slower than the training runs.
lint(code, train): hardcoded constants in the program source (AST)
  - literal tables (dict / list / set literals with >= 6 int-ish entries);
  - literal coordinates: subscripts with two int literals, one of them >= 2 in size, e.g. g[3][5] or g[3, 5];
  - (lists of 8 neighbour offsets in -1..1 are not flagged)
  - literal grids: nested list literals equal to a training grid, or comparisons against nested list literals;
  - branches on exact sizes: comparisons of len(...) / .shape with an int literal > 3.
"""
import ast
import re

SLOW_FACTOR, SLOW_MIN_S = 5.0, 2.0


def _colours(g):
    return {v for row in g for v in row}


def _shape(g):
    return len(g), len(g[0])


def _bg(g):
    counts = {}
    for row in g:
        for v in row:
            counts[v] = counts.get(v, 0) + 1
    return max(sorted(counts), key=lambda v: counts[v])


def size_rule_flag(train, test_input, out):
    ins = [_shape(p["input"]) for p in train]
    outs = [_shape(p["output"]) for p in train]
    ti, to = _shape(test_input), _shape(out)
    if all(i == o for i, o in zip(ins, outs)):
        if to != ti:
            return f"every training output has its input's size, but your test output is {to[0]}x{to[1]} for a {ti[0]}x{ti[1]} input"
        return None
    if len(set(outs)) == 1:
        if to != outs[0]:
            return f"every training output is {outs[0][0]}x{outs[0][1]}, but your test output is {to[0]}x{to[1]}"
        return None
    for d, name in ((0, "height"), (1, "width")):
        ratios = {o[d] / i[d] for i, o in zip(ins, outs)}
        if len(ratios) == 1:
            r = ratios.pop()
            if abs(to[d] - r * ti[d]) > 1e-9:
                return (f"in every training pair the output {name} is {r:g} x the input {name}, but your test output "
                        f"{name} is {to[d]} for an input {name} of {ti[d]}")
    return None


def test_audit(train, test_input, out, runtimes=None, summary_text=None):
    flags = []
    train_out_cols = set().union(*[_colours(p["output"]) for p in train])
    new = sorted(_colours(out) - train_out_cols - _colours(test_input))
    if new:  # colours invented by the program (in no training output and not in test_input)
        flags.append(f"colour(s) {','.join(map(str, new))} appear in your test output but in no training output "
                     "and not in test_input")
    flag = size_rule_flag(train, test_input, out)
    if flag:
        flags.append(flag)
    if out == test_input and not any(p["output"] == p["input"] for p in train):
        flags.append("your test output is identical to test_input, but no training output equals its input")
    bg = _bg(out)
    frac = sum(v != bg for row in out for v in row) / (len(out) * len(out[0]))
    train_fracs = []
    for p in train:
        b = _bg(p["output"])
        o = p["output"]
        train_fracs.append(sum(v != b for row in o for v in row) / (len(o) * len(o[0])))
    if len(_colours(out)) == 1 and all(len(_colours(p["output"])) > 1 for p in train):
        flags.append("your test output is a single colour, unlike every training output")
    elif frac < 0.5 * min(train_fracs):
        flags.append(f"your test output is nearly empty: {frac:.0%} non-background cells (training outputs: "
                     f"{min(train_fracs):.0%}-{max(train_fracs):.0%})")
    if summary_text:
        m = re.search(r"Outside train range: (.*)", summary_text)
        if m and not m.group(1).startswith("nothing"):
            flags.append("test_input is outside the training range: " + m.group(1).strip()[:600])
    if runtimes and len(runtimes) > 1:
        tr, te = max(runtimes[:-1]), runtimes[-1]
        if te > SLOW_MIN_S and te > SLOW_FACTOR * max(tr, 0.05):
            flags.append(f"solve took {te:.1f}s on test_input against at most {tr:.1f}s on the training inputs")
    return flags


def _int_like(node):
    if isinstance(node, ast.Constant) and type(node.value) is int:
        return True
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return _int_like(node.operand)
    if isinstance(node, (ast.Tuple, ast.List)):
        return bool(node.elts) and all(_int_like(e) for e in node.elts)
    return False


def _int_value(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.UnaryOp):
        return -_int_value(node.operand)
    return None


def _nested_list(node):
    return isinstance(node, ast.List) and node.elts and all(isinstance(e, ast.List) for e in node.elts)


def lint(code, train=None):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return []
    grids = [p[k] for p in (train or []) for k in ("input", "output")]
    flags, seen = [], set()

    def add(kind, node, text):
        key = (kind, getattr(node, "lineno", 0))
        if key not in seen:
            seen.add(key)
            flags.append(f"line {getattr(node, 'lineno', '?')}: {text}")

    for node in ast.walk(tree):
        if isinstance(node, ast.Dict) and len(node.keys) >= 6 and sum(
                _int_like(k) for k in node.keys if k is not None) + sum(_int_like(v) for v in node.values) >= 6:
            add("table", node, f"a literal table with {len(node.keys)} entries (a lookup fitted to the training "
                               "pairs will not generalise; derive the mapping from the grid)")
        elif isinstance(node, (ast.List, ast.Set, ast.Tuple)) and len(node.elts) >= 6 and all(
                isinstance(e, (ast.Tuple, ast.List)) and _int_like(e) for e in node.elts) and not all(
                _safe(e) and all(abs(v) <= 1 for v in _flat(ast.literal_eval(e))) for e in node.elts):  # not offsets
            if _nested_list(node) and any(ast.literal_eval(node) == g for g in grids if _safe(node)):
                add("grid", node, "a literal copy of a training grid")
            else:
                add("table", node, f"a literal list of {len(node.elts)} coordinate/value tuples")
        if _nested_list(node) and _safe(node) and any(ast.literal_eval(node) == g for g in grids):
            add("grid", node, "a literal copy of a training grid")
        if isinstance(node, ast.Subscript):
            idx = node.slice
            if isinstance(idx, ast.Tuple) and len(idx.elts) == 2 and all(_int_like(e) for e in idx.elts):
                vals = [_int_value(e) for e in idx.elts]
                if all(isinstance(v, int) for v in vals) and max(abs(v) for v in vals) >= 2:
                    add("coord", node, f"a hardcoded cell index [{vals[0]}, {vals[1]}]")
            elif (isinstance(node.value, ast.Subscript) and _int_like(idx) and _int_like(node.value.slice)
                  and not isinstance(node.value.slice, ast.Tuple)):
                vals = [_int_value(node.value.slice), _int_value(idx)]
                if all(isinstance(v, int) for v in vals) and max(abs(v) for v in vals) >= 2:
                    add("coord", node, f"a hardcoded cell index [{vals[0]}][{vals[1]}]")
        if isinstance(node, ast.Compare):
            parts = [node.left] + node.comparators
            if any(_nested_list(p) for p in parts):
                add("cmp", node, "a comparison with a literal grid (special-casing particular inputs)")
            sized = any((isinstance(p, ast.Call) and getattr(p.func, "id", "") == "len") or
                        (isinstance(p, ast.Attribute) and p.attr == "shape") or
                        (isinstance(p, ast.Subscript) and isinstance(p.value, ast.Attribute)
                         and p.value.attr == "shape") for p in parts)
            lits = [_int_value(p) for p in parts if isinstance(p, ast.Constant) and type(p.value) is int]
            if sized and any(v > 3 for v in lits) and any(isinstance(o, (ast.Eq, ast.NotEq)) for o in node.ops):
                add("size", node, f"a branch on an exact grid size ({max(lits)})")
    return flags[:8]


def _flat(x):
    return [v for e in x for v in _flat(e)] if isinstance(x, (list, tuple)) else [x]


def _safe(node):
    try:
        ast.literal_eval(node)
        return True
    except (ValueError, SyntaxError, TypeError):
        return False


def audit_report(n_train, out, flags, lint_flags, digits):
    lines = [f"[program passes all {n_train} training pairs. Before it becomes your answer, review this host-side "
             f"audit. Your output for test_input ({len(out)}x{len(out[0])}):", digits]
    if flags:
        lines.append("Audit flags (things that differ from every training pair):")
        lines += [f"- {f}" for f in flags]
    if lint_flags:
        lines.append("Hardcoded constants in your program (often fit the training pairs only):")
        lines += [f"- {f}" for f in lint_flags]
    if not flags and not lint_flags:
        lines.append("No audit flags.")
    lines.append("Check the output against your rule and against what is different about test_input. If it is right, "
                 "call submit_program again with the same code: that makes it your answer. Otherwise fix solve and "
                 "submit the fixed program.]")
    return "\n".join(lines)


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/selection.py
"""Kaggle answer selection (E021, notebook v3): attempt_1 / attempt_2 for one test pair, and which pairs get a
gated second attempt.

Pure functions over attempt records, shared by kaggle/src/solver.py and offline simulations. An attempt record
needs only `pred`, `train_pass`, `train_frac`, `answer_source`, `tokens` and optionally `alternatives` (the
agent_qwen `record_alternatives` list: other test grids of the same attempt, [{'src', 'pred', 'train_frac'}]).

Ranking (tiers, then order inside a tier):
  0. train-passing grids (the attempt's answer or an alternative with train_frac 1.0);
  1. attempts' own answers that are not known to fail train (e.g. submit_answer of `answer`, no check possible);
  2. the rest (answers known to fail train, alternatives below 1.0, unknown alternatives), by train_frac.
Inside tiers 0 and 1: attempt answers before alternatives, then attempt order, then the alternative's order.
With no train checks and no alternatives this is v2's rule: attempt answers in attempt order.
"""
from .grids import valid_grid

# answer sources that come from a fallback rather than a submission the model chose (E012/E019: ~0-20% correct)
WEAK_SOURCES = {"forced", "session", "salvage", "parse", "lenient", "program_best_fail", "session_solve"}


def candidates(attempts):
    """attempts: {attempt_index: record}. Valid grids only; duplicates are kept (select_two dedups)."""
    out = []
    for a in sorted(attempts):
        r = attempts[a] or {}
        if valid_grid(r.get("pred")):
            out.append({"grid": r["pred"], "attempt": a, "rank": 0, "primary": True, "src": r.get("answer_source"),
                        "train_pass": r.get("train_pass"), "train_frac": r.get("train_frac")})
        for i, alt in enumerate(r.get("alternatives") or []):
            if not isinstance(alt, dict) or not valid_grid(alt.get("pred")):
                continue
            f = alt.get("train_frac")
            out.append({"grid": alt["pred"], "attempt": a, "rank": i + 1, "primary": False,
                        "src": f"alt:{alt.get('src')}", "train_pass": (f == 1.0) if f is not None else None,
                        "train_frac": f})
    return out


def tier(c):
    if c.get("train_pass") is True:
        return 0
    if c["primary"] and c.get("train_pass") is not False:
        return 1
    return 2


def rank_key(c):
    t = tier(c)
    return (t, -(c.get("train_frac") or 0.0) if t == 2 else 0.0, not c["primary"], c["attempt"], c["rank"])


def select_two(cands):
    """The top-2 distinct candidates (0, 1 or 2 of them) in rank order."""
    out = []
    for c in sorted(cands, key=rank_key):
        if all(c["grid"] != o["grid"] for o in out):
            out.append(c)
        if len(out) == 2:
            break
    return out


def retry_category(rec, include_unknown=True):
    """Gated second attempt, by expected value per token (E012 full set, v2 config, 2 attempts per pair: a retry
    of a pair whose first attempt submitted a wrong answer solved 10/66 = 15%; of a submitted answer of unknown
    correctness 10/108 = 9%; after no answer or a forced/fallback answer only 2/59 = 3.4%, at 43k vs 34k tokens):
    0 = the model's own submitted answer fails the train check;
    1 = a submitted answer that cannot be train-checked (only with include_unknown);
    2 = no answer, or only a fallback answer (forced / session / salvage / best failing program ...);
    None = no retry (a train-passing grid exists: the answer or one of its alternatives)."""
    alts = [a for a in rec.get("alternatives") or [] if isinstance(a, dict) and valid_grid(a.get("pred"))]
    if rec.get("train_pass") is True or any(a.get("train_frac") == 1.0 for a in alts):
        return None
    if not valid_grid(rec.get("pred")) or rec.get("answer_source") in WEAK_SOURCES:
        return 2
    if rec.get("train_pass") is False:
        return 0
    return 1 if include_unknown else None


def retry_priority(rec, include_unknown=True):
    """Sort key for the gated second attempt (smaller first), or None if the pair should not be retried:
    retry_category first, then higher train fraction (of the answer or an alternative), then fewer tokens spent in
    the first attempt (a cheaper attempt finishes within the leftover time more often)."""
    cat = retry_category(rec, include_unknown)
    if cat is None:
        return None
    fracs = [rec.get("train_frac") or 0.0] + [a.get("train_frac") or 0.0 for a in rec.get("alternatives") or []
                                               if isinstance(a, dict)]
    return (cat, -max(fracs), rec.get("tokens") or 0)


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/worked_example.txt


Here is how a good attempt went on a different, easier task (an ARC training task, not this one):
1. Inspect with code instead of reading the grids:
```python
for i, p in enumerate(train):
    a, b = np.array(p['input']), np.array(p['output'])
    print(i, a.shape, b.shape, [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in zip(*np.nonzero(a != b))])
print([(o['colour'], o['size'], o['bbox']) for o in objects(train[0]['input'])])
```
Output:
```
0 (7, 7) (7, 7) [(1, 4, 0, 1), (2, 3, 0, 1), (5, 3, 0, 1)]
1 (7, 7) (7, 7) [(1, 2, 0, 1), (4, 4, 0, 1)]
[(8, 3, (0, 4, 1, 5)), (8, 3, (2, 2, 3, 3)), (8, 3, (5, 3, 6, 4))]
```
2. Hypothesis in one line: every object is an L of three 8-cells, and the empty cell of its 2x2 box becomes 1.
3. Draft solve at once and check it on every training pair:
```python
def solve(grid):
    g = np.array(grid)
    out = g.copy()
    for o in objects(g):
        r0, c0, r1, c1 = o['bbox']
        box = out[r0:r1 + 1, c0:c1 + 1]
        box[box == 0] = 1
    return out
check(solve)
```
Output: `pair 0: OK`, `pair 1: OK`, `ALL PASS`.
4. Check the rule's assumptions on test_input: `print([(o['colour'], o['size'], o['bbox']) for o in objects(test_input)])` lists four 3-cell objects of colour 8, each in a 2x2 box, as in train, so the rule applies unchanged.
5. Submit solve with submit_program. If the harness reports a failing pair, read its diff, fix the rule and submit again.
That attempt used four tool calls and little thinking: discover with code, not in your head.


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_helpers.py
"""E017 P3: numpy grid helpers. Used on the host by task_summary.py and, with --helpers, preloaded into the model's
sandbox session (the harness executes this file's source there, so it must stay plain numpy + stdlib).

Grids are lists of lists or numpy arrays of ints 0-9. Coordinates are (row, col); boxes are inclusive
(r0, c0, r1, c1).
"""
import copy as _copy
import hashlib as _hashlib
from collections import Counter as _Counter

import numpy as np

D8_NAMES = ("id", "rot90", "rot180", "rot270", "flipud", "fliplr", "transpose", "antitranspose")


def _arr(g):
    return np.array(g, dtype=int)


def bg_colour(g):
    """Most frequent colour (ties: the smaller colour)."""
    vals, counts = np.unique(_arr(g), return_counts=True)
    return int(vals[counts == counts.max()].min())


def bbox(mask):
    """(r0, c0, r1, c1) inclusive box of the True cells of `mask`, or None if empty."""
    rows, cols = np.nonzero(np.asarray(mask))
    if not len(rows):
        return None
    return int(rows.min()), int(cols.min()), int(rows.max()), int(cols.max())


def crop(g, box):
    r0, c0, r1, c1 = box
    return _arr(g)[r0:r1 + 1, c0:c1 + 1]


def _d8_op(a, name):
    return {"id": lambda x: x, "rot90": lambda x: np.rot90(x, 1), "rot180": lambda x: np.rot90(x, 2),
            "rot270": lambda x: np.rot90(x, 3), "flipud": np.flipud, "fliplr": np.fliplr,
            "transpose": lambda x: x.T, "antitranspose": lambda x: np.rot90(x, 2).T}[name](a)


def d8(g):
    """{name: array} for the 8 rotations/reflections (rot90 is numpy's counter-clockwise rotation)."""
    a = _arr(g)
    return {name: _d8_op(a, name) for name in D8_NAMES}


def match_d8(a, b):
    """Name of the first D8 transform t with t(a) == b, else None."""
    a, b = _arr(a), _arr(b)
    for name in D8_NAMES:
        t = _d8_op(a, name)
        if t.shape == b.shape and (t == b).all():
            return name
    return None


def _components(mask, diag=True):
    """Connected components of a boolean mask as lists of (r, c), ordered by their first cell (reading order)."""
    mask = np.asarray(mask, dtype=bool)
    h, w = mask.shape
    seen = np.zeros_like(mask)
    steps = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)] if diag else \
        [(-1, 0), (0, -1), (0, 1), (1, 0)]
    out = []
    for r in range(h):
        for c in range(w):
            if mask[r, c] and not seen[r, c]:
                seen[r, c] = True
                stack, cells = [(r, c)], []
                while stack:
                    y, x = stack.pop()
                    cells.append((y, x))
                    for dy, dx in steps:
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True
                            stack.append((ny, nx))
                out.append(sorted(cells))
    return out


def _local_mask(cells):
    rs = [r for r, _ in cells]
    cs = [c for _, c in cells]
    r0, c0 = min(rs), min(cs)
    m = np.zeros((max(rs) - r0 + 1, max(cs) - c0 + 1), dtype=bool)
    for r, c in cells:
        m[r - r0, c - c0] = True
    return m


def _holes(local):
    """Number of 4-connected regions of non-object cells fully enclosed by the object."""
    padded = np.pad(local, 1, constant_values=False)
    regions = _components(~padded, diag=False)
    h, w = padded.shape
    return sum(not any(r in (0, h - 1) or c in (0, w - 1) for r, c in reg) for reg in regions)


def shape_key(cells_or_mask):
    """8-hex-digit id of a cell set's shape, the same for all rotations/reflections (and positions)."""
    m = np.asarray(cells_or_mask)
    if m.dtype != bool:
        m = _local_mask([tuple(x) for x in cells_or_mask])
    else:
        box = bbox(m)
        m = crop(m.astype(int), box).astype(bool)
    forms = []
    for name in D8_NAMES:
        t = _d8_op(m, name)
        forms.append((t.shape, np.packbits(t).tobytes()))
    shape, packed = min(forms)
    return _hashlib.sha1(repr((shape, packed)).encode()).hexdigest()[:8]


def objects(g, by="colour", diag=True, bg=None):
    """Objects of a grid, in reading order of their first cell.

    by='colour': connected cells of one colour (excluding bg); by='nonbg': connected non-background cells of any
    colours. Each object: {colour (most common), colours, cells [(r, c)], bbox (r0, c0, r1, c1), size, holes,
    touches_border, shape_key}."""
    a = _arr(g)
    if bg is None:
        bg = bg_colour(a)
    h, w = a.shape
    comps = []
    if by == "colour":
        for colour in sorted(set(np.unique(a).tolist()) - {bg}):
            comps.extend(_components(a == colour, diag))
        comps.sort()
    elif by == "nonbg":
        comps = _components(a != bg, diag)
    else:
        raise ValueError("by must be 'colour' or 'nonbg'")
    out = []
    for cells in comps:
        counts = _Counter(int(a[r, c]) for r, c in cells)
        rs = [r for r, _ in cells]
        cs = [c for _, c in cells]
        box = (min(rs), min(cs), max(rs), max(cs))
        local = _local_mask(cells)
        out.append({"colour": max(sorted(counts), key=lambda k: counts[k]), "colours": sorted(counts),
                    "cells": cells, "bbox": box, "size": len(cells), "holes": _holes(local),
                    "touches_border": box[0] == 0 or box[1] == 0 or box[2] == h - 1 or box[3] == w - 1,
                    "shape_key": shape_key(local)})
    return out


def separators(g, bg=None):
    """Full rows/columns of one non-background colour that split the grid into panels.

    Returns {colour, rows, cols} (line indices) for the colour with the most such lines, or None."""
    a = _arr(g)
    if bg is None:
        bg = bg_colour(a)
    best = None
    for colour in sorted(set(np.unique(a).tolist()) - {bg}):
        rows = [r for r in range(a.shape[0]) if (a[r] == colour).all()]
        cols = [c for c in range(a.shape[1]) if (a[:, c] == colour).all()]
        if rows == list(range(a.shape[0])) or cols == list(range(a.shape[1])):
            continue
        if len(_spans(a.shape[0], rows)) * len(_spans(a.shape[1], cols)) < 2:
            continue
        if best is None or len(rows) + len(cols) > len(best["rows"]) + len(best["cols"]):
            best = {"colour": colour, "rows": rows, "cols": cols}
    return best


def _spans(n, lines):
    """Maximal runs of indices in range(n) not in `lines`."""
    spans, start = [], None
    for i in range(n + 1):
        inside = i < n and i not in lines
        if inside and start is None:
            start = i
        elif not inside and start is not None:
            spans.append((start, i - 1))
            start = None
    return spans


def panels(g, bg=None):
    """Sub-grids split by full separator lines: list of {'bbox', 'grid', 'pos': (i, j)} in reading order
    (empty list if the grid has no separator lines)."""
    a = _arr(g)
    sep = separators(a, bg)
    if sep is None:
        return []
    out = []
    for i, (r0, r1) in enumerate(_spans(a.shape[0], sep["rows"])):
        for j, (c0, c1) in enumerate(_spans(a.shape[1], sep["cols"])):
            out.append({"bbox": (r0, c0, r1, c1), "grid": a[r0:r1 + 1, c0:c1 + 1], "pos": (i, j)})
    return out


def stamp(g, pattern, at, clip=True, transparent=None):
    """Copy of g with `pattern` pasted with its top-left cell at `at` (may be negative or overhang);
    cells outside the grid are dropped (clip=True) or raise; cells equal to `transparent` are skipped."""
    a = _arr(g).copy()
    p = _arr(pattern)
    r0, c0 = at
    for (dr, dc), v in np.ndenumerate(p):
        if transparent is not None and v == transparent:
            continue
        r, c = r0 + dr, c0 + dc
        if 0 <= r < a.shape[0] and 0 <= c < a.shape[1]:
            a[r, c] = v
        elif not clip:
            raise IndexError(f"cell {(r, c)} outside the grid")
    return a


def ray(g, start, d, stop=None, bg=None):
    """Cells visited from `start` (exclusive) stepping by d=(dr, dc) until the grid edge or the first cell whose
    value satisfies stop(value) (that cell is not included). Default stop: any non-background value."""
    a = _arr(g)
    if stop is None:
        b = bg_colour(a) if bg is None else bg
        stop = lambda v: v != b  # noqa: E731
    r, c = start
    out = []
    while True:
        r, c = r + d[0], c + d[1]
        if not (0 <= r < a.shape[0] and 0 <= c < a.shape[1]) or stop(int(a[r, c])):
            return out
        out.append((r, c))


def show(g):
    """Print one string of digits per row."""
    for row in _arr(g):
        print("".join(str(int(v)) for v in row))


def check(fn, pairs=None, max_diffs=8):
    """Run fn on every training input (a copy) and print one line per pair: OK, the shape mismatch, an error, or
    up to `max_diffs` differing cells as (r,c) got/exp; then ALL PASS or k/n pass. Returns True if all pass."""
    if pairs is None:
        pairs = globals()["train"]
    ok = 0
    for i, pair in enumerate(pairs):
        exp = _arr(pair["output"])
        try:
            got = fn(_copy.deepcopy(pair["input"]))
            got = _arr(got.tolist() if hasattr(got, "tolist") else got)
        except Exception as exc:  # report and continue with the next pair
            print(f"pair {i}: error {type(exc).__name__}: {str(exc)[:200]}")
            continue
        if got.shape != exp.shape:
            print(f"pair {i}: shape {got.shape[0]}x{got.shape[1] if got.ndim > 1 else 0} "
                  f"vs expected {exp.shape[0]}x{exp.shape[1]}")
            continue
        diff = list(zip(*np.nonzero(got != exp)))
        if not diff:
            ok += 1
            print(f"pair {i}: OK")
            continue
        cells = ", ".join(f"({int(r)},{int(c)}) {int(got[r, c])}/{int(exp[r, c])}" for r, c in diff[:max_diffs])
        more = f", +{len(diff) - max_diffs} more" if len(diff) > max_diffs else ""
        print(f"pair {i}: {len(diff)} cells differ (r,c got/exp): {cells}{more}")
    print("ALL PASS" if ok == len(pairs) else f"{ok}/{len(pairs)} pass")
    return ok == len(pairs)


HELPER_NAMES = ("bg_colour", "bbox", "crop", "d8", "match_d8", "shape_key", "objects", "separators", "panels",
                "stamp", "ray", "show", "check")
for _name in HELPER_NAMES:
    globals()[_name]._arc_helper = True  # the train-check labeller skips these when probing session functions
del _name


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/task_summary.py
"""E017 P1 (--task-summary): a host-computed perception pack appended to the first user message.

Deterministic and LLM-free. For each training pair: a header line (sizes, background, changed cells, colour
counts, simple whole-grid relations), the input's object table (8-connected same-colour components; multi-colour
components when they exist; separator panels), and an input -> output object matching. For the test input: the
same table plus an explicit "Outside train range" line (size, colours, object counts, border contact, object
sizes, unseen shapes; the features of experiments/E017_reasoning_efficiency/scripts/test_novelty.py).

Shapes get letter ids shared across the whole task: equal letters mean the same shape up to rotation/reflection.
The text is capped at about 3k tokens (`max_chars`) by showing fewer object rows; the test section is kept whole.
"""
from collections import Counter

import numpy as np

from arc_helpers import _arr, _local_mask, bg_colour, match_d8, objects, separators

INTRO = ("A computed summary of the grids follows; trust it over reading the grids by hand, and use code for any "
         "count.")
LEGEND = ("Objects = 8-connected cells of one colour, background excluded, numbered in reading order; "
          "bbox=(r0,c0)-(r1,c1) inclusive; holes = enclosed regions; border = touches the grid edge; "
          "shape letters are shared by all grids (same letter = same shape up to rotation/reflection; "
          "'dot' = 1 cell).")
MAX_CHARS = 10000  # ~3k tokens of this digit-heavy text
ROW_STEPS = (12, 8, 5, 3, 0)


def _counts(a):
    vals, cnt = np.unique(a, return_counts=True)
    return "{" + ",".join(f"{int(v)}:{int(n)}" for v, n in zip(vals, cnt)) + "}"


def _box(b):
    return f"({b[0]},{b[1]})-({b[2]},{b[3]})"


def _ids(nums, limit=10):
    shown = " ".join(f"#{n}" for n in nums[:limit])
    return shown + (f" +{len(nums) - limit} more" if len(nums) > limit else "")


class _Shapes:
    """Task-wide letter ids for shape keys, in order of first appearance."""

    def __init__(self):
        self.names = {}

    def name(self, obj):
        if obj["size"] == 1:
            return "dot"
        key = obj["shape_key"]
        if key not in self.names:
            n, s = len(self.names), ""
            while True:
                s = chr(65 + n % 26) + s
                n = n // 26 - 1
                if n < 0:
                    break
            self.names[key] = s
        return self.names[key]


def _grid_relation(a, b):
    """Short description of a simple whole-grid relation from input a to a differently sized output b."""
    name = match_d8(a, b)
    if name:
        return f"out = in {name}"
    (h, w), (H, W) = a.shape, b.shape
    if H <= h and W <= w:
        for r in range(h - H + 1):
            for c in range(w - W + 1):
                if (a[r:r + H, c:c + W] == b).all():
                    return f"out = crop of in at ({r},{c})"
    if H % h == 0 and W % w == 0:
        ky, kx = H // h, W // w
        if (np.kron(a, np.ones((ky, kx), dtype=int)) == b).all():
            return f"out = in scaled x{ky}" if ky == kx else f"out = in scaled {ky}x{kx}"
        if (np.tile(a, (ky, kx)) == b).all():
            return f"out = in tiled {ky}x{kx}"
    return None


def _object_rows(objs, shapes, rows):
    """Object table lines: the `rows` largest objects in reading order, the rest summarised per colour."""
    keep = sorted(sorted(range(len(objs)), key=lambda i: (-objs[i]["size"], i))[:rows])
    lines = [f"  #{i + 1} c{objs[i]['colour']} size={objs[i]['size']} bbox={_box(objs[i]['bbox'])} "
             f"holes={objs[i]['holes']} border={'yes' if objs[i]['touches_border'] else 'no'} "
             f"shape={shapes.name(objs[i])}" for i in keep]
    rest = [objs[i] for i in range(len(objs)) if i not in set(keep)]
    if rest:
        by = {}
        for o in rest:
            by.setdefault(o["colour"], []).append(o["size"])
        lines.append(f"  +{len(rest)} more: " + ", ".join(
            f"c{c} x{len(s)} (sizes {min(s)}-{max(s)})" for c, s in sorted(by.items())))
    return lines


def _grid_block(label, a, objs, shapes, rows):
    lines = [f" {label} objects ({len(objs)}{'' if len(objs) <= rows else f'; {rows} largest shown'}):"
             if objs else f" {label} objects: none"]
    if objs:
        lines += _object_rows(objs, shapes, rows)
    multi = [o for o in objects(a, by="nonbg") if len(o["colours"]) > 1]
    if multi:
        shown = "; ".join(f"colours {','.join(map(str, o['colours']))} size={o['size']} bbox={_box(o['bbox'])}"
                          for o in multi[:min(4, max(rows, 1))])
        lines.append(f" {label} multi-colour objects ({len(multi)}): {shown}"
                     + (f"; +{len(multi) - 4} more" if len(multi) > 4 and rows >= 4 else ""))
    sep = separators(a)
    if sep:
        nr, nc = len(_runs(a.shape[0], sep["rows"])), len(_runs(a.shape[1], sep["cols"]))
        where = "; ".join(x for x in (f"rows {','.join(map(str, sep['rows']))}" if sep["rows"] else "",
                                       f"cols {','.join(map(str, sep['cols']))}" if sep["cols"] else "") if x)
        lines.append(f" {label} panels: {nr}x{nc} split by colour {sep['colour']} at {where}")
    return lines


def _runs(n, lines):
    from arc_helpers import _spans
    return _spans(n, lines)


def _match(a, b, in_objs, out_objs, out_bg):
    """Input -> output object matching for same-sized grids. Returns (summary line, unmatched output objects)."""
    out_cells = {tuple(o["cells"]): j for j, o in enumerate(out_objs)}
    claimed, groups = set(), {}

    def add(kind, text):
        groups.setdefault(kind, []).append(text)

    pending = []
    for i, o in enumerate(in_objs):
        j = out_cells.get(tuple(o["cells"]))
        if j is not None and j not in claimed:
            claimed.add(j)
            if out_objs[j]["colour"] == o["colour"]:
                add("unchanged", i + 1)
            else:
                add(f"recoloured {o['colour']}->{out_objs[j]['colour']}", i + 1)
        else:
            pending.append(i)
    for i in pending:
        o = in_objs[i]
        local = _local_mask(o["cells"])
        best = None
        for j, q in enumerate(out_objs):
            if j in claimed or q["colour"] != o["colour"] or q["shape_key"] != o["shape_key"]:
                continue
            dist = abs(q["bbox"][0] - o["bbox"][0]) + abs(q["bbox"][1] - o["bbox"][1])
            if best is None or dist < best[0]:
                best = (dist, j)
        if best is not None:
            j = best[1]
            claimed.add(j)
            q = out_objs[j]
            op = match_d8(local, _local_mask(q["cells"]))
            dy, dx = q["bbox"][0] - o["bbox"][0], q["bbox"][1] - o["bbox"][1]
            if op == "id":
                add("moved", f"#{i + 1} ({dy:+d},{dx:+d})")
            else:
                add("transformed", f"#{i + 1} {op} to {_box(q['bbox'])}")
            continue
        vals = [int(b[r, c]) for r, c in o["cells"]]
        if all(v == out_bg for v in vals):
            add("removed", i + 1)
        else:
            kept = sum(v == o["colour"] for v in vals)
            add("changed", f"#{i + 1} ({kept}/{o['size']} cells keep c{o['colour']})")
    parts = []
    order = ["unchanged", "moved", "transformed", "removed", "changed"]
    for kind in sorted(groups, key=lambda k: (order.index(k) if k in order else 1.5, k)):
        items = groups[kind]
        if items and isinstance(items[0], int):
            parts.append(f"{kind} {_ids(items)}")
        else:
            parts.append(f"{kind} " + ", ".join(items[:8]) + (f", +{len(items) - 8} more" if len(items) > 8 else ""))
    unmatched = [q for j, q in enumerate(out_objs) if j not in claimed]
    return " in->out: " + ("; ".join(parts) if parts else "no input objects"), unmatched


def _match_resized(in_objs, out_objs):
    """For differently sized grids: which input objects reappear (same colour and shape) in the output."""
    found, used = [], set()
    for i, o in enumerate(in_objs):
        for j, q in enumerate(out_objs):
            if j not in used and q["colour"] == o["colour"] and q["shape_key"] == o["shape_key"] and o["size"] > 1:
                op = match_d8(_local_mask(o["cells"]), _local_mask(q["cells"]))
                found.append(f"#{i + 1} at {_box(q['bbox'])}" + ("" if op == "id" else f" ({op})"))
                used.add(j)
                break
    return " in objects found in out (same colour+shape): " + (", ".join(found[:10]) + (
        f", +{len(found) - 10} more" if len(found) > 10 else "") if found else "none")


def _feats(a, objs):
    return {"h": a.shape[0], "w": a.shape[1], "colours": set(np.unique(a).tolist()), "bg": bg_colour(a),
            "ncol": len(np.unique(a)), "comps": len(objs), "touch": sum(o["touches_border"] for o in objs),
            "maxsize": max((o["size"] for o in objs), default=0), "minsize": min((o["size"] for o in objs), default=0)}


def _range(vals):
    lo, hi = min(vals), max(vals)
    return f"train: {lo}" if lo == hi else f"train {lo}-{hi}"


def novelty_line(train_inputs, train_objs, test, test_objs, shapes):
    """The 'Outside train range' line for the test input."""
    tr = [_feats(a, o) for a, o in zip(train_inputs, train_objs)]
    te = _feats(test, test_objs)
    flags = []

    def out(k):
        return not min(f[k] for f in tr) <= te[k] <= max(f[k] for f in tr)

    if out("h") or out("w"):
        flags.append(f"size {te['h']}x{te['w']} (train heights {_range([f['h'] for f in tr])[6:].strip(': ')}, "
                     f"widths {_range([f['w'] for f in tr])[6:].strip(': ')})")
    new = sorted(te["colours"] - set().union(*[f["colours"] for f in tr]))
    if new:
        flags.append(("new colour " if len(new) == 1 else "new colours ") + ",".join(map(str, new)))
    always = sorted(set.intersection(*[f["colours"] for f in tr]) - te["colours"])
    if always:
        flags.append(("colour " if len(always) == 1 else "colours ") + ",".join(map(str, always))
                     + " absent (present in every train input)")
    labels = {"ncol": "number of colours", "comps": "object count", "touch": "objects touching the border",
              "maxsize": "largest object size", "minsize": "smallest object size"}
    for k, text in labels.items():
        if out(k):
            flags.append(f"{text} {te[k]} ({_range([f[k] for f in tr])})")
    seen = {o["shape_key"] for objs in train_objs for o in objs}
    unseen = []
    for i, o in enumerate(test_objs):
        if o["shape_key"] not in seen and o["size"] > 1:
            name = shapes.name(o)
            if name not in [u[0] for u in unseen]:
                unseen.append((name, i + 1))
    if unseen:
        flags.append("shapes not in any train input: " + ", ".join(f"{n} (#{i})" for n, i in unseen[:8])
                     + (f", +{len(unseen) - 8} more" if len(unseen) > 8 else ""))
    if te["bg"] not in {f["bg"] for f in tr}:
        flags.append(f"background {te['bg']} (train: {','.join(map(str, sorted({f['bg'] for f in tr})))})")
    if not flags:
        return (" Outside train range: nothing (size, colours, object count, border contact and object sizes are "
                "within the train inputs' ranges; every shape appears in train).")
    return " Outside train range: " + "; ".join(flags)


def _render(train, test_input, rows):
    shapes = _Shapes()
    ins = [_arr(p["input"]) for p in train]
    outs = [_arr(p["output"]) for p in train]
    in_objs = [objects(a) for a in ins]
    out_objs = [objects(b) for b in outs]
    test = _arr(test_input)
    test_objs = objects(test)
    # assign shape letters in a fixed order: pair by pair (input, then output), then the test input
    for i in range(len(train)):
        for o in in_objs[i] + out_objs[i]:
            shapes.name(o)
    for o in test_objs:
        shapes.name(o)
    pair_blocks = []
    for i, (a, b) in enumerate(zip(ins, outs)):
        abg, bbg = bg_colour(a), bg_colour(b)
        head = f"Ex{i}: in {a.shape[0]}x{a.shape[1]} bg={abg} -> out {b.shape[0]}x{b.shape[1]} bg={bbg}"
        if a.shape == b.shape:
            diff = a != b
            trans = Counter((int(x), int(y)) for x, y in zip(a[diff], b[diff]))
            top = ", ".join(f"{x}->{y} x{n}" for (x, y), n in sorted(trans.items(), key=lambda t: (-t[1], t[0]))[:6])
            more = f", +{len(trans) - 6} kinds" if len(trans) > 6 else ""
            head += f" | {int(diff.sum())} cells changed" + (f" ({top}{more})" if trans else "")
            op = match_d8(a, b)
            if op not in (None, "id"):
                head += f" | out = in {op}"
        else:
            rel = _grid_relation(a, b)
            if rel:
                head += f" | {rel}"
        head += f" | colours in {_counts(a)} out {_counts(b)}"
        lines = [head] + _grid_block("in", a, in_objs[i], shapes, rows)
        if a.shape == b.shape:
            match, unmatched = _match(a, b, in_objs[i], out_objs[i], bbg)
            lines.append(match)
            if unmatched:
                lines.append(f" out objects not matching an input object ({len(unmatched)}): " + "; ".join(
                    f"c{q['colour']} size={q['size']} bbox={_box(q['bbox'])} shape={shapes.name(q)}"
                    for q in unmatched[:max(rows // 2, 2)])
                    + (f"; +{len(unmatched) - max(rows // 2, 2)} more" if len(unmatched) > max(rows // 2, 2) else ""))
        else:
            lines += _grid_block("out", b, out_objs[i], shapes, max(rows // 2, min(rows, 3)))
            lines.append(_match_resized(in_objs[i], out_objs[i]))
        pair_blocks.append("\n".join(lines))
    tl = [f"Test: in {test.shape[0]}x{test.shape[1]} bg={bg_colour(test)} | colours {_counts(test)}"]
    tl += _grid_block("test", test, test_objs, shapes, max(rows, 5))
    tl.append(novelty_line(ins, in_objs, test, test_objs, shapes))
    return pair_blocks, "\n".join(tl)


def task_summary(train, test_input, max_chars=MAX_CHARS):
    """The perception pack text (starts with a blank line), at most about `max_chars` characters."""
    for rows in ROW_STEPS:
        pairs, test = _render(train, test_input, rows)
        text = "\n\n" + "\n".join([INTRO, LEGEND, ""] + pairs + [test])
        if len(text) <= max_chars:
            return text
    # still too long at 0 rows: keep the test section, cut the pair blocks
    head = "\n\n" + "\n".join([INTRO, LEGEND, ""])
    budget = max_chars - len(head) - len(test) - 40
    body = "\n".join(pairs)[:max(budget, 0)]
    return head + body + "\n[summary truncated]\n" + test


In [ ]:
%%writefile /kaggle/working/arc_solver/evals/arc_harness/agent_qwen.py
"""Qwen3.8 tool-use agent for ARC-AGI-2 over a raw text-completion API (E012).

Backend-agnostic: the model is any `LLM` with `async complete(prompt, max_tokens, stop) -> Completion`
(`OpenAICompletionsLLM` covers SGLang / vLLM `/v1/completions`; an offline vLLM engine can implement the same
method). Code runs in an `executor.py` executor (Docker sandbox here, `LocalExecutor` on Kaggle).

Why raw completions instead of chat + server-side tool parsing:
- the prompt only ever grows (every past turn is appended verbatim), so the server's prefix cache hits on every
  turn; the Qwen template keeps the thinking of all turns after the last real user message anyway, so this is
  exactly what the chat template would render (checked by run_qwen_tools.py --verify-template);
- a turn cut by max_tokens, or an EOS in the middle of thinking, is continued in place;
- tool calls are parsed here (`parse_tool_calls`), so parse failures are visible and get feedback;
- no dependence on a particular server's tool / reasoning parser.

Harness logic follows E009 sub80s40 (DeepSeek's best): grid mode, `lean2sub` prompt, `run_python` +
`submit_answer` tools, budget note on every tool output with a soft target and hard cap, and at the hard cap a
forced `submit_answer` call (the model only writes the expression) with the session `answer` as fallback.
Exactly one scored grid per attempt; `answer_source` is submit | forced | parse | lenient | session | salvage | None.

answer_mode="program" (qwen-tools-4, E019): the answer can only be a program. `submit_answer` is replaced by
`submit_program(code)`; `code` must define solve(grid). The harness runs it in a fresh process on every training
input (host-side exact comparison) and only if all pass on the test input; failures return feedback and the attempt
continues (at most `max_submissions`). No forced typed answer at the cap: the fallback is the best submitted program
or the session's `solve`, whichever reproduces more training pairs. `answer_source` is program_pass |
program_best_fail | session_solve | None.
"""
import asyncio
import json
import re
import threading
import time
from dataclasses import asdict, dataclass

from .arcprize import build_prompt, parse
from .executor import (check_answer_on_train, program_feedback, run_program, run_session_solve, salvage_answer,
                       score_outputs, submit_answer, tool_output)
from .grids import valid_grid
from .prompts import PROMPT_SUFFIXES

HARNESS_VERSION = "qwen-tools-4"  # 2: forced-submit text names the expression; 3: robustness (R2/R3/R7/R8, S4/S7/S8)
# 4: opt-in answer_mode="program" (E019) and cache/train-check bookkeeping; answer_mode="submit" behaves as 3
# 5 (E020) is recorded only when an E020 option is active (V5_DEFAULTS); with all of them at their defaults the
# harness behaves exactly as 4 and records "qwen-tools-4", so older result files still resume.
HARNESS_VERSION_V5 = "qwen-tools-5"
V5_DEFAULTS = {"task_summary": False, "helpers": False, "turn_think_cap": 0, "turn_think_grace": 2048,
               "feedback": "basic", "confirm_test": False, "test_audit": False, "lint": False,
               "worked_example": False, "record_alternatives": False,
               "think_call_rescue": False}
V5_PROMPTS = {"lean2progroles"}
CONTEXT = 262144
PROGRAM_MEMORY_MB = 3072  # address-space limit of a submitted program's process (runner --session-mem-mb lowers it;
# an execution-safety setting, deliberately not part of the config so result files resume)
THINK_CLOSE = "</think>"
GEN_PROMPT = "<|im_start|>assistant\n<think>\n"
TIER_SYSTEM = {
    "xhigh": "Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, "
             "consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final "
             "answer.",
    "medium": None,
    "low": "Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion "
           "without unnecessary elaboration.",
}
# E009 lean2sub: lean2 + the submit_answer hand-off (byte-identical to run_deepseek_tools.py).
_SUBMIT = ("\n\nWhen your code reproduces every training output and its assumptions hold on test_input, call "
           "submit_answer with an expression such as `solve(test_input)`: the harness evaluates it in your session "
           "and that grid is your answer, so never retype the grid. Keep your current best guess for the test "
           "output in a variable `answer`; if you run out of budget it is submitted automatically.")
_PROGRAM = ("\n\nYour answer must be a program: typed grids are not accepted. When your code reproduces every "
            "training output and its assumptions hold on test_input, call submit_program with self-contained code "
            "that defines `solve(grid)`. The harness runs it on every training input and checks the outputs exactly "
            "before it runs it on the test input; if a pair fails it tells you what went wrong and you can fix the "
            "program and submit again. Keep your latest `solve` defined in your session too: if you run out of "
            "budget, the best program you submitted (or that `solve`) is used.")
# E017 P2 roles text (byte-identical to run_deepseek_tools.py lean2roles), on top of lean2prog.
_ROLES = ("\n\nBefore writing solve(), write a short ROLES note: what each kind of object or colour does (template, "
          "marker, key/legend, obstacle, container, background). Do not call anything noise or irrelevant unless code "
          "shows it is removed or unchanged in every training output. When solve() reproduces every training output, "
          "print what is different about test_input compared with all training inputs (sizes, colours, objects clipped "
          "by the border, counts, key entries, shapes not seen) and say in one line how solve handles each difference; "
          "if your rule ignores some non-background cells of test_input, explain them before submitting. If any "
          "training pair still fails, or a test difference is unhandled, do not submit: fix the rule.")
PROMPTS = {"lean2sub": PROMPT_SUFFIXES["lean2"] + _SUBMIT, "lean2prog": PROMPT_SUFFIXES["lean2"] + _PROGRAM,
           "lean2progroles": PROMPT_SUFFIXES["lean2"] + _PROGRAM + _ROLES}
# E017 P3 (--helpers): evals/arc_helpers.py preloaded in the session and in every submitted program's namespace.
_HELPERS_PATH = __import__("pathlib").Path(__file__).resolve().parents[1] / "arc_helpers.py"
HELPERS_SRC = _HELPERS_PATH.read_text() if _HELPERS_PATH.exists() else ""
HELPERS_DOC = (  # byte-identical to run_deepseek_tools.py HELPERS_DOC
    " Also preloaded, numpy grid helpers (g = list or array; boxes (r0,c0,r1,c1) inclusive): "
    "objects(g, by='colour'|'nonbg', diag=True, bg=None) -> list of dicts {colour, colours, cells [(r,c)], bbox, "
    "size, holes, touches_border, shape_key} in reading order (equal shape_key = same shape up to "
    "rotation/reflection); bbox(mask); crop(g, box); bg_colour(g); d8(g) -> {name: array} for the 8 "
    "rotations/reflections (id, rot90, rot180, rot270, flipud, fliplr, transpose, antitranspose); match_d8(a, b) "
    "-> name of the transform mapping a to b, or None; panels(g) -> [{'bbox', 'grid', 'pos'}] sub-grids split by "
    "full separator lines ([] if none); stamp(g, pattern, at, clip=True, transparent=None) -> copy of g with "
    "pattern pasted at top-left `at` (may overhang; clipped at the edges); ray(g, start, d, stop=None) -> cells "
    "from start (exclusive) stepping by d=(dr,dc) until the edge or the first cell where stop(value) (default: "
    "non-background); show(g) prints one digit string per row; check(fn) runs fn on a copy of every training "
    "input and prints per pair OK, the shape mismatch, the error, or up to 8 differing cells as (r,c) got/exp, "
    "then ALL PASS; returns True if all pass.")
HELPERS_PROMPT = "\n\nUse check(solve) after every change to solve."
# E020 --worked-example: a short workflow demonstration on ARC *training* task 3aa6fb7a (never an eval task), with
# the real outputs of its code (objects/check are the --helpers functions).
_EXAMPLE_PATH = __import__("pathlib").Path(__file__).resolve().parent / "worked_example.txt"
WORKED_EXAMPLE = _EXAMPLE_PATH.read_text().rstrip() if _EXAMPLE_PATH.exists() else ""
HELPERS_PROGRAM_DOC = " The grid helpers listed for run_python (objects, crop, d8, ...) are preloaded here too."
# E020 per-turn thinking cap: after `turn_think_cap` tokens of thinking in one turn, this note is appended to the
# thinking (in the model's voice); if the thinking is still open `turn_think_grace` tokens later, it is closed and
# a run_python call is opened (the model writes the code).
THINK_NUDGE = ("\n\nI have been reasoning for a long time without running anything. I should stop here and test my "
               "current hypothesis with code now (and use code, not my head, for any counting or cell reading).\n")
CONFIRM_TEXT = ("[program passes all {n} training pairs. Its output for test_input ({h}x{w}):\n{grid}\n"
                "Check it critically against your rule and against what is different about test_input. If it is "
                "right, call submit_program again with the same code: that makes it your answer. If not, fix solve "
                "and submit the fixed program.]")


def _review(cfg, train, test_input, rec, code, confirm):
    """The one-time review text for the first train-passing program, or None to accept it directly
    (lint only, nothing flagged)."""
    from .audit import audit_report, lint, test_audit
    out = rec["test"]
    flags = []
    if cfg.test_audit:
        try:
            from task_summary import task_summary
            summary = task_summary(train, test_input)
        except Exception:
            summary = None
        flags = test_audit(train, test_input, out, rec.get("runtimes"), summary)
    lint_flags = lint(code, train) if cfg.lint else []
    if not (cfg.confirm_test or cfg.test_audit) and not lint_flags:
        return None
    confirm["shown"], confirm["audit"] = out, {"flags": flags, "lint": lint_flags}
    if not (cfg.test_audit or lint_flags):
        return CONFIRM_TEXT.format(n=len(train), h=len(out), w=len(out[0]), grid=_digit_rows(out))
    return audit_report(len(train), out, flags, lint_flags, _digit_rows(out))


def _digit_rows(g):
    return "\n".join("".join(str(v) for v in row) for row in g)


THINK_FORCE_ACT = "\n\nLet me test this with code now.\n</think>\n\n<tool_call>\n<function=run_python>\n<parameter=code>\n"
# A final text reply in program mode (no tool call): typed answers are not accepted.
PROGRAM_NUDGE = ("Typed answers are not accepted. Submit your answer as a program: call submit_program with "
                 "self-contained code that defines solve(grid).")
TOOLS_HEADER = ("# Tools\n\nYou have access to the following functions:\n\n<tools>")
TOOLS_FOOTER = (
    "\n</tools>\n\nIf you choose to call a function ONLY reply in the following format with NO suffix:\n\n"
    "<tool_call>\n<function=example_function_name>\n<parameter=example_parameter_1>\nvalue_1\n</parameter>\n"
    "<parameter=example_parameter_2>\nThis is the value for the second parameter\nthat can span\nmultiple lines\n"
    "</parameter>\n</function>\n</tool_call>\n\n<IMPORTANT>\nReminder:\n- Function calls MUST follow the specified "
    "format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags\n"
    "- Required parameters MUST be specified\n- You may provide optional reasoning for your function call in "
    "natural language BEFORE the function call, but NOT after\n- If there is no function call available, answer "
    "the question like normal with your current knowledge and do not tell the user about function calls\n"
    "</IMPORTANT>")
# At the hard cap: close the thinking and open a submit_answer call; the model writes only the expression.
FORCE_THINK = ("\n\nI am out of budget, so I must submit my best answer now as one short expression: `answer` if I "
               "set it, otherwise solve(test_input) or the name of my best function applied to test_input.\n</think>\n\n")
FORCE_CALL = "<tool_call>\n<function=submit_answer>\n<parameter=code>\n"
LOW_YIELD = 20


@dataclass
class AgentConfig:
    tier: str = "xhigh"
    prompt: str = "lean2sub"
    max_total_output: int = 80000   # hard cap: generated tokens per attempt, all turns
    budget_soft: int = 40000        # soft target stated in the budget notes (0 = none)
    budget_notes: float = 0.01      # notes on tool outputs once this fraction of the hard cap is used (0 = off)
    budget_warn: float = 0.75       # from this fraction on, the note says to submit now
    soft_wording: str = "gentle"    # past the soft target: gentle = submit once a rule fits; strong = submit now (E010)
    chunk_tokens: int = 16384       # max tokens per completion request (a turn may take several)
    max_turns: int = 60
    tool_timeout: float = 30
    max_tool_output: int = 16000
    verify_submit: bool = False     # E011: re-run the submitted expression on train inputs; reject once on mismatch
    force_submit_tokens: int = 256  # tokens for the forced submit_answer expression at the cap (0 = off)
    temperature: float = 1.0
    top_p: float = 0.95
    top_k: int = 20
    max_eos_continues: int = 4      # EOS inside thinking: continue the turn at most this many times
    context: int = CONTEXT          # the server's context length (max_model_len); Kaggle runs 131072
    salvage: bool = True            # no answer at the end: apply the session's best-scoring function (qwen-tools-3)
    exec_margin: float = 35         # hard wall-clock cap per executor call = tool_timeout + exec_margin
    answer_mode: str = "submit"     # submit (qwen-tools-3 behaviour) | program (E019: submit_program only)
    max_submissions: int = 6        # program mode: submit_program calls per attempt
    program_timeout: float = 10     # program mode: seconds per grid when running a submitted program
    max_nudges: int = 2             # program mode: text replies answered with PROGRAM_NUDGE before the attempt ends
    record_train_check: bool = False  # submit mode: after the attempt, train-check the submitted expression (no tokens)
    # E020 (qwen-tools-5), all opt-in:
    task_summary: bool = False      # append the E017 host-computed perception pack (evals/task_summary.py)
    helpers: bool = False           # preload evals/arc_helpers.py in the session and in submitted programs
    turn_think_cap: int = 0         # thinking tokens per turn before THINK_NUDGE (0 = off)
    turn_think_grace: int = 2048    # tokens after the nudge before the thinking is closed into a run_python call
    feedback: str = "basic"         # program mode: basic | rich (adds colour confusions and small-grid diffs)
    confirm_test: bool = False      # program mode: show the first train-passing test output; accept on re-submit
    test_audit: bool = False        # program mode: like confirm_test, plus host-side audit flags (audit.py)
    lint: bool = False              # program mode: flag hardcoded constants (audit.lint) in rejected and passing programs
    worked_example: bool = False    # append the worked_example.txt workflow demo (ARC training task; needs --helpers)
    record_alternatives: bool = False  # bookkeeping: other test grids of the attempt (free attempt_2), no tokens
    think_call_rescue: bool = False  # after a think nudge, a complete <tool_call> inside open thinking is executed


# ---------------------------------------------------------------- rendering (Qwen3.8 chat template, tools on)

def tool_specs(cfg):
    descriptions = {
        "run_python": (
            "Execute a Python cell in your persistent IPython session. Variables/functions persist. "
            "Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. "
            f"scipy is available. No internet or host files. Timeout {cfg.tool_timeout:g}s. "
            "Use print() to inspect results. Check transformations on all training pairs."),
        "submit_answer": (
            "End this attempt with your test output. `code` is a Python expression evaluated in your session, "
            "e.g. solve(test_input) or answer, giving a rectangular grid of ints 0-9 (list or numpy array); "
            "statements that set a variable `answer` also work. The harness reads the grid directly: do not "
            "retype it. Invalid grids return an error and the attempt continues."),
    }
    if cfg.answer_mode == "program":
        del descriptions["submit_answer"]
        descriptions["submit_program"] = (
            "Submit your answer as a program; this is the only way to answer (typed grids are not accepted). "
            "`code` is self-contained Python source that defines solve(grid): grid is a list of lists of ints, "
            "and solve returns the output grid (list of lists or numpy array of ints 0-9). It runs in a fresh "
            "Python process: numpy is preloaded as np, but include every other import and helper function (your "
            "session variables are not available). The harness first runs solve on every training input and "
            "compares the results exactly with the training outputs. If any pair fails, you get feedback (failing "
            "pairs, shapes, wrong cells, errors) and the attempt continues. If all pass, it runs solve on the test "
            f"input and that grid is your answer, which ends the attempt. Limit {cfg.program_timeout:g}s per grid "
            f"and {cfg.max_submissions} submissions per attempt.")
    if cfg.helpers:
        descriptions["run_python"] += HELPERS_DOC
        if "submit_program" in descriptions:
            descriptions["submit_program"] += HELPERS_PROGRAM_DOC
    return [{"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}},
                       "required": ["code"], "additionalProperties": False},
    }} for name, description in descriptions.items()]


def render_system(tier, tools):
    s = "<|im_start|>system\n"
    if TIER_SYSTEM[tier]:
        s += TIER_SYSTEM[tier] + "\n\n"
    s += TOOLS_HEADER + "".join("\n" + json.dumps(t, ensure_ascii=False) for t in tools) + TOOLS_FOOTER
    return s + "<|im_end|>\n"


def render_user(content):
    return f"<|im_start|>user\n{content.strip()}<|im_end|>\n"


def render_tool_responses(outputs):
    return "<|im_start|>user" + "".join(f"\n<tool_response>\n{o.strip()}\n</tool_response>" for o in outputs) + \
        "<|im_end|>\n"


def render_assistant(reasoning, content, calls):
    """A past assistant turn as the chat template renders it (used to check our verbatim appends)."""
    s = f"<|im_start|>assistant\n<think>\n{reasoning.strip()}\n</think>\n\n{content.strip()}"
    for i, (name, params) in enumerate(calls):
        s += ("\n\n" if content.strip() else "") if i == 0 else "\n"
        s += f"<tool_call>\n<function={name}>\n"
        s += "".join(f"<parameter={k}>\n{v}\n</parameter>\n" for k, v in params.items())
        s += "</function>\n</tool_call>"
    return s + "<|im_end|>\n"


def user_content(train, test_input, prompt, helpers=False, summary=False, example=False):
    text = build_prompt(train, test_input) + PROMPTS[prompt]
    if helpers:
        text += HELPERS_PROMPT
    if example:
        text += WORKED_EXAMPLE
    if summary:  # E017 P1, imported lazily (needs evals/ on sys.path; not used on Kaggle unless enabled)
        from task_summary import task_summary
        text += task_summary(train, test_input)
    return text


def initial_prompt(train, test_input, cfg, note=None):
    """note (E020 --memory): extra text appended to the first user message (e.g. earlier attempts' failures)."""
    return render_system(cfg.tier, tool_specs(cfg)) + render_user(
        user_content(train, test_input, cfg.prompt, cfg.helpers, cfg.task_summary, cfg.worked_example)
        + (note or "")) + GEN_PROMPT


# ---------------------------------------------------------------- parsing

_CALL = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)
_FUNC = re.compile(r"^\s*<function=([^>\n]+)>(.*?)</function>\s*$", re.DOTALL)
_PARAM = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)


def split_turn(text):
    """(reasoning, content) of one generated assistant turn; content is None while still thinking."""
    i = text.find(THINK_CLOSE)
    if i < 0:
        return text, None
    return text[:i], text[i + len(THINK_CLOSE):]


def parse_tool_calls(content):
    """Tool calls in the answer part of a turn: (calls, errors). calls = [(name, {param: str})].

    errors lists malformed blocks (unclosed <tool_call>, missing <function=...>)."""
    calls, errors = [], []
    for block in _CALL.findall(content):
        m = _FUNC.match(block)
        if not m:
            errors.append("malformed <tool_call> block: expected <function=NAME>...</function>")
            continue
        calls.append((m.group(1).strip(), {k.strip(): v for k, v in _PARAM.findall(m.group(2))}))
    tail = _CALL.sub("", content)
    if "<tool_call>" in tail or "<function=" in tail:
        errors.append("unclosed or malformed tool call: use <tool_call><function=NAME><parameter=code>..."
                      "</parameter></function></tool_call>")
    return calls, errors


def lenient_parse(text):
    """Last valid JSON grid in the text (trailing commas tolerated), else the last block of one-row-per-line."""
    text = text or ""
    for start in [i for i, ch in enumerate(text) if ch == "["][::-1]:
        if not re.match(r"\[\s*\[", text[start:start + 50]):
            continue
        depth = 0
        for j in range(start, len(text)):
            depth += (text[j] == "[") - (text[j] == "]")
            if depth == 0:
                block = text[start:j + 1]
                for candidate in (block, re.sub(r",\s*\]", "]", block)):
                    try:
                        grid = json.loads(candidate)
                    except json.JSONDecodeError:
                        continue
                    if valid_grid(grid):
                        return grid
                break
    rows, best = [], None
    for line in text.splitlines() + [""]:
        line = line.strip().rstrip(",")
        try:
            row = json.loads(line) if line.startswith("[") else None
        except json.JSONDecodeError:
            row = None
        if isinstance(row, list) and row and all(type(v) is int for v in row):
            rows.append(row)
            continue
        if rows and valid_grid(rows):
            best = rows
        rows = []
    return best


def budget_note(cfg, used):
    """Status line appended to the last tool output of a turn (E009 wording)."""
    if not cfg.budget_notes or used < cfg.budget_notes * cfg.max_total_output:
        return ""
    hard = cfg.max_total_output
    tool = "submit_program" if cfg.answer_mode == "program" else "submit_answer"
    if cfg.budget_soft:
        note = (f"\n[budget: {used / 1000:.0f}k output tokens used; aim to finish within {cfg.budget_soft / 1000:.0f}k "
                f"(hard limit {hard / 1000:.0f}k)]")
        if cfg.budget_soft <= used < cfg.budget_warn * hard:
            note += (f" Past the target: submit your best answer now with {tool}, unless one more experiment "
                     "will very likely make every training pair pass." if cfg.soft_wording == "strong"
                     else " Past the target: submit as soon as a rule fits every training pair.")
    else:
        note = f"\n[budget: {used / 1000:.0f}k of {hard / 1000:.0f}k output tokens used]"
    if used >= cfg.budget_warn * hard:
        note += (f" Budget nearly spent: submit your best answer now with {tool} "
                 + ("(or at least keep `solve` defined in your session)." if tool == "submit_program"
                    else "(or at least set `answer`)."))
    return note


# ---------------------------------------------------------------- model backend

@dataclass
class Completion:
    text: str
    completion_tokens: int
    prompt_tokens: int
    finish: str  # "stop" (EOS or stop string) or "length"
    cached_tokens: int = None  # prompt tokens served from the server's prefix cache, when it reports them
    served_by: str = None      # worker that served the request, when a gateway reports it in a response header


class OpenAICompletionsLLM:
    """Raw /v1/completions on an OpenAI-compatible server (SGLang, vLLM). Retries transient errors only:
    a completion has no side effects, so a retry is safe."""

    def __init__(self, base_url, api_key, model, temperature=1.0, top_p=0.95, top_k=20, timeout=3600,
                 max_retries=8, extra_body=None):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI(api_key=api_key or "none", base_url=base_url, timeout=timeout, max_retries=0)
        self.model, self.temperature, self.top_p, self.top_k = model, temperature, top_p, top_k
        self.max_retries, self.extra_body = max_retries, extra_body or {}
        self.transient_errors = 0

    async def complete(self, prompt, max_tokens, stop=None, extra_headers=None):
        import openai
        errs = 0
        while True:
            try:
                raw = await self.client.completions.with_raw_response.create(
                    model=self.model, prompt=prompt, max_tokens=max_tokens, temperature=self.temperature,
                    top_p=self.top_p, stop=stop, extra_body={"top_k": self.top_k, **self.extra_body},
                    extra_headers=extra_headers)
                r = raw.parse()
                served = next((v for k, v in raw.headers.items() if "worker" in k.lower()), None)
                c = r.choices[0]
                details = getattr(r.usage, "prompt_tokens_details", None)
                cached = details.get("cached_tokens") if isinstance(details, dict) else getattr(details, "cached_tokens", None)
                return Completion(c.text, r.usage.completion_tokens, r.usage.prompt_tokens, c.finish_reason, cached,
                                  served)
            except (openai.APIConnectionError, openai.APITimeoutError, openai.APIStatusError) as exc:
                status = getattr(exc, "status_code", None)
                if status is not None and status < 500 and status not in (408, 429):
                    raise
                errs += 1
                self.transient_errors += 1
                if errs > self.max_retries:
                    raise
                await asyncio.sleep(min(60, 5 * 2 ** min(errs - 1, 4)))

    async def close(self):
        await self.client.close()


class RoutedLLM:
    """Per-attempt view of an OpenAICompletionsLLM that sends a fixed routing header on every request (E019: the
    SGLang gateway's manual routing pins one attempt's conversation to one worker, so its prefix cache is reused)."""

    def __init__(self, llm, header, key):
        self.llm, self.headers = llm, {header: key}

    async def complete(self, prompt, max_tokens, stop=None):
        return await self.llm.complete(prompt, max_tokens, stop, extra_headers=self.headers)


# ---------------------------------------------------------------- agent loop

async def _call(fn, *args):
    # Blocking executor calls run in the loop's default thread pool. With many concurrent attempts, set a pool with
    # one thread per attempt: loop.set_default_executor(ThreadPoolExecutor(concurrency + 4)).
    return await asyncio.to_thread(fn, *args)


class ExecutorTimeout(RuntimeError):
    """An executor call did not return within its wall-clock cap; the session was replaced."""


def _quiet_close(session):
    try:
        session.close()
    except Exception:
        pass


class _Session:
    """The attempt's executor behind a hard wall-clock cap per call (the executor's own timeout is advisory: a
    wedged sandbox or child must not block the attempt). On a cap hit the session is abandoned (closed in a
    background thread) and a fresh one with only the task data takes its place."""

    def __init__(self, make_executor, train, test_input, call_limit, start_limit=180):
        self.make, self.train, self.test_input = make_executor, train, test_input
        self.call_limit, self.start_limit = call_limit, start_limit
        self.session, self.resets = None, 0

    async def start(self):
        session = self.make(self.train, self.test_input)
        try:
            await asyncio.wait_for(_call(session.__enter__), self.start_limit)
        except BaseException:
            threading.Thread(target=_quiet_close, args=(session,), daemon=True).start()
            raise
        self.session = session

    async def run(self, fn, *args):
        if self.session is None:
            raise RuntimeError("no executor session")
        try:
            return await asyncio.wait_for(_call(fn, self.session, *args), self.call_limit)
        except asyncio.TimeoutError:
            old, self.session = self.session, None
            threading.Thread(target=_quiet_close, args=(old,), daemon=True).start()
            self.resets += 1
            await self.start()
            raise ExecutorTimeout(f"executor call exceeded {self.call_limit:g}s") from None

    async def close(self):
        if self.session is not None:
            session, self.session = self.session, None
            try:
                await asyncio.wait_for(_call(session.close), 30)
            except BaseException:
                threading.Thread(target=_quiet_close, args=(session,), daemon=True).start()


def _execute(session, code):
    return session.execute(code)


def estimate_tokens(known_tokens, known_chars, text):
    """Prompt tokens of `text`, given that its first `known_chars` characters were measured as `known_tokens`.
    The unmeasured rest (tool output, the turn so far) is counted at 2 characters per token (conservative for
    JSON grids)."""
    return known_tokens + max(0, len(text) - known_chars + 1) // 2


async def solve(llm, make_executor, train, test_input, cfg, note=None):
    """One attempt. Returns (transcript, result). transcript is the full raw prompt+generation text.

    make_executor(train, test_input) -> executor (context manager with .execute(code)); train holds the
    training pairs with outputs (used host-side for --verify-submit), test_input the test grid only.
    Never raises (except on cancellation): errors are recorded, and any answer already set is still taken."""
    prompt = initial_prompt(train, test_input, cfg, note)
    used, turns, n_calls, n_tool_err, n_parse_err, n_eos, n_chunks = 0, 0, 0, 0, 0, 0, 0
    pred = source = stop = error = salvage_info = None
    max_prompt, content, submit_verified, forced_codes = 0, "", None, set()
    known_tokens, known_chars = 0, 0  # last measured prompt (tokens, chars) for the context estimate
    turn_log, t0 = [], time.monotonic()
    ex = _Session(make_executor, train, test_input, cfg.tool_timeout + cfg.exec_margin)
    program_mode = cfg.answer_mode == "program"
    answer_tool = "submit_program" if program_mode else "submit_answer"
    submissions, n_nudges, answer_code, program_info = [], 0, None, {}
    cache = {"prompt": 0, "cached": 0, "ideal": 0, "reported": 0, "last": ""}
    last_solve_cell, servers = None, {}
    think_stats = {"nudges": 0, "forced": 0, "rescued": 0}
    confirm = {"shown": None}

    async def complete(text, max_tokens, stop_strings=None):
        nonlocal used, n_chunks, max_prompt, known_tokens, known_chars
        c = await llm.complete(text, max_tokens, stop_strings) if stop_strings else await llm.complete(text, max_tokens)
        n_chunks += 1
        used += c.completion_tokens
        max_prompt = max(max_prompt, c.prompt_tokens)
        cache["prompt"] += c.prompt_tokens
        # ideal hit: everything sent before (previous prompt + its completion) is a prefix of this prompt
        cache["ideal"] += min(known_tokens, c.prompt_tokens) if text.startswith(cache["last"]) else 0
        cache["last"] = text + c.text
        if getattr(c, "served_by", None):
            servers[c.served_by] = servers.get(c.served_by, 0) + 1
        if getattr(c, "cached_tokens", None) is not None:
            cache["cached"] += c.cached_tokens
            cache["reported"] += 1
        known_tokens, known_chars = c.prompt_tokens + c.completion_tokens, len(text) + len(c.text)
        return c

    async def run_on(code, grids):
        """Submitted program on each grid in a fresh process; executor timeouts count as errors."""
        outs = []
        for g in grids:
            t_run = time.monotonic()
            try:
                outs.append(await ex.run(run_program, code, g, cfg.program_timeout, PROGRAM_MEMORY_MB,
                                         HELPERS_SRC if cfg.helpers else ""))
            except ExecutorTimeout:
                outs.append({"error": "timeout (executor call cap; Python session restarted)", "kind": "timeout"})
            outs[-1]["seconds"] = round(time.monotonic() - t_run, 2)
            err = outs[-1].get("error") or ""
            if outs[-1].get("kind") in ("SyntaxError", "IndentationError") or "does not define solve(grid)" in err:
                outs += [outs[-1]] * (len(grids) - len(outs))  # same failure on every grid
                break
        return outs

    async def check_program(code):
        """Train-check a submitted program, then the test input if all train pairs pass. Returns a record."""
        train_outs = await run_on(code, [pair["input"] for pair in train])
        rows = score_outputs(train_outs, train)
        n_pass = sum(r["ok"] for r in rows)
        runtimes = [o.get("seconds", 0) for o in train_outs]
        rec = {"n_pass": n_pass, "n_train": len(train), "train_frac": n_pass / len(train), "code": code,
               "fail_kinds": sorted({r.get("kind") or ("shape" if "shape" in r else "cells")
                                     for r in rows if not r["ok"]}), "test": None, "test_error": None}
        rec["runtimes"] = runtimes
        if n_pass == len(train):
            out = (await run_on(code, [test_input]))[0]
            rec["runtimes"] = runtimes + [out.get("seconds", 0)]
            if "error" in out:
                rec["test_error"] = out["error"].splitlines()[-1] if out["error"] else "error"
            elif not valid_grid(out["out"]):
                from .executor import grid_shape
                rec["test_error"] = f"not a valid grid ({grid_shape(out['out'])})"
            else:
                rec["test"] = out["out"]
        return rec, rows

    async def submit(code):
        try:
            return await ex.run(submit_answer, code)
        except ExecutorTimeout:
            return "[answer not accepted] evaluation timed out; Python state lost (train/test_input/np restored)", None

    try:
        try:
            await ex.start()
            while True:
                if turns >= cfg.max_turns:
                    stop = "max_turns"
                    break
                turns += 1
                text, low_streak, after_eos, out_of_budget = "", 0, False, None
                turn_tokens = 0
                nudge_at, act_forced, nudge_pos = None, False, 0  # E020 per-turn thinking cap state
                while True:  # one assistant turn, possibly over several requests
                    left = cfg.max_total_output - used
                    room = cfg.context - estimate_tokens(known_tokens, known_chars, prompt + text) - 64
                    if left <= 0 or room < 256:
                        out_of_budget = "output_budget" if left <= 0 else "context"
                        break
                    want = min(cfg.chunk_tokens, left, room)
                    capped = bool(cfg.turn_think_cap) and not act_forced and THINK_CLOSE not in text
                    if capped:
                        limit = cfg.turn_think_cap if nudge_at is None else nudge_at + cfg.turn_think_grace
                        want = min(want, max(1, limit - turn_tokens))
                    c = await complete(prompt + text, want)
                    turn_tokens += c.completion_tokens
                    text += c.text
                    if after_eos:
                        low_streak = low_streak + 1 if c.completion_tokens < LOW_YIELD else 0
                    after_eos = False
                    if cfg.think_call_rescue and nudge_at is not None and THINK_CLOSE not in text:
                        # E021 finding: after the nudge the model may write its tool call inside the still-open
                        # thinking and stop; close the thinking before the call and keep the call
                        i = text.find("<tool_call>", nudge_pos)
                        j = text.find("</tool_call>", i) if i >= 0 else -1
                        if j >= 0:
                            text = text[:i] + THINK_CLOSE + "\n\n" + text[i:j + len("</tool_call>")]
                            think_stats["rescued"] += 1
                            break
                    if capped and THINK_CLOSE not in text and turn_tokens >= limit:
                        if nudge_at is None:  # first: a note in the model's voice; it may keep thinking briefly
                            text += THINK_NUDGE
                            nudge_at, nudge_pos = turn_tokens, len(text)
                            think_stats["nudges"] += 1
                        else:  # still thinking after the grace: close it and open a run_python call
                            text += THINK_FORCE_ACT
                            act_forced = True
                            think_stats["forced"] += 1
                        continue
                    if c.finish == "length":
                        continue
                    if THINK_CLOSE not in text and n_eos < cfg.max_eos_continues and low_streak < 2:
                        n_eos += 1  # EOS inside thinking: let it carry on
                        after_eos = True
                        continue
                    break
                reasoning, answer_part = split_turn(text)
                entry = {"turn": turns, "tokens": turn_tokens, "closed_think": answer_part is not None}
                if nudge_at is not None:
                    entry["think_nudge"] = nudge_at
                    if act_forced:
                        entry["think_forced"] = True
                turn_log.append(entry)
                if out_of_budget or answer_part is None:
                    # Hard cap, context limit or thinking that never closes: force a submit_answer call; the
                    # session answer and salvage follow below.
                    stop = out_of_budget or "unclosed_think"
                    if program_mode:  # no forced typed answer: the program fallbacks below decide
                        prompt += text
                        break
                    base = text + FORCE_THINK if answer_part is None else text[:len(text) - len(answer_part)] + "\n\n"
                    forced = base + FORCE_CALL
                    fits = (estimate_tokens(known_tokens, known_chars, prompt + forced) + cfg.force_submit_tokens
                            + 64 <= cfg.context)
                    if cfg.force_submit_tokens and fits:
                        c = await complete(prompt + forced, cfg.force_submit_tokens, ["</parameter>"])
                        expr = c.text.strip()
                        entry["forced_expr"] = expr[:200]
                        text = forced + c.text + "\n</parameter>\n</function>\n</tool_call>"
                        if expr:
                            out, grid = await submit(expr)
                            entry["forced_result"] = out[:200]
                            if grid is not None:
                                pred, source, answer_code = grid, "forced", expr
                    prompt += text
                    break
                calls, perrs = parse_tool_calls(answer_part)
                if not calls and not perrs and program_mode:  # text reply: typed answers are not accepted
                    content = answer_part
                    if n_nudges < cfg.max_nudges:
                        n_nudges += 1
                        entry["nudge"] = True
                        prompt += text + "<|im_end|>\n" + render_user(PROGRAM_NUDGE + budget_note(cfg, used)) \
                            + GEN_PROMPT
                        continue
                    prompt += text + "<|im_end|>\n"
                    stop = "final"
                    break
                if not calls and not perrs:  # a final text reply
                    content = answer_part
                    candidate = parse(answer_part)
                    if valid_grid(candidate):
                        pred, source = candidate, "parse"
                    else:
                        candidate = lenient_parse(answer_part)
                        if candidate is not None:
                            pred, source = candidate, "lenient"
                    prompt += text + "<|im_end|>\n"
                    stop = "final"
                    break
                outputs, accepted, cap_hit = [], None, False
                n_parse_err += len(perrs)
                for name, params in calls:
                    n_calls += 1
                    code = params.get("code")
                    if accepted is not None:
                        outputs.append("[skipped: answer already accepted]")
                        continue
                    if not isinstance(code, str) or name not in ("run_python", answer_tool):
                        n_parse_err += 1
                        n_tool_err += 1
                        outputs.append(f"[tool error: unknown tool {name!r} or missing `code` parameter; tools are "
                                       f"run_python(code) and {answer_tool}(code)]"
                                       + (" Typed grids and submit_answer are not accepted: the answer must be a "
                                          "program." if program_mode and name == "submit_answer" else ""))
                        continue
                    if cap_hit:
                        outputs.append("[skipped: submission limit reached]")
                        continue
                    if name == "run_python":
                        if "def solve" in code:
                            last_solve_cell = code
                        try:
                            result = await ex.run(_execute, code)
                        except ExecutorTimeout as exc:
                            result = {"process_status": "timeout", "stdout": "", "state_reset": True,
                                      "stderr": f"[{exc}; the session was restarted]"}
                        n_tool_err += result["process_status"] != "completed"
                        outputs.append(tool_output(result, cfg.max_tool_output))
                        continue
                    if program_mode:
                        rec, rows = await check_program(code)
                        rec["turn"], rec["tokens_used"] = turns, used
                        submissions.append(rec)
                        if rec["test"] is not None and confirm["shown"] is None and (
                                cfg.confirm_test or cfg.test_audit or cfg.lint):
                            report = _review(cfg, train, test_input, rec, code, confirm)
                            if report:  # E020: show the test output (+ audit / lint) once before accepting
                                outputs.append(report)
                                continue
                        if rec["test"] is not None:
                            outputs.append(f"[program accepted: passes all {len(train)} training pairs; test output "
                                           f"{len(rec['test'])}x{len(rec['test'][0])}]")
                            accepted = rec["test"]
                            continue
                        n_tool_err += 1
                        outputs.append(program_feedback(rows, len(submissions), cfg.max_submissions,
                                                        test_error=rec["test_error"],
                                                        rich=cfg.feedback == "rich", train=train))
                        if cfg.lint:
                            from .audit import lint
                            flags = lint(code, train)
                            if flags:
                                rec["lint"] = flags
                                outputs[-1] += ("\nHardcoded constants in your program (often fit the training pairs "
                                                "only):\n" + "\n".join(f"- {f}" for f in flags))
                        if len(submissions) >= cfg.max_submissions:
                            cap_hit = True
                            outputs[-1] += "\n[submission limit reached: the attempt ends]"
                        continue
                    out, grid = await submit(code)
                    if grid is not None and cfg.verify_submit:
                        try:
                            check = await ex.run(check_answer_on_train, code)
                        except ExecutorTimeout:
                            check = None
                        if check and check[0] and code not in forced_codes:
                            forced_codes.add(code)
                            out, grid = (f"[answer not accepted yet: the same expression with test_input = each "
                                         f"training input fails training pairs {check[0]} ({check[1]}). Fix the "
                                         "rule, or call submit_answer again with exactly the same code to submit "
                                         "anyway.]"), None
                        else:
                            submit_verified = None if check is None else not check[0]
                    n_tool_err += grid is None
                    outputs.append(out)
                    if grid is not None:
                        accepted, answer_code = grid, code
                for e in perrs:
                    n_tool_err += 1
                    outputs.append(f"[tool error: {e}]")
                entry["calls"] = [name for name, _ in calls] + ["<malformed>"] * len(perrs)
                if accepted is not None:
                    pred, source, stop = accepted, "program_pass" if program_mode else "submit", "submitted"
                    prompt += text + "<|im_end|>\n"
                    break
                if cap_hit:
                    stop = "submission_cap"
                    prompt += text + "<|im_end|>\n" + render_tool_responses(outputs)
                    break
                outputs[-1] += budget_note(cfg, used)
                prompt += text + "<|im_end|>\n" + render_tool_responses(outputs) + GEN_PROMPT
        except Exception as exc:
            error, stop = f"{type(exc).__name__}: {exc}"[:1000], "error"
        # Program mode fallbacks (also after an error): the best submitted program or the session's `solve`,
        # whichever reproduces more training pairs (ties: the session's solve, the latest).
        if program_mode and pred is None and ex.session is not None:
            try:
                pred, source, program_info = await _program_fallback(ex, train, test_input, cfg, submissions,
                                                                      run_on)
            except Exception as exc:
                program_info = {"fallback_error": f"{type(exc).__name__}: {exc}"[:200]}
        # Fallbacks, also after an error (R2): the session's `answer`, then the session's best function (R7).
        if not program_mode and pred is None and ex.session is not None:
            try:
                out, grid = await submit("answer")
                if grid is not None:
                    pred, source = grid, "session"
                elif cfg.salvage:
                    salvage_info, grid = await ex.run(salvage_answer)
                    if grid is not None:
                        pred, source = grid, "salvage"
            except Exception as exc:
                salvage_info = {"error": f"{type(exc).__name__}: {exc}"[:200]}
        train_check = None
        if cfg.record_train_check and not program_mode and pred is not None and ex.session is not None:
            train_check = await _submit_train_check(ex, answer_code if source in ("submit", "forced") else None, train)
        alternatives = None
        if cfg.record_alternatives and ex.session is not None:
            try:
                alternatives = await _alternatives(ex, train, test_input, cfg, pred, submissions, run_on, submit,
                                                   program_mode)
            except Exception as exc:
                alternatives = [{"error": f"{type(exc).__name__}: {exc}"[:200]}]
    finally:
        await ex.close()
    result = {"pred": pred, "answer_source": source if pred is not None else None, "content": content[-2000:],
              "stop_reason": stop, "turns": turns, "tool_calls": n_calls, "tool_errors": n_tool_err,
              "parse_failures": n_parse_err, "eos_continues": n_eos, "chunks": n_chunks,
              "usage": {"completion": used, "max_prompt": max_prompt, "prompt_total": cache["prompt"],
                        "ideal_cached": cache["ideal"],
                        **({"cached": cache["cached"]} if cache["reported"] else {})}, "turn_log": turn_log,
              "executor_resets": ex.resets, "harness_version": harness_version(cfg),
              **({"served_by": servers} if servers else {}),
              "seconds": round(time.monotonic() - t0, 1)}
    if cfg.turn_think_cap:
        result["think_nudges"], result["think_forced"] = think_stats["nudges"], think_stats["forced"]
        if cfg.think_call_rescue:
            result["think_call_rescued"] = think_stats["rescued"]
    if cfg.confirm_test or cfg.test_audit or cfg.lint:  # the first train-passing test output that was shown
        result["first_tp_pred"] = confirm["shown"]
        if confirm.get("audit") is not None:
            result["audit"] = confirm["audit"]
    if alternatives is not None:
        result["alternatives"] = alternatives
    if salvage_info is not None:
        result["salvage"] = salvage_info
    if cfg.verify_submit:
        result["submit_verified"] = submit_verified if source == "submit" else None
        result["submit_rejections"] = len(forced_codes)
    if program_mode:
        final = program_info.get("record") if source != "program_pass" else submissions[-1]
        if source == "program_pass":
            program_info = {"train_pass": True, "train_frac": 1.0}
        result.update({
            "train_pass": program_info.get("train_pass") if pred is not None else None,
            "train_frac": program_info.get("train_frac") if pred is not None else None,
            "n_submissions": len(submissions), "nudges": n_nudges,
            "program": ((final or {}).get("code") or (last_solve_cell if source == "session_solve" else None))
            if pred is not None else None,
            "submissions": [{k: v for k, v in r.items() if k not in ("code", "test")} | {"code_chars": len(r["code"])}
                            for r in submissions]})
        if "fallback_error" in program_info:
            result["fallback_error"] = program_info["fallback_error"]
    elif cfg.record_train_check:
        result.update(train_check or {"train_pass": None, "train_frac": None})
    if error:
        result["error"] = error
    return prompt, result


async def _program_fallback(ex, train, test_input, cfg, submissions, run_on):
    """(pred, source, info) at the end of a program-mode attempt without an accepted program."""
    cands = []  # (train_frac, order, kind, record)
    for i, rec in enumerate(submissions):
        cands.append((rec["train_frac"], i, "program_best_fail", rec))
    # the session's own solve (may use session helpers): train-check it the same way (host-side comparison)
    outs = []
    for pair in train:
        try:
            o = await ex.run(run_session_solve, pair["input"], cfg.program_timeout)
        except ExecutorTimeout:
            o = {"error": "timeout"}
        if o is None:
            break
        outs.append(o)
    if len(outs) == len(train):
        rows = score_outputs(outs, train)
        frac = sum(r["ok"] for r in rows) / len(train)
        cands.append((frac, len(submissions), "session_solve", {"train_frac": frac, "code": None}))
    for frac, _, kind, rec in sorted(cands, key=lambda c: (c[0], c[1]), reverse=True):
        if kind == "session_solve":
            try:
                out = await ex.run(run_session_solve, test_input, cfg.program_timeout)
            except ExecutorTimeout:
                out = None
        else:
            out = (await run_on(rec["code"], [test_input]))[0]
        if out and "out" in out and valid_grid(out["out"]):
            return out["out"], kind, {"train_pass": frac == 1.0, "train_frac": frac, "record": rec}
    return None, None, {"candidates": len(cands)}


async def _alternatives(ex, train, test_input, cfg, pred, submissions, run_on, submit, program_mode):
    """E020 free attempt_2 (bookkeeping only, no model tokens; the attempt's answer is unchanged): other test grids
    the attempt produced, distinct from `pred` and from each other, with their train fraction where known.
    program mode: earlier failing submissions run on test_input, and the session's solve;
    submit mode: the session's `answer`, `solve(test_input)` and the salvage function."""
    seen = [pred] if pred is not None else []
    out = []

    def add(src, grid, frac):
        if grid is not None and valid_grid(grid) and grid not in seen:
            seen.append(grid)
            out.append({"src": src, "pred": grid, "train_frac": frac})

    if program_mode:
        for i, rec in sorted(enumerate(submissions), key=lambda x: -x[1]["train_frac"]):
            if rec.get("test") is not None:
                add(f"submission_{i}", rec["test"], rec["train_frac"])
            else:
                o = (await run_on(rec["code"], [test_input]))[0]
                add(f"submission_{i}", o.get("out"), rec["train_frac"])
        outs = []
        for pair in train:
            try:
                o = await ex.run(run_session_solve, pair["input"], cfg.program_timeout)
            except ExecutorTimeout:
                o = None
            if o is None:
                break
            outs.append(o)
        if len(outs) == len(train):
            frac = sum(r["ok"] for r in score_outputs(outs, train)) / len(train)
            try:
                o = await ex.run(run_session_solve, test_input, cfg.program_timeout)
            except ExecutorTimeout:
                o = None
            add("session_solve", (o or {}).get("out"), frac)
    else:
        for expr in ("answer", "solve(test_input)"):
            _, grid = await submit(expr)
            frac = None
            if grid is not None and expr != "answer":
                chk = await _submit_train_check(ex, expr, train)
                frac = chk["train_frac"]
            add(expr, grid, frac)
        try:
            info, grid = await ex.run(salvage_answer)
            frac = info.get("train_matched") / info["train_pairs"] if info and info.get("train_pairs") else None
            add("salvage:" + str((info or {}).get("function")), grid, frac)
        except Exception:
            pass
    return out


async def _submit_train_check(ex, code, train):
    """Submit mode, bookkeeping only: re-evaluate the answer expression with test_input bound to each training
    input (E011 check). train_pass None when the expression does not read test_input (e.g. `answer`)."""
    if not code:
        return {"train_pass": None, "train_frac": None}
    try:
        check = await ex.run(check_answer_on_train, code)
    except Exception:
        check = None
    if check is None:
        return {"train_pass": None, "train_frac": None}
    return {"train_pass": not check[0], "train_frac": 1 - len(check[0]) / len(train)}


def harness_version(cfg):
    v5 = any(getattr(cfg, k) != v for k, v in V5_DEFAULTS.items()) or cfg.prompt in V5_PROMPTS
    return HARNESS_VERSION_V5 if v5 else HARNESS_VERSION


def config_dict(cfg, model):
    d = {**asdict(cfg), "model": model, "harness_version": harness_version(cfg)}
    for key, default in V5_DEFAULTS.items():  # E020 options: recorded only when set
        if d[key] == default:
            del d[key]
    if "turn_think_grace" in d and not cfg.turn_think_cap:
        del d["turn_think_grace"]
    for key, default in (("soft_wording", "gentle"), ("context", CONTEXT), ("salvage", True), ("exec_margin", 35)):
        if d[key] == default:  # keep configs of runs made before the option existed unchanged
            del d[key]
    if cfg.answer_mode != "program":  # program-only options
        for key in ("max_submissions", "program_timeout", "max_nudges"):
            del d[key]
    if not cfg.record_train_check:
        del d["record_train_check"]
    return d


The ARC Prize official prompt (exact bytes, no trailing newline, so it is written from Python rather than with `%%writefile`).

In [ ]:
ARC_PROMPT = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:"""
open('/kaggle/working/arc_solver/notes/prompts/arcprize_official_system_prompt.txt', 'w').write(ARC_PROMPT)

### Solver

Phases: fallback `submission.json` -> executor self-test -> offline install of vLLM from the attached wheel source (uv, pip fallback; versions logged) -> model preload -> vLLM server (retried once) -> scheduler with the tool agent -> summary. A watchdog restarts a dead server; an emergency timer guarantees a valid file.

In [ ]:
%%writefile /kaggle/working/arc_solver/solver.py
"""ARC Prize 2026 (ARC-AGI-2) offline solver: Qwen3.8-27B-FP8 on vLLM (TP=4, 4xL4), no tools.

One self-contained file; kaggle/build_notebook.py embeds it in the Kaggle notebook. Modes (env ARC_MODE):

- submit:   full run on the competition test file (used automatically on the competition rerun).
- fallback: write a valid fallback submission.json and exit (used for the commit run before submitting).
- bench:    test run: provenance hashes, server start, long-context decode benchmark, short end-to-end run on
            evaluation tasks with a near deadline, scoring against the evaluation solutions.
            ARC_HASH=0 / ARC_BENCH_DECODE=0 skip the first two (validation run: ARC_E2E_TASKS, ARC_E2E_MIN).

Robustness rules: a complete, valid submission.json is written before anything else and rewritten atomically
after every finished attempt; every failure path keeps the last valid file.

Prompt, parser and Qwen raw-template rendering follow evals/arc_harness/arcprize.py and evals/run_qwen_generate.py
(ARC Prize official prompt, tier xhigh, temp 1.0, top_p 0.95, top_k 20, forced final answer at the budget).
"""
import asyncio
import glob
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

T0 = float(os.environ.get("ARC_T0", time.time()))  # notebook start (set by the first cell)
MODE = os.environ.get("ARC_MODE", "submit")
WORK = "/kaggle/working"
WORK = os.environ.get("ARC_WORK", WORK)
OUT_SUB = os.path.join(WORK, "submission.json")
LOG_DIR = os.path.join(WORK, "arc_logs")
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
SERVED = os.environ.get("ARC_MODEL_ID", "qwen3.8-27b")
# Off-Kaggle validation: use an already running OpenAI-compatible server and skip install / launch.
EXTERNAL = os.environ.get("ARC_EXTERNAL_SERVER")  # e.g. http://localhost:30000/v1
if EXTERNAL:
    BASE = EXTERNAL.rstrip("/").removesuffix("/v1")
INPUT = os.environ.get("ARC_INPUT", "/kaggle/input")

# ---------------------------------------------------------------- settings (env overridable)

def envf(name, default):
    return type(default)(os.environ.get(name, default))

FORCE_AT_S = envf("ARC_FORCE_AT_S", 11.0 * 3600)    # force final answers for all in-flight attempts
HARD_STOP_S = envf("ARC_HARD_STOP_S", 11.5 * 3600)  # stop everything, final write (notebook guard: 11.75 h)
B_MAX1 = envf("ARC_B_MAX1", 96000)                  # attempt-1 budget cap (thinking + answer)
B_MIN1 = envf("ARC_B_MIN1", 16000)
B_MAX2 = envf("ARC_B_MAX2", 64000)
B_MIN2 = envf("ARC_B_MIN2", 32000)                  # attempt 2 is launched only if it can get this much
FINAL_TOKENS = envf("ARC_FINAL_TOKENS", 3000)
RATE_PRIOR = envf("ARC_RATE_PRIOR", 350.0)          # tok/s before the run measures its own rate
MAX_MODEL_LEN = envf("ARC_MAX_MODEL_LEN", 131072)
MAX_SEQS = envf("ARC_MAX_SEQS", 64)
CONCURRENCY = envf("ARC_CONCURRENCY", 0)            # 0 = from KV capacity
# KV tokens per sequence when sizing concurrency = p90 context of a live tool attempt (E014/E019 rule: KV / p90,
# not the mean): 1.40M (vLLM 0.23) / 48k = 29. v2 used 40000 (-> 35).
TOKENS_PER_SEQ_KV = envf("ARC_KV_PER_SEQ", 48000)
MTP = envf("ARC_MTP", 0)                            # speculative tokens (0 = off)
TIER = os.environ.get("ARC_TIER", "xhigh")
TEMPERATURE = envf("ARC_TEMPERATURE", 1.0)
GPU_UTIL = envf("ARC_GPU_UTIL", 0.95)
TP = int(envf("ARC_TP", 4))                          # tensor parallel size; main() sets it from the GPUs present
SERVER = os.environ.get("ARC_SERVER", "vllm")        # vllm (user's vllm-0.23.0 cu130 source) | sglang (ours)
AGENT = os.environ.get("ARC_AGENT", "tools")         # tools (E012 agent_qwen) | plain (v1 no-tool prompt)
TOOLS_CAP = envf("ARC_TOOLS_CAP", 48000)             # E012: hard cap per attempt (min with the adaptive share)
TOOLS_SOFT = envf("ARC_TOOLS_SOFT", 24000)
TOOLS_TIER = os.environ.get("ARC_TOOLS_TIER", "medium")
TOOL_TIMEOUT = envf("ARC_TOOL_TIMEOUT", 30.0)
TOOLS_ANSWER_MODE = os.environ.get("ARC_TOOLS_ANSWER_MODE", "submit")  # E019: "program" = submit_program only
# ---- v3 (E021) toggles; every default reproduces v2 except the robustness items (memory cap, KV/p90 concurrency)
TOOLS_PROMPT = os.environ.get("ARC_TOOLS_PROMPT", "")  # "" = lean2sub (submit) / lean2prog (program); lean2progroles
TASK_SUMMARY = envf("ARC_TASK_SUMMARY", 0)          # E017/E020: host-computed perception pack in the first message
HELPERS = envf("ARC_HELPERS", 0)                    # E017/E020: arc_helpers preloaded in session + programs
FEEDBACK = os.environ.get("ARC_FEEDBACK", "basic")  # program mode: basic | rich (E020)
TURN_THINK_CAP = envf("ARC_TURN_THINK_CAP", 0)      # E020: thinking tokens per turn before the "run code" nudge
TURN_THINK_GRACE = envf("ARC_TURN_THINK_GRACE", 2048)  # then close the thinking into a run_python call
THINK_CALL_RESCUE = envf("ARC_THINK_CALL_RESCUE", 1)  # E020/E021: after a nudge, run a <tool_call> written inside
                                                    # still-open thinking (only acts with the think cap on)
FREE_ATTEMPT2 = envf("ARC_FREE_ATTEMPT2", 0)        # 1: attempt_2 from the attempt's own alternatives (no tokens)
ATTEMPT2 = os.environ.get("ARC_ATTEMPT2", "legacy")  # legacy (v2/v1 rule) | off | gated (E021)
ATTEMPT2_UNKNOWN = envf("ARC_ATTEMPT2_UNKNOWN", 1)  # gated: also retry submitted answers without a train check (E012)
ATTEMPT2_SAFETY = envf("ARC_ATTEMPT2_SAFETY", 0.85)  # gated: budget <= safety x per-attempt speed x time to force
SESSION_MEM_MB = envf("ARC_SESSION_MEM_MB", 2048)   # session kernel RLIMIT_AS = VmSize + this; programs min(3072, .)
                                                    # 0 = v2 (only the executor's absolute 4 GB RLIMIT_AS)
LOCAL_FORCE_MIN = envf("ARC_LOCAL_FORCE_MIN", 40.0)  # local validation: force at max(T0 + this, ready + 20 min)
LOCAL_FORCE_CAP_MIN = envf("ARC_LOCAL_FORCE_CAP_MIN", 60.0)  # ... but never later than T0 + this
LOCAL_READY_MIN = envf("ARC_LOCAL_READY_MIN", 0.0)  # > 0: local force = server ready + this (overrides the above)
LOCAL_HARD_MIN = envf("ARC_LOCAL_HARD_MIN", 8.0)    # local hard stop = force + this
LOCAL_TASKS = os.environ.get("ARC_LOCAL_TASKS", "")  # comma-separated subset of the local test set
LOCAL_SOURCE = os.environ.get("ARC_LOCAL_SOURCE", "local")  # local: arc2-local-testcases; eval: competition evaluation set
                                                    # (load probe: real scheduler on public-eval pairs, self-scored)
PUBLIC_TEST_IDS_SHA256 = "4cab35dea7bde6ee39bbfcfdcfc101153e19f333c14b68c68b84a7a3278c265e"  # public test file
PHASES = {}  # phase timings (s since T0), written to the summary
PKG = os.path.join(WORK, "arc_solver")  # evals/arc_harness + notes/prompts + solver.py, written by the notebook

PROMPT_TEMPLATE = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:"""  # noqa: F821  (ARC Prize official prompt, injected by build_notebook.py)

CLOSE = "</think>"
FORCE_TEXT = "\n\nI have to give my final answer now.\n</think>\n\n"
LOW_YIELD = 20
TIER_SYSTEM = {
    "xhigh": "Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, "
             "consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final "
             "answer.",
    "medium": None,
}


def log(*a):
    el = time.time() - T0
    print(f"[{el / 3600:5.2f}h {time.strftime('%H:%M:%S')}]", *a, flush=True)


# ---------------------------------------------------------------- prompt / parse (verbatim from arc_harness)

def build_prompt(train, test_input):
    ex = ""
    for i, pair in enumerate(train):
        ex += f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(pair["input"]) + "\n\n"
        ex += "OUTPUT: \n\n" + json.dumps(pair["output"]) + "\n\n"
    return PROMPT_TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))


def render_prompt(user_content, tier=TIER):
    s = ""
    if TIER_SYSTEM.get(tier):
        s += f"<|im_start|>system\n{TIER_SYSTEM[tier]}<|im_end|>\n"
    return s + f"<|im_start|>user\n{user_content.strip()}<|im_end|>\n<|im_start|>assistant\n<think>\n"


def answer_section(gen):
    i = gen.rfind(CLOSE)
    return None if i < 0 else gen[i + len(CLOSE):]


def is_grid(x):
    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)


def extract_from_boxed(s):
    m = re.search(r"\\boxed\{(.*?)\}", s, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            return g if is_grid(g) else None
        except json.JSONDecodeError:
            return None
    return None


def backscan_json_parser(s):
    end = max(s.rfind("]"), s.rfind("}"))
    if end < 0:
        return None
    close = s[end]
    open_ = "[" if close == "]" else "{"
    depth = 1
    for i in range(end - 1, -1, -1):
        depth += (s[i] == close) - (s[i] == open_)
        if depth == 0:
            try:
                g = json.loads(s[i:end + 1])
                return g if is_grid(g) else None
            except json.JSONDecodeError:
                return None
    return None


def parse(s):
    return extract_from_boxed(s or "") or backscan_json_parser(s or "")


def valid_grid(grid):
    return (isinstance(grid, list) and 1 <= len(grid) <= 30
            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)
            and len({len(row) for row in grid}) == 1
            and all(type(v) is int and 0 <= v <= 9 for row in grid for v in row))


# ---------------------------------------------------------------- paths

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None


def comp_file(name):
    return (find_one(f"{INPUT}/competitions/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/**/{name}"))


def model_dir():
    for pat in [f"{INPUT}/models/mikedan7/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/**/qwen3-8-27b-fp8*/**/config.json"]:
        p = find_one(pat)
        if p:
            return os.path.dirname(p)
    raise FileNotFoundError("model dir")


def wheel_dir():
    """Our own wheel sources (attached as kernel sources): SGLang 0.5.20 cu130 or the user's vLLM 0.23.0 cu128."""
    pat = "sglang-0.5.20-*.whl" if SERVER == "sglang" else "vllm-0.23.0*.whl"
    p = find_one(f"{INPUT}/notebooks/chankhavu/**/{pat}") or find_one(f"{INPUT}/**/{pat}")
    if not p:
        raise FileNotFoundError(f"wheel source for {SERVER}")
    return os.path.dirname(p)


# ---------------------------------------------------------------- submission

class Submission:
    def __init__(self, tasks):
        self.tasks = tasks
        self.answers = {}  # (tid, pair) -> {attempt_index: grid}
        self.meta = {}     # (tid, pair) -> {attempt_index: {train_pass, train_frac, answer_source, alternatives}}
        self.sources = {}  # (tid, pair) -> (source of attempt_1, source of attempt_2), set by build()

    def fallback(self, tid, p):
        g = self.tasks[tid]["test"][p]["input"]
        return g if valid_grid(g) else [[0]]

    def _pick(self, tid, p):
        """Top-2 distinct grids with their sources (v3 ranking, arc_harness.selection); v2's rule if that fails."""
        answers = self.answers.get((tid, p), {})
        try:
            from arc_harness import selection as SEL
            meta = self.meta.get((tid, p), {})
            recs = {a: {**meta.get(a, {}), "pred": answers.get(a)} for a in set(answers) | set(meta)}
            top = SEL.select_two(SEL.candidates(recs))
            return [c["grid"] for c in top], [f"a{c['attempt']}:{c['src']}" for c in top]
        except Exception:
            got = [g for _, g in sorted(answers.items()) if valid_grid(g)]
            return got, ["v2"] * len(got)

    def build(self):
        sub = {}
        for tid, task in self.tasks.items():
            lst = []
            for p in range(len(task["test"])):
                got, src = self._pick(tid, p)
                fb = self.fallback(tid, p)
                a1 = got[0] if got else fb
                a2 = next((g for g in got[1:] if g != a1), None)
                s2 = src[1] if a2 is not None else None
                if a2 is None:
                    a2, s2 = (fb, "fallback") if a1 != fb else (a1, "dup")
                self.sources[(tid, p)] = (src[0] if got else "fallback", s2)
                lst.append({"attempt_1": a1, "attempt_2": a2})
            sub[tid] = lst
        return sub

    def write(self):
        sub = self.build()
        tmp = OUT_SUB + ".tmp"
        with open(tmp, "w") as f:
            json.dump(sub, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, OUT_SUB)


def check_submission(path, tasks):
    sub = json.load(open(path))
    assert set(sub) == set(tasks), "task ids differ"
    for tid, task in tasks.items():
        assert len(sub[tid]) == len(task["test"]), tid
        for slot in sub[tid]:
            assert valid_grid(slot["attempt_1"]) and valid_grid(slot["attempt_2"]), tid
    return True


# ---------------------------------------------------------------- environment

def sh(cmd, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or "") + (r.stderr or "")


def driver_env():
    """Host driver libcuda first (images also carry an older compat libcuda); set before any CUDA process starts."""
    real = "/usr/local/nvidia/lib64"
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"


def cuda_toolkit_env(env=None, shim=None):
    """CUDA 13 toolkit from the pip wheels for JIT builds (SGLang's sglang-kernel JIT, FlashInfer, tvm-ffi).

    The image has no /usr/local/cuda/bin/nvcc; our SGLang wheel source installs nvidia-cuda-nvcc/crt/nvvm/cccl/runtime
    into site-packages/nvidia/cu13. JIT builders expect $CUDA_HOME/{bin/nvcc,include,nvvm} and link with
    `-L$CUDA_HOME/lib64 -lcudart`, while the wheels ship lib/ with only libcudart.so.13, so a shim directory of
    symlinks gives the expected layout. Updates `env` (default os.environ); returns the shim path or None."""
    env = os.environ if env is None else env
    shim = shim or os.environ.get("ARC_CUDA_SHIM", "/tmp/cuda13")
    import site
    roots = list(dict.fromkeys(site.getsitepackages() + glob.glob("/usr/local/lib/python3*/dist-packages")
                               + glob.glob("/usr/lib/python3/dist-packages")))
    cu = next((os.path.join(r, "nvidia/cu13") for r in roots if os.path.exists(os.path.join(r, "nvidia/cu13/bin/nvcc"))),
              None)
    if cu is None:
        log("cuda toolkit: no nvidia/cu13/bin/nvcc in", roots)
        return None
    os.makedirs(os.path.join(shim, "lib64"), exist_ok=True)
    for d in ("bin", "include", "nvvm", "lib", "share"):
        src, dst = os.path.join(cu, d), os.path.join(shim, d)
        if os.path.exists(src) and not os.path.lexists(dst):
            os.symlink(src, dst)
    for f in os.listdir(os.path.join(cu, "lib")):
        dst = os.path.join(shim, "lib64", f)
        if not os.path.lexists(dst):
            os.symlink(os.path.join(cu, "lib", f), dst)
    for name in ("cudart", "nvrtc"):
        real = sorted(glob.glob(os.path.join(cu, "lib", f"lib{name}.so.*")))
        dst = os.path.join(shim, "lib64", f"lib{name}.so")
        if real and not os.path.lexists(dst):
            os.symlink(real[0], dst)
    env.update(CUDA_HOME=shim, CUDA_PATH=shim, CUDA_ROOT=shim,
               PATH=os.path.join(shim, "bin") + ":" + env.get("PATH", ""),
               LD_LIBRARY_PATH=(env.get("LD_LIBRARY_PATH", "") + ":" + os.path.join(shim, "lib64")).lstrip(":"),
               TORCH_CUDA_ARCH_LIST=os.environ.get("ARC_CUDA_ARCH", "8.9"))
    return shim


def sglang_jit_env(env):
    """SGLang JIT kernel cache: a writable dir, seeded from a pre-built cache attached as input if there is one."""
    import shutil
    jit = os.environ.get("ARC_SGL_JIT_DIR", "/tmp/sglang_jit")
    seed = find_one(f"{INPUT}/**/sglang_jit_cache/sm89") if os.path.isdir(INPUT) else None
    if seed and not os.path.exists(os.path.join(jit, "sm89")):
        shutil.copytree(seed, os.path.join(jit, "sm89"), symlinks=True)
        log("sglang JIT cache seeded from", seed)
    os.makedirs(jit, exist_ok=True)
    env["SGLANG_JIT_CACHE_DIR"] = jit
    return jit


def gpu_inventory():
    """[(name, memory MiB, compute capability)] of the GPUs actually present (nvidia-smi)."""
    _, out = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader,nounits", timeout=60)
    gpus = []
    for ln in out.splitlines():
        parts = [x.strip() for x in ln.split(",")]
        if len(parts) == 3:
            try:
                gpus.append((parts[0], int(float(parts[1])), float(parts[2])))
            except ValueError:
                pass
    return gpus


def choose_tp(gpus, weights_gb=29.0, reserve_gb=6.0):
    """Tensor-parallel size for Qwen3.8-27B-FP8 on the GPUs present, or 0 if they cannot run it.

    Kaggle once gave a `machine_shape: NvidiaL4` session 2x T4 (E014). FP8 needs SM89+; TP must divide the 4 KV
    heads; each GPU needs its weight shard plus `reserve_gb` for activations, graphs, KV and Mamba state.
    ARC_TP overrides (0 = auto)."""
    forced = int(envf("ARC_TP", 0))
    if forced:
        return forced
    good = [g for g in gpus if g[2] >= 8.9]
    for tp in (4, 2, 1):
        if len(good) >= tp and min(g[1] for g in good[:tp]) / 1024 >= weights_gb / tp + reserve_gb:
            return tp
    return 0


def install_server():
    wd = wheel_dir()
    t = time.time()
    if SERVER == "sglang":
        cmd = f"{sys.executable} -m pip install --no-index --find-links {wd} -r {wd}/requirements.lock"
    else:
        cmd = (f"{sys.executable} -m pip install --no-index --find-links {wd} vllm==0.23.0 flashinfer-python "
               f"flashinfer-jit-cache")
    code, out = sh("uv " + cmd.replace(f"{sys.executable} -m pip install", "pip install --system --python "
                                       + sys.executable), timeout=2400)
    log(f"uv offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-1500:])
        t = time.time()
        code, out = sh(cmd, timeout=2400)
        log(f"pip offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-4000:])
        raise RuntimeError("pip install failed")
    mod = "sglang" if SERVER == "sglang" else "vllm"
    code, out = sh(f"{sys.executable} -c 'import torch; print(torch.version.cuda)'", timeout=600)
    if out.strip().splitlines()[-1:] == ["None"]:
        # A CPU torch of the same version in the image satisfies the pin and is kept: force the CUDA build.
        tw = sorted(glob.glob(f"{wd}/torch-*.whl")) + sorted(glob.glob(f"{wd}/torchvision-*.whl")) + \
            sorted(glob.glob(f"{wd}/torchaudio-*.whl"))
        code, out2 = sh(f"{sys.executable} -m pip install --no-index --force-reinstall --no-deps " + " ".join(tw),
                        timeout=1800)
        log(f"CPU torch found; reinstalled CUDA torch from the wheel source: rc={code}")
    code, out = sh(f"{sys.executable} -c 'import torch, {mod}; print(torch.__version__, torch.version.cuda, "
                   f"torch.cuda.is_available(), torch.cuda.device_count(), {mod}.__version__)'", timeout=600)
    log("torch/server:", out.strip().splitlines()[-1] if out.strip() else out)
    if SERVER == "sglang":
        shim = cuda_toolkit_env()
        code, out = sh(f"{shim}/bin/nvcc --version | tail -2", timeout=120) if shim else (1, "no shim")
        log(f"cuda toolkit shim {shim}: nvcc rc={code} {out.strip().splitlines()[-1:] if out.strip() else ''}")
    if SERVER == "vllm":
        _, ldd = sh(f"ldd $({sys.executable} -c 'import glob, os, vllm; "
                    f"print(glob.glob(os.path.dirname(vllm.__file__) + \"/_C*.so\")[0])') | grep -i 'not found'",
                    timeout=120)
        log("vllm._C missing libraries:", ldd.strip() or "none")
    _, mem = sh("free -g | head -2", timeout=30)
    log("host memory:", " | ".join(mem.strip().splitlines()))
    _, freeze = sh(f"{sys.executable} -m pip freeze", timeout=300)
    open(os.path.join(LOG_DIR, "pip_freeze.txt"), "w").write(freeze)
    log("installed:", " ".join(l for l in freeze.splitlines() if l.split("==")[0].lower() in
                               ("vllm", "torch", "transformers", "flashinfer-python", "triton", "sglang", "numpy")))


def file_hashes(paths, workers=16):
    def h(p):
        s256, s1 = hashlib.sha256(), hashlib.sha1()
        size = os.path.getsize(p)
        s1.update(f"blob {size}\0".encode())
        with open(p, "rb") as f:
            while True:
                b = f.read(1 << 24)
                if not b:
                    break
                s256.update(b)
                s1.update(b)
        return p, {"size": size, "sha256": s256.hexdigest(), "git_sha1": s1.hexdigest()}
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(h, paths))


def preload(mdir, workers=16):
    files = [os.path.join(r, f) for r, _, fs in os.walk(mdir) for f in fs]

    def rd(p):
        with open(p, "rb") as f:
            while f.read(1 << 26):
                pass
    t = time.time()
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(rd, files))
    log(f"preloaded {len(files)} model files in {time.time() - t:.0f}s")


def launch_server(mdir):
    env = os.environ.copy()
    env.update(PYTORCH_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1",
               TRANSFORMERS_OFFLINE="1", OMP_NUM_THREADS="1")
    if SERVER == "sglang":
        cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", mdir, "--served-model-name", SERVED,
               "--host", "127.0.0.1", "--port", str(PORT), "--tp-size", str(TP), "--kv-cache-dtype", "fp8_e4m3",
               "--context-length", str(MAX_MODEL_LEN), "--mem-fraction-static", os.environ.get("ARC_SGL_MEM", "0.85"),
               "--max-running-requests", str(MAX_SEQS), "--trust-remote-code",
               "--attention-backend", os.environ.get("ARC_SGL_ATTN", "triton"), "--log-level", "info",
               "--enable-cache-report"]
        # Radix (prefix) cache is mandatory for the tool agent (every turn re-sends the whole transcript); the
        # hybrid model's Mamba state slots bound the running batch: max_running = slots / slots-per-request
        # (5 with extra_buffer + overlap; 3 with extra_buffer_lazy + SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK=1).
        if os.environ.get("ARC_RADIX", "1") != "1":
            cmd.append("--disable-radix-cache")
        if os.environ.get("ARC_SGL_MAMBA_SLOTS"):
            cmd += ["--max-mamba-cache-size", os.environ["ARC_SGL_MAMBA_SLOTS"]]
        if os.environ.get("ARC_SGL_MAMBA_STRATEGY"):
            cmd += ["--mamba-radix-cache-strategy", os.environ["ARC_SGL_MAMBA_STRATEGY"]]
        if os.environ.get("ARC_SGL_SKIP_LOCK", "0") == "1":
            env["SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK"] = "1"
        if envf("ARC_SGL_HICACHE", 0) > 0:  # host-memory KV offload (hierarchical cache), ratio = host / device
            cmd += ["--enable-hierarchical-cache", "--hicache-ratio", os.environ["ARC_SGL_HICACHE"],
                    "--hicache-write-policy", os.environ.get("ARC_SGL_HICACHE_POLICY", "write_through")]
        cuda_toolkit_env(env)
        sglang_jit_env(env)
    else:
        attn = "TRITON_ATTN"  # FlashInfer fp8-KV kernels need SM90+; L4 is SM89
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", mdir,
               "--served-model-name", SERVED, "--host", "127.0.0.1", "--port", str(PORT),
               "--tensor-parallel-size", str(TP), "--gpu-memory-utilization", str(GPU_UTIL),
               "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_SEQS),
               "--kv-cache-dtype", "fp8", "--attention-backend", attn, "--enable-prefix-caching",
               "--trust-remote-code", "--no-async-scheduling"]
        if MTP > 0:
            cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": MTP})]
        env.update(VLLM_FORCE_ATTN_BACKEND=attn, VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1")
    cmd += json.loads(os.environ.get("ARC_SERVER_EXTRA", "[]"))  # extra server flags (bench / experiments)
    log("server cmd:", " ".join(cmd))
    path = os.path.join(LOG_DIR, f"{SERVER}_server.log")
    fh = open(path, "a")
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            start_new_session=True, env=env)
    return proc, path


def kill_server(proc=None):
    """Kill the server's whole process tree (vLLM renames its workers VLLM::Worker_TP*), then wait for GPU memory."""
    if proc is not None:
        try:
            os.killpg(proc.pid, 9)
        except Exception:
            pass
    if proc is not None:
        for pid in descendants(proc.pid):
            try:
                os.kill(pid, 9)
            except OSError:
                pass
    if ON_KAGGLE:  # name patterns only on Kaggle (a shared host has other users' vllm/sglang processes)
        sh("pkill -9 -f '[v]llm.entrypoints' ; pkill -9 -f '[V]LLM::' ; pkill -9 -f '[s]glang.launch_server' ; "
           "pkill -9 -f '[s]glang::'", timeout=60)
    time.sleep(5)
    for _ in range(24):
        _, out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", timeout=30)
        used = [int(x) for x in out.split() if x.strip().isdigit()]
        if used and max(used) < 2000:
            break
        time.sleep(5)


def server_log_tail(path, n=60):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception:
        return ""


def wait_ready(proc, path, timeout):
    import urllib.request
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            log(f"server exited rc={proc.returncode}\n" + server_log_tail(path))
            return False
        try:
            with urllib.request.urlopen(BASE + "/v1/models", timeout=5) as r:
                if r.status == 200:
                    log(f"server ready in {time.time() - t:.0f}s")
                    return True
        except Exception:
            pass
        time.sleep(5)
    log("server not ready in time\n" + server_log_tail(path))
    return False


def server_capacity(path):
    """(kv_tokens, max_running) from the server log."""
    txt = open(path, errors="ignore").read()
    if SERVER == "sglang":
        kv, mr = None, None
        try:  # runtime values (after the Mamba cap); the log is the fallback
            import urllib.request
            with urllib.request.urlopen(BASE + "/get_server_info", timeout=20) as r:
                st = json.loads(r.read())["internal_states"][0]
            kv = st.get("memory_usage", {}).get("token_capacity")
            mr = st.get("effective_max_running_requests_per_dp")
        except Exception as exc:
            log("get_server_info failed:", exc)
        kvl = re.findall(r"KV Cache is allocated\. dtype: \S+ #tokens: (\d+)", txt)
        mrl = re.findall(r"max_running_requests is capped to (\d+)", txt)
        out = (kv or (int(kvl[-1]) if kvl else None), mr or (int(mrl[-1]) if mrl else None))
    else:
        m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", txt)
        out = (int(m[-1].replace(",", "")) if m else None, None)
    log("server capacity (kv tokens, max running):", out)
    return out


_VLLM_METRICS = {"vllm:prefix_cache_queries_total": "pc_queries", "vllm:prefix_cache_hits_total": "pc_hits",
                 "vllm:num_preemptions_total": "preemptions", "vllm:kv_cache_usage_perc": "kv_usage",
                 "vllm:num_requests_running": "running", "vllm:num_requests_waiting": "waiting",
                 "vllm:prompt_tokens_total": "prompt_tokens", "vllm:generation_tokens_total": "gen_tokens"}


def server_stats(timeout=5.0):
    """Cache / load counters from the server (None if unavailable). Cumulative counters are summed over engines.

    vLLM: /metrics (prefix-cache queries/hits in tokens, preemptions, KV usage fraction, running/waiting).
    SGLang: /v1/loads (running/waiting, token_usage, cumulative uncached prefill tokens, retracted queue)."""
    import urllib.request
    try:
        if SERVER == "sglang" and _SGL_PROM.get("use"):
            return _sglang_prometheus(timeout)
        if SERVER == "sglang":
            try:
                r = urllib.request.urlopen(BASE + "/v1/loads", timeout=timeout)
            except Exception:  # older SGLang (e.g. 0.5.14 locally): /v1/loads fails; use /metrics from now on
                _SGL_PROM["use"] = True
                return _sglang_prometheus(timeout)
            with r:
                ld = json.loads(r.read())["loads"]
            return {"running": sum(x.get("num_running_reqs", 0) for x in ld),
                    "waiting": sum(x.get("num_waiting_reqs", 0) for x in ld),
                    "kv_usage": max(x.get("token_usage", 0.0) for x in ld),
                    "kv_used_tokens": sum(x.get("num_used_tokens", 0) for x in ld),
                    "uncached_prefill": sum(x.get("total_prefill_uncached_tokens", 0) for x in ld),
                    "retracted": sum((x.get("queues") or {}).get("retracted", 0) for x in ld)}
        with urllib.request.urlopen(BASE + "/metrics", timeout=timeout) as r:
            txt = r.read().decode(errors="ignore")
        out = {}
        for line in txt.splitlines():
            if line.startswith("#"):
                continue
            name = line.split("{", 1)[0].split(" ", 1)[0]
            if name in _VLLM_METRICS:
                try:
                    v = float(line.rsplit(" ", 1)[1])
                except ValueError:
                    continue
                k = _VLLM_METRICS[name]
                out[k] = max(out.get(k, 0.0), v) if k == "kv_usage" else out.get(k, 0.0) + v
        return out or None
    except Exception:
        return None


_SGL_PROM = {}
_SGL_METRICS = {"sglang:num_running_reqs": "running", "sglang:num_queue_reqs": "waiting",
                "sglang:token_usage": "kv_usage", "sglang:prompt_tokens_total": "pc_queries",
                "sglang:cached_tokens_total": "pc_hits", "sglang:num_retracted_reqs": "preemptions"}


def _sglang_prometheus(timeout):
    """SGLang /metrics (needs --enable-metrics) in the vLLM field names: cached / prompt tokens, retractions."""
    import urllib.request
    with urllib.request.urlopen(BASE + "/metrics", timeout=timeout) as r:
        txt = r.read().decode(errors="ignore")
    out = {"_prom": True}
    for line in txt.splitlines():
        name = line.split("{", 1)[0].split(" ", 1)[0]
        if line.startswith("#") or name not in _SGL_METRICS:
            continue
        try:
            v = float(line.rsplit(" ", 1)[1])
        except ValueError:
            continue
        k = _SGL_METRICS[name]
        out[k] = max(out.get(k, 0.0), v) if k == "kv_usage" else out.get(k, 0.0) + v
    return out


def cache_line(cur, prev):
    """One progress-line fragment: prefix-cache hit rate over the window, KV usage, preemptions/retractions."""
    if not cur:
        return "server stats n/a"
    prev = prev or {}
    if SERVER == "sglang" and not cur.get("_prom"):
        new = cur["uncached_prefill"] - prev.get("uncached_prefill", 0)
        sent = cur.get("_client_prompt", 0) - prev.get("_client_prompt", 0)
        hit = f"{100 * (1 - new / sent):.1f}%" if sent > 0 else "n/a"
        return (f"cache hit {hit} (uncached prefill {new} of {sent}), running {cur['running']}, "
                f"waiting {cur['waiting']}, "
                f"KV usage {100 * cur['kv_usage']:.1f}%, retracted queue {cur['retracted']}")
    dq = cur.get("pc_queries", 0) - prev.get("pc_queries", 0)
    dh = cur.get("pc_hits", 0) - prev.get("pc_hits", 0)
    hit = f"{100 * dh / dq:.1f}%" if dq > 0 else "n/a"
    return (f"cache hit {hit} (of {dq:.0f} prompt tokens), running {cur.get('running', 0):.0f}, "
            f"waiting {cur.get('waiting', 0):.0f}, KV usage {100 * cur.get('kv_usage', 0):.1f}%, "
            f"preemptions {cur.get('preemptions', 0) - prev.get('preemptions', 0):.0f} "
            f"(total {cur.get('preemptions', 0):.0f})")


# ---------------------------------------------------------------- client

class Client:
    def __init__(self):
        import httpx
        self.last_prompt_tokens = None
        self.prompt_tokens = 0  # summed over finished requests (final usage), for the cache hit rate
        self.cached_tokens = 0  # usage.prompt_tokens_details.cached_tokens when the server reports it
        self.http = httpx.AsyncClient(base_url=BASE, timeout=httpx.Timeout(connect=30.0, read=900.0, write=120.0, pool=None),
                                      limits=httpx.Limits(max_connections=512, max_keepalive_connections=512))

    async def count(self, prompt):
        try:
            r = await self.http.post("/tokenize", json={"model": SERVED, "prompt": prompt})
            return int(r.json()["count"])
        except Exception:
            return int(len(prompt) / 2.5)

    async def stream(self, prompt, max_tokens, deadline, on_tokens, stop=None, info=None):
        """Streamed completion bounded by `deadline` even when no data arrives (finish 'deadline')."""
        parts = {"text": [], "ntok": 0}
        try:
            return await asyncio.wait_for(self._stream(prompt, max_tokens, deadline, on_tokens, stop, info, parts),
                                          timeout=max(1.0, deadline - time.time() + 5))
        except asyncio.TimeoutError:
            return "".join(parts["text"]), parts["ntok"], "deadline"

    async def _stream(self, prompt, max_tokens, deadline, on_tokens, stop, info, parts):
        """Streamed completion. Returns (text, completion_tokens, finish_reason); finish 'deadline' if cut off.

        Breaking out of the stream closes the connection, which makes vLLM abort the request.
        """
        body = {"model": SERVED, "prompt": prompt, "max_tokens": max_tokens, "temperature": TEMPERATURE,
                "top_p": 0.95, "top_k": 20, "stream": True, "skip_special_tokens": False,
                "stream_options": {"include_usage": True, "continuous_usage_stats": True}}
        if stop:
            body["stop"] = stop
        text, ntok, finish = parts["text"], 0, None
        last_usage = None
        async with self.http.stream("POST", "/v1/completions", json=body) as r:
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {(await r.aread())[:300]!r}")
            async for line in r.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[6:]
                if data.strip() == "[DONE]":
                    break
                d = json.loads(data)
                if d.get("error"):
                    raise RuntimeError(f"server error event: {str(d['error'])[:300]}")
                u = d.get("usage")
                got_text = False
                for c in d.get("choices") or []:
                    text.append(c.get("text") or "")
                    got_text = got_text or bool(c.get("text"))
                    if c.get("finish_reason"):
                        finish = c["finish_reason"]
                if u:
                    last_usage = u
                if u and u.get("completion_tokens") is not None:
                    self.last_prompt_tokens = u.get("prompt_tokens") or self.last_prompt_tokens
                    if info is not None and u.get("prompt_tokens"):
                        info["prompt_tokens"] = u["prompt_tokens"]
                    on_tokens(u["completion_tokens"] - ntok)
                    ntok = u["completion_tokens"]
                elif got_text:  # server without continuous usage stats: ~1 token per chunk until usage arrives
                    on_tokens(1)
                    ntok += 1
                parts["ntok"] = ntok
                if time.time() > deadline:
                    finish = "deadline"
                    break
        if last_usage and last_usage.get("prompt_tokens"):
            self.prompt_tokens += last_usage["prompt_tokens"]
            self.cached_tokens += (last_usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
        return "".join(text), ntok, finish or "stop"


# ---------------------------------------------------------------- tool executor (Kaggle hardening)

def session_prelude(mem_mb=None, helpers=None):
    """Extra session prelude (re-run after every restart): the E020 --session-mem-mb cap first, then the helpers."""
    from arc_harness import agent_qwen as AQ
    from arc_harness import executor as EX
    mem_mb = SESSION_MEM_MB if mem_mb is None else mem_mb
    helpers = HELPERS if helpers is None else helpers
    extra = EX.SESSION_MEM_CAP.format(mb=int(mem_mb)) if mem_mb and hasattr(EX, "SESSION_MEM_CAP") else ""
    return extra + (AQ.HELPERS_SRC if helpers else "")


def robust_executor_factory(timeout, prelude_extra=None):
    """E012 LocalExecutor (qwen-tools-3 executor: RLIMIT_AS/CPU/FSIZE, nice, oom_score_adj=1000, SIGINT to the
    process group, retried start with a longer timeout). nice 15 so tool code never starves the model server.
    v3: the session prelude adds the per-session memory cap and (with ARC_HELPERS) the helper functions."""
    from arc_harness import executor as EX
    extra = session_prelude() if prelude_extra is None else prelude_extra
    try:
        return EX.local_factory(timeout=timeout, nice=15, **({"prelude_extra": extra} if extra else {}))
    except TypeError:  # older executor without the hardening knobs
        return EX.local_factory(timeout=timeout)


def build_agent_config(AQ, cap):
    """The AgentConfig every tool attempt runs with (hard cap `cap` = min(ARC_TOOLS_CAP, the attempt's budget))."""
    fields = AQ.AgentConfig.__dataclass_fields__
    extra = {"context": MAX_MODEL_LEN} if "context" in fields else {}
    extra.update(agent_options(fields))
    return AQ.AgentConfig(**extra, tier=TOOLS_TIER, max_total_output=cap, budget_soft=min(TOOLS_SOFT, cap),
                          budget_notes=0.01, budget_warn=0.75, verify_submit=False, max_turns=60,
                          chunk_tokens=16384, force_submit_tokens=256, tool_timeout=TOOL_TIMEOUT,
                          temperature=TEMPERATURE, top_p=0.95, top_k=20)


def agent_options(fields):
    """v3 AgentConfig options from the settings (only fields the bundled agent knows). The defaults give v2's
    config exactly. record_train_check / record_alternatives are bookkeeping: no model tokens."""
    opt = {}
    if TOOLS_ANSWER_MODE == "program":
        opt.update(answer_mode="program", prompt="lean2prog")  # E019 (qwen-tools-4)
    if TOOLS_PROMPT:
        opt["prompt"] = TOOLS_PROMPT
    if TASK_SUMMARY:
        opt["task_summary"] = True
    if HELPERS:
        opt["helpers"] = True
    if FEEDBACK != "basic" and TOOLS_ANSWER_MODE == "program":
        opt["feedback"] = FEEDBACK
    if TURN_THINK_CAP:
        opt.update(turn_think_cap=int(TURN_THINK_CAP), turn_think_grace=int(TURN_THINK_GRACE))
        if THINK_CALL_RESCUE:
            opt["think_call_rescue"] = True
    if FREE_ATTEMPT2:
        opt["record_alternatives"] = True
    if (FREE_ATTEMPT2 or ATTEMPT2 == "gated") and TOOLS_ANSWER_MODE != "program":
        opt["record_train_check"] = True  # train check of the submitted expression (ranking, retry gating)
    return {k: v for k, v in opt.items() if k in fields}


# ---------------------------------------------------------------- scheduler

class Scheduler:
    def __init__(self, tasks, sub, client, force_at, hard_stop, concurrency, attempts=2, tag="run"):
        self.tasks, self.sub, self.client = tasks, sub, client
        self.force_at, self.hard_stop = force_at, hard_stop
        self.concurrency, self.attempts = concurrency, attempts
        self.pending1 = []
        self.pending2 = []
        self.inflight = {}  # job -> [budget, used]
        self.tokens = 0
        self.rate_samples = []  # (time, total tokens)
        self.gen_start = None
        self.records = []
        self.rec_path = os.path.join(LOG_DIR, f"attempts_{tag}.jsonl")
        self.launch_margin = 120  # no new attempts in the last 2 min before forcing
        self.retries = {}
        self.server_down = lambda: False  # set by main(): True while the vLLM server is dead or restarting
        self.cache_stats = []  # periodic server cache/load counters (summary)
        self.priority = {}  # gated attempt 2: job -> selection.retry_priority key
        self.gated = {"eligible": 0, "started": 0, "dropped_time": 0}

    def may_get_work(self):
        """Gated mode: a worker with nothing to do now waits while first attempts can still queue retries."""
        if ATTEMPT2 != "gated" or time.time() >= self.force_at - self.launch_margin:
            return False
        return bool(self.pending1 or self.pending2 or any(j[2] == 0 for j in self.inflight))

    def on_tokens(self, n):
        self.tokens += n
        if "first_token" not in PHASES and n > 0:
            PHASES["first_token"] = round(time.time() - T0)

    def rate(self):
        now = time.time()
        self.rate_samples.append((now, self.tokens))
        self.rate_samples = [s for s in self.rate_samples if now - s[0] < 1800] or [(now, self.tokens)]
        t_old, k_old = self.rate_samples[0]
        if self.gen_start is None or now - self.gen_start < 900 or now - t_old < 300:
            return RATE_PRIOR
        return max(50.0, (self.tokens - k_old) / (now - t_old))

    def capacity_left(self):
        cap = self.rate() * max(0.0, self.force_at - time.time())
        return cap - sum(max(0, b - u) for b, u in self.inflight.values())

    def attempt_speed(self):
        """Generated tokens per second of one attempt (tool time included): median over finished first attempts
        that ran >= 60 s, else the total rate / concurrency."""
        sp = sorted(r["tokens"] / r["elapsed_s"] for r in self.records
                    if r.get("attempt") == 0 and (r.get("elapsed_s") or 0) >= 60 and r.get("tokens"))
        if len(sp) >= 5:
            return sp[len(sp) // 2]
        return self.rate() / max(1, self.concurrency)

    def gated_budget(self):
        """(budget, reason) for a gated second attempt now; budget 0 if it cannot start now."""
        left_s = self.force_at - self.launch_margin - time.time()
        by_time = ATTEMPT2_SAFETY * self.attempt_speed() * left_s
        cap = min(B_MAX2, TOOLS_CAP) if AGENT == "tools" else B_MAX2
        budget = min(cap, self.capacity_left(), by_time)
        if by_time < B_MIN2:
            return 0, "time"
        return (int(budget), "ok") if budget >= B_MIN2 else (0, "capacity")

    def next_job(self):
        """(job, budget) or None. Attempt-1 jobs first; attempt 2 only with spare capacity."""
        if self.pending1:
            job = self.pending1.pop(0)
            share = self.capacity_left() / (len(self.pending1) + 1)
            return job, int(min(B_MAX1, max(B_MIN1, share)))
        if ATTEMPT2 == "off":
            return None
        if ATTEMPT2 == "gated":
            if not self.pending2:
                return None
            budget, why = self.gated_budget()
            if why == "time":  # time only shrinks: no gated attempt can start any more
                log(f"gated attempt 2: no time left for {len(self.pending2)} queued pairs; dropping them")
                self.gated["dropped_time"] += len(self.pending2)
                self.pending2.clear()
                return None
            if not budget:
                return None  # capacity frees up as running attempts finish under budget
            self.pending2.sort(key=lambda j: self.priority.get(j, (9,)))
            self.gated["started"] += 1
            return self.pending2.pop(0), budget
        while self.pending2:
            job = self.pending2.pop(0)
            share = self.capacity_left() / (len(self.pending2) + 1)
            if share < B_MIN2:
                continue
            return job, int(min(B_MAX2, share))
        return None

    async def attempt(self, job, budget):
        if AGENT == "tools":
            return await self.attempt_tools(job, budget)
        return await self.attempt_plain(job, budget)

    async def attempt_tools(self, job, budget):
        """One E012 tool-agent attempt (agent_qwen.solve + LocalExecutor) with deadline forcing.

        DeadlineLLM streams each completion; past force_at it cuts the stream and reports the attempt's budget as
        used, so the agent goes straight to its forced submit_answer (then the session `answer`). The forced call
        (stop=["</parameter>"]) may run until the hard stop."""
        import dataclasses
        from arc_harness import agent_qwen as AQ
        tid, p, a = job
        task = self.tasks[tid]
        cap = int(min(TOOLS_CAP, budget))
        cfg = build_agent_config(AQ, cap)
        sch, client = self, self.client
        stats = {"tokens": 0, "requests": 0, "deadline_cuts": 0, "stream_errors": 0}

        class DeadlineLLM:
            async def complete(self, prompt, max_tokens, stop=None):
                info = {}
                stats["requests"] += 1
                if stop is None and time.time() >= sch.force_at:
                    stats["deadline_cuts"] += 1
                    return AQ.Completion("", cap, 0, "length")
                # forced submit expression: must end well before the hard-stop cancel so that submit_answer and
                # the session-answer fallback still run (review S2)
                dl = (max(sch.hard_stop - 180, min(time.time() + 30, sch.hard_stop - 10)) if stop
                      else sch.force_at)

                def cb(n):
                    stats["tokens"] += n
                    sch.on_tokens(n)
                    if job in sch.inflight:
                        sch.inflight[job][1] = stats["tokens"]
                try:
                    text, ntok, finish = await client.stream(prompt, max_tokens, dl, cb, stop=stop, info=info)
                except Exception as exc:
                    stats["stream_errors"] += 1
                    stats["last_stream_error"] = f"{type(exc).__name__}: {exc}"[:300]
                    if sch.server_down():
                        raise  # attempt is requeued once the server is back
                    # otherwise: end the attempt through the forced submit / session answer path
                    return AQ.Completion("", cap, 0, "length")
                if finish == "deadline":
                    stats["deadline_cuts"] += 1
                    return AQ.Completion(text, ntok + cap, info.get("prompt_tokens", 0), "length")
                return AQ.Completion(text, ntok, info.get("prompt_tokens", 0),
                                     "length" if finish == "length" else "stop")

        self.inflight[job] = [cap, 0]
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": cap, "agent": "tools"}
        t0 = time.time()
        try:
            transcript, res = await AQ.solve(DeadlineLLM(), robust_executor_factory(TOOL_TIMEOUT), task["train"],
                                             task["test"][p]["input"], cfg)
        except Exception as exc:
            transcript, res = "", {"pred": None, "error": f"{type(exc).__name__}: {exc}"[:400]}
        finally:
            self.inflight.pop(job, None)
        pred = res.get("pred")
        if hasattr(pred, "tolist"):
            pred = pred.tolist()
        rec.update(tokens=stats["tokens"], requests=stats["requests"], deadline_cuts=stats["deadline_cuts"],
                   stream_errors=stats["stream_errors"], last_stream_error=stats.get("last_stream_error"),
                   answered=pred is not None and valid_grid(pred), pred=pred,
                   forced=res.get("answer_source") in ("forced", "session") or stats["deadline_cuts"] > 0,
                   force_reason=("deadline" if stats["deadline_cuts"] else res.get("stop_reason")),
                   elapsed_s=round(time.time() - t0),
                   **{k: res.get(k) for k in ("answer_source", "stop_reason", "turns", "tool_calls", "tool_errors",
                                              "parse_failures", "eos_continues", "chunks", "seconds", "turn_log",
                                              "train_pass", "train_frac", "alternatives", "think_nudges",
                                              "think_forced", "think_call_rescued", "n_submissions",
                                              "harness_version")})
        if res.get("error"):
            rec["error"] = res["error"]
        try:
            with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
                f.write(transcript)
        except Exception:
            pass
        return rec

    async def attempt_plain(self, job, budget):
        tid, p, a = job
        task = self.tasks[tid]
        raw = render_prompt(build_prompt(task["train"], task["test"][p]["input"]))
        ptok = await self.client.count(raw)
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": budget, "prompt_tokens": ptok}
        self.inflight[job] = [budget, 0]
        gen, used, conts, streak, forced, force_reason, after_eos, chunks = "", 0, 0, 0, False, None, False, []
        t0 = time.time()
        try:
            while True:
                room = MAX_MODEL_LEN - ptok - used - 64
                if forced:
                    mt, dl = min(FINAL_TOKENS, room), self.hard_stop
                else:
                    mt, dl = min(budget - used, room - FINAL_TOKENS), self.force_at
                if mt <= 0 and not forced:
                    forced, force_reason = True, "budget"
                    gen += FORCE_TEXT
                    continue
                if mt <= 0:
                    break
                text, ntok, finish = await self.client.stream(raw + gen, mt, dl, self.on_tokens)
                gen += text
                used += ntok
                self.inflight[job][1] = used
                chunks.append([ntok, finish])
                if after_eos:
                    streak = streak + 1 if ntok < LOW_YIELD else 0
                if forced:
                    break
                if answer_section(gen) is not None:
                    if finish == "length" and used < budget and room - ntok > 64:
                        continue
                    break
                if finish == "deadline" or used >= budget or finish == "length":
                    forced, force_reason = True, ("deadline" if finish == "deadline" else "budget")
                    gen += FORCE_TEXT
                    continue
                # EOS mid-thought: continue, force after two low-yield continuations in a row
                if streak >= 2:
                    forced, force_reason = True, "low_yield_eos"
                    gen += FORCE_TEXT
                    continue
                conts += 1
                after_eos = True
        except Exception as exc:
            rec["error"] = f"{type(exc).__name__}: {exc}"[:400]
        finally:
            self.inflight.pop(job, None)
        content = answer_section(gen)
        pred = parse(content) if content is not None else None
        rec.update(tokens=used, continuations=conts, forced=forced, force_reason=force_reason, chunks=chunks,
                   elapsed_s=round(time.time() - t0), answered=pred is not None and valid_grid(pred), pred=pred,
                   tail=(content or gen[-300:])[-300:])
        with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
            f.write(gen)
        return rec

    async def worker(self, wid):
        while time.time() < self.force_at - self.launch_margin:
            nj = self.next_job()
            if nj is None:
                if self.may_get_work():
                    await asyncio.sleep(5)
                    continue
                return
            job, budget = nj
            rec = await self.attempt(job, budget)
            if "error" in rec and not rec["answered"] and self.server_down():
                # server crashed: requeue the job (at most twice) and wait for the watchdog to restart it
                self.retries[job] = self.retries.get(job, 0) + 1
                if self.retries[job] <= 2:
                    (self.pending1 if job[2] == 0 else self.pending2).insert(0, job)
                    log(f"server down; requeued {job}, waiting")
                    while self.server_down() and time.time() < self.force_at - self.launch_margin:
                        await asyncio.sleep(15)
                    continue
            self.records.append(rec)
            with open(self.rec_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
            tid, p, a = job
            meta = {k: rec.get(k) for k in ("train_pass", "train_frac", "answer_source", "alternatives")}
            if any(v is not None for v in meta.values()):
                self.sub.meta.setdefault((tid, p), {})[a] = meta
            if rec["answered"]:
                self.sub.answers.setdefault((tid, p), {})[a] = rec["pred"]
            if rec["answered"] or rec.get("alternatives"):
                self.sub.write()
            if a == 0 and self.attempts > 1 and ATTEMPT2 == "legacy":
                self.pending2.append((tid, p, 1))
            elif a == 0 and self.attempts > 1 and ATTEMPT2 == "gated":
                from arc_harness.selection import retry_priority
                key = retry_priority(rec, include_unknown=bool(ATTEMPT2_UNKNOWN))
                if key is not None:
                    self.priority[(tid, p, 1)] = key
                    self.pending2.append((tid, p, 1))
                    self.gated["eligible"] += 1
            log(f"done {tid}.{p}#{a} answered={rec['answered']} tok={rec['tokens']} budget={budget} "
                f"forced={rec['forced']}/{rec['force_reason']} "
                f"{rec['elapsed_s']}s {rec.get('error', '')}")

    async def monitor(self):
        last_t, last_k = time.time(), 0
        prev = None
        while True:
            await asyncio.sleep(120)
            now = time.time()
            r = (self.tokens - last_k) / (now - last_t)
            last_t, last_k = now, self.tokens
            cur = await asyncio.to_thread(server_stats)
            if cur is not None:
                cur["_client_prompt"] = getattr(self.client, "prompt_tokens", 0)
            log(f"progress: {len(self.records)} attempts done, in flight {len(self.inflight)}, pending1 "
                f"{len(self.pending1)}, pending2 {len(self.pending2)}"
                + (f" (gated {self.gated}, attempt speed {self.attempt_speed():.1f} tok/s)" if ATTEMPT2 == "gated"
                   else "") + f", tokens {self.tokens}, window {r:.0f} tok/s, "
                f"rate est {self.rate():.0f} tok/s, avg {self.tokens / max(1, now - self.gen_start):.0f} tok/s | "
                + cache_line(cur, prev))
            if cur is not None:
                self.cache_stats.append({"t": round(now - T0), **cur})
                prev = cur

    async def run(self, jobs1):
        self.pending1 = list(jobs1)
        self.gen_start = time.time()
        mon = asyncio.create_task(self.monitor())
        try:
            await asyncio.wait_for(asyncio.gather(*(self.worker(i) for i in range(self.concurrency))),
                                   timeout=max(60, self.hard_stop - time.time()))
        except asyncio.TimeoutError:
            log("hard stop reached; cancelling in-flight attempts")
        mon.cancel()
        el = time.time() - self.gen_start
        log(f"scheduler finished: {len(self.records)} attempts, {self.tokens} tokens in {el / 60:.1f} min "
            f"({self.tokens / max(1, el):.0f} tok/s)")


def order_jobs(tasks):
    """Attempt-1 jobs, shortest prompts first (approximated by character length)."""
    jobs = [(tid, p, 0) for tid, t in tasks.items() for p in range(len(t["test"]))]
    size = {tid: len(json.dumps(t["train"])) + len(json.dumps(t["test"])) for tid, t in tasks.items()}
    return sorted(jobs, key=lambda j: (size[j[0]], j[0], j[1]))


# ---------------------------------------------------------------- benchmark (test runs only)

async def bench_decode(client, prefixes, n_seqs, gen_tokens, tasks, max_chars=None, n_unique=None):
    """Decode throughput at long context: real ARC prompts + real Qwen trace prefixes, n_seqs in parallel.

    n_unique: number of distinct prompts (sequence i uses prompt i % n_unique), so repeats can hit a prefix cache.
    """
    total = {"tok": 0}
    prompts = []
    for i in range(n_seqs):
        pr = prefixes[i % (n_unique or len(prefixes))]
        t = tasks[pr["task_id"]]
        prompts.append(render_prompt(build_prompt(t["train"], t["test"][pr["pair"]]["input"]))
                       + pr["text"][:max_chars])
    ctx = [await client.count(p) for p in prompts]
    log(f"bench: {n_seqs} seqs, context tokens mean {sum(ctx) / len(ctx):.0f} min {min(ctx)} max {max(ctx)}")
    first = {}
    t_start = time.time()

    def cb(n):
        total["tok"] += n

    async def one(i):
        def cbi(n):
            if i not in first:
                first[i] = time.time()
            cb(n)
        return await client.stream(prompts[i], gen_tokens, time.time() + 3600, cbi)

    samples = []

    async def sampler():
        while True:
            await asyncio.sleep(10)
            samples.append((time.time(), total["tok"]))
    s = asyncio.create_task(sampler())
    res = await asyncio.gather(*(one(i) for i in range(n_seqs)), return_exceptions=True)
    s.cancel()
    t_end = time.time()
    t_all_first = max(first.values()) if first else t_end
    dec = [x for x in samples if x[0] >= t_all_first]
    steady = (dec[-1][1] - dec[0][1]) / (dec[-1][0] - dec[0][0]) if len(dec) >= 2 else None
    errs = [repr(r)[:200] for r in res if isinstance(r, Exception)]
    out = {"n_seqs": n_seqs, "ctx_mean": sum(ctx) / len(ctx), "gen_tokens": gen_tokens, "total_tokens": total["tok"],
           "wall_s": t_end - t_start, "prefill_phase_s": t_all_first - t_start, "steady_decode_tok_s": steady,
           "overall_tok_s": total["tok"] / (t_end - t_start), "errors": errs}
    log("bench result:", json.dumps(out))
    return out


def score(sub, solutions, tasks):
    tot, pairs = 0.0, 0
    for tid, t in tasks.items():
        n = len(t["test"])
        ok = 0
        for p in range(n):
            s = sub[tid][p]
            ok += solutions[tid][p] in (s["attempt_1"], s["attempt_2"])
        tot += ok / n
        pairs += n
    return tot, len(tasks)


# ---------------------------------------------------------------- main

def setup_harness():
    """Make the bundled E012 harness importable and self-test the executor; False -> use the plain agent."""
    try:
        sys.path.insert(0, os.path.join(PKG, "evals"))
        from arc_harness import agent_qwen
        from arc_harness.executor import LocalExecutor, submit_answer
        agent_qwen.CONTEXT = MAX_MODEL_LEN  # the server's context, not the model's 262k
        t = time.time()
        with LocalExecutor([{"input": [[1, 2]], "output": [[2, 1]]}], [[3, 4]], timeout=3) as ex:
            r1 = ex.execute("import numpy as np\nx = np.array(test_input)[:, ::-1]\nprint(x.tolist())")
            r2 = ex.execute("print(x.sum())")
            r3 = ex.execute("1/0")
            r4 = ex.execute("while True:\n    pass")
            r5 = ex.execute("print(x.shape)")
            out, grid = submit_answer(ex, "x")
        ok = (r1["stdout"].strip() == "[[4, 3]]" and r2["stdout"].strip() == "7" and "ZeroDivisionError" in r3["stderr"]
              and r4["process_status"] == "timeout" and r5["stdout"].strip() == "(1, 2)" and grid == [[4, 3]])
        log(f"harness {agent_qwen.HARNESS_VERSION}; executor self-test {'OK' if ok else 'FAILED'} in "
            f"{time.time() - t:.1f}s")
        if not ok:
            log("self-test results:", json.dumps([r1, r2, r3, r4, r5, out])[:1500])
        if ok:
            v3_selftest()
        return ok
    except Exception as exc:
        log(f"harness/executor self-test error: {type(exc).__name__}: {exc}")
        return False


def v3_selftest():
    """Check each enabled v3 feature on CPU before any GPU work; a broken one is switched off (v2 behaviour for
    that feature) and logged loudly. Sets agent_qwen.PROGRAM_MEMORY_MB from the session memory cap."""
    global SESSION_MEM_MB, HELPERS, TASK_SUMMARY, TOOLS_ANSWER_MODE, TOOLS_PROMPT, FREE_ATTEMPT2, TURN_THINK_CAP, \
        THINK_CALL_RESCUE
    from arc_harness import agent_qwen as AQ
    from arc_harness import executor as EX
    fields = AQ.AgentConfig.__dataclass_fields__
    train = [{"input": [[1, 0], [0, 0]], "output": [[0, 1], [0, 0]]}, {"input": [[2, 3]], "output": [[3, 2]]}]
    test_input = [[4, 5], [6, 7]]
    status = {}

    def run(prelude, cells):
        with EX.LocalExecutor(train, test_input, timeout=10, prelude_extra=prelude) as ex:
            return [ex.execute(c) for c in cells]

    if SESSION_MEM_MB:
        try:
            big = int(SESSION_MEM_MB) + 512
            r = run(session_prelude(SESSION_MEM_MB, False),
                    [f"x = bytearray({big} * 2 ** 20)", "print('alive', len(train))"])
            ok = "MemoryError" in r[0]["stderr"] and r[1]["stdout"].strip() == "alive 2" and not r[1].get("state_reset")
            status["session_mem"] = ok
            if ok:
                AQ.PROGRAM_MEMORY_MB = min(AQ.PROGRAM_MEMORY_MB, int(SESSION_MEM_MB))
            else:
                log("SELF-TEST: session memory cap FAILED; using the executor's 4 GB limit only:", json.dumps(r)[:600])
                SESSION_MEM_MB = 0
        except Exception as exc:
            log(f"SELF-TEST: session memory cap error {exc!r}; off")
            SESSION_MEM_MB, status["session_mem"] = 0, False
    if HELPERS:
        try:
            r = run(session_prelude(SESSION_MEM_MB, True), ["print(d8([[1, 2], [3, 4]])['fliplr'].tolist(), len(objects(train[0]['input'])))"])
            ok = "helpers" in fields and bool(AQ.HELPERS_SRC) and r[0]["stdout"].strip() == "[[2, 1], [4, 3]] 1"
            status["helpers"] = ok
            if not ok:
                log("SELF-TEST: helpers FAILED; off:", json.dumps(r)[:400])
                HELPERS = 0
        except Exception as exc:
            log(f"SELF-TEST: helpers error {exc!r}; off")
            HELPERS, status["helpers"] = 0, False
    if TOOLS_ANSWER_MODE == "program":
        try:
            code = ("def solve(grid):\n    return [row[::-1] for row in grid]" if not HELPERS else
                    "def solve(grid):\n    return d8(grid)['fliplr'].tolist()")
            with EX.LocalExecutor(train, test_input, timeout=10, prelude_extra=session_prelude()) as ex:
                outs = [EX.run_program(ex, code, g, 10, AQ.PROGRAM_MEMORY_MB, AQ.HELPERS_SRC if HELPERS else "")
                        for g in [pair["input"] for pair in train] + [test_input]]
                hog = EX.run_program(ex, "def solve(grid):\n    x = bytearray(%d * 2 ** 20)\n    return grid"
                                     % (AQ.PROGRAM_MEMORY_MB + 512), test_input, 10, AQ.PROGRAM_MEMORY_MB)
            ok = ("answer_mode" in fields and [o.get("out") for o in outs] == [p["output"] for p in train]
                  + [[[5, 4], [7, 6]]] and hog.get("kind") in ("MemoryError", "crash"))
            status["program"] = ok
            if not ok:
                log("SELF-TEST: program mode FAILED; back to submit_answer:", json.dumps([outs, hog])[:600])
                TOOLS_ANSWER_MODE, TOOLS_PROMPT = "submit", ""
        except Exception as exc:
            log(f"SELF-TEST: program mode error {exc!r}; back to submit_answer")
            TOOLS_ANSWER_MODE, TOOLS_PROMPT, status["program"] = "submit", "", False
    if TOOLS_PROMPT and TOOLS_PROMPT not in getattr(AQ, "PROMPTS", {}):
        log(f"SELF-TEST: unknown prompt {TOOLS_PROMPT!r}; using the mode default")
        TOOLS_PROMPT = ""
    if TOOLS_PROMPT and ("prog" in TOOLS_PROMPT) != (TOOLS_ANSWER_MODE == "program"):
        log(f"SELF-TEST: prompt {TOOLS_PROMPT!r} does not match answer mode {TOOLS_ANSWER_MODE}; mode default")
        TOOLS_PROMPT = ""
    if TASK_SUMMARY:
        try:
            from task_summary import task_summary
            txt = task_summary(train, test_input)
            ok = "task_summary" in fields and isinstance(txt, str) and len(txt) > 50
            status["task_summary"] = ok
            if not ok:
                log("SELF-TEST: task summary FAILED; off")
                TASK_SUMMARY = 0
        except Exception as exc:
            log(f"SELF-TEST: task summary error {exc!r}; off")
            TASK_SUMMARY, status["task_summary"] = 0, False
    if TURN_THINK_CAP and "turn_think_cap" not in fields:
        log("SELF-TEST: bundled agent has no turn_think_cap; off")
        TURN_THINK_CAP, status["think_cap"] = 0, False
    if TURN_THINK_CAP and THINK_CALL_RESCUE and "think_call_rescue" not in fields:
        log("SELF-TEST: bundled agent has no think_call_rescue; off")
        THINK_CALL_RESCUE, status["think_call_rescue"] = 0, False
    if FREE_ATTEMPT2 and "record_alternatives" not in fields:
        log("SELF-TEST: bundled agent has no record_alternatives; free attempt_2 off")
        FREE_ATTEMPT2, status["free_attempt2"] = 0, False
    try:  # the agent config the attempts will use (also checks the field names)
        cfg = build_agent_config(AQ, int(TOOLS_CAP))
        AQ.initial_prompt(train, test_input, cfg)
        PHASES["agent_config"] = AQ.config_dict(cfg, SERVED)  # what a full-budget attempt runs with
    except Exception as exc:
        log(f"SELF-TEST: agent config/prompt error {exc!r}; dropping all v3 agent options")
        HELPERS = TASK_SUMMARY = TURN_THINK_CAP = FREE_ATTEMPT2 = 0
        TOOLS_ANSWER_MODE, TOOLS_PROMPT = "submit", ""
    PHASES["v3_selftest"] = status
    log(f"v3 self-test {status}; effective: answer_mode={TOOLS_ANSWER_MODE} prompt={TOOLS_PROMPT or 'default'} "
        f"summary={TASK_SUMMARY} helpers={HELPERS} think_cap={TURN_THINK_CAP}/{TURN_THINK_GRACE} "
        f"rescue={THINK_CALL_RESCUE} tier={TOOLS_TIER} feedback={FEEDBACK} "
        f"free_attempt2={FREE_ATTEMPT2} attempt2={ATTEMPT2} session_mem={SESSION_MEM_MB} MB "
        f"program_mem={AQ.PROGRAM_MEMORY_MB} MB")


def is_rerun(test_path):
    """Belt and braces: the env var, or a test file whose task ids differ from the public one."""
    import hashlib
    env = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
    try:
        ids = sorted(json.load(open(test_path)))
        differs = hashlib.sha256(",".join(ids).encode()).hexdigest() != PUBLIC_TEST_IDS_SHA256
    except Exception:
        differs = False
    return env or differs, env, differs


def descendants(pid=None):
    """PIDs of all descendants of `pid` (default: this process), from /proc parent links."""
    pid = os.getpid() if pid is None else pid
    children = {}
    for d in os.listdir("/proc"):
        if d.isdigit():
            try:
                with open(f"/proc/{d}/stat") as f:
                    ppid = int(f.read().rsplit(")", 1)[1].split()[1])
                children.setdefault(ppid, []).append(int(d))
            except (OSError, ValueError, IndexError):
                pass
    out, todo = [], [pid]
    while todo:
        for c in children.get(todo.pop(), []):
            out.append(c)
            todo.append(c)
    return out


ON_KAGGLE = os.path.isdir("/kaggle/input") and not EXTERNAL


def kill_children():
    """Kill everything this solver started (server, its workers, tool executors), by descendant PIDs. The
    name-based pkill of the server is added only on Kaggle: on a shared host it would hit other users'
    processes whose command line mentions vllm/sglang (E021 incident)."""
    for pid in descendants():
        try:
            os.kill(pid, 9)
        except OSError:
            pass
    if ON_KAGGLE:
        subprocess.run("pkill -9 -f '[V]LLM::'; pkill -9 -f '[v]llm.entrypoints'; pkill -9 -f '[s]glang'",
                       shell=True)


def emergency_timer():
    """Last-resort timer (review S1): at T0 + ARC_EMERGENCY_S, make sure a valid submission.json exists, kill the
    server and all children, and exit, whatever the asyncio side is doing."""
    import threading

    def fire():
        log("EMERGENCY timer fired: validating submission.json, killing children, exiting")
        try:
            path = os.path.join(WORK, "submission.json")
            tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
            try:
                check_submission(path, tasks)
            except Exception:
                global OUT_SUB
                OUT_SUB = path
                Submission(tasks).write()
        finally:
            try:
                kill_children()
            finally:
                os._exit(0)
    t = threading.Timer(max(1.0, T0 + envf("ARC_EMERGENCY_S", 11.7 * 3600) - time.time()), fire)
    t.daemon = True
    t.start()


def main():
    global OUT_SUB
    emergency_timer()
    os.makedirs(LOG_DIR, exist_ok=True)
    driver_env()
    global AGENT
    rerun, env_rerun, ids_differ = is_rerun(comp_file("arc-agi_test_challenges.json"))
    mode = "submit" if rerun else MODE
    log(f"rerun={rerun} (env {env_rerun}, test ids differ from public {ids_differ})")
    log(f"mode={mode} server={SERVER} agent={AGENT} rerun={rerun} "
        f"T0={time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(T0))} UTC")
    test_tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
    Submission(test_tasks).write()  # valid fallback for the competition test file, always
    log(f"fallback submission written ({len(test_tasks)} tasks)")
    if mode == "fallback":
        return
    if mode == "local":
        # Non-rerun commit/interactive run: small local test set with known solutions, self-scored, time-boxed.
        if LOCAL_SOURCE == "eval":
            tasks = json.load(open(comp_file("arc-agi_evaluation_challenges.json")))
            solutions = json.load(open(comp_file("arc-agi_evaluation_solutions.json")))
        else:
            tasks = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_challenges.json")))
            solutions = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_solutions.json")))
        if LOCAL_TASKS:
            tasks = {t: tasks[t] for t in LOCAL_TASKS.split(",") if t in tasks}
        force_at = hard_stop = None  # set once the server is ready
        OUT_SUB = os.path.join(WORK, "local_submission.json")
    else:
        tasks, solutions = test_tasks, None
        force_at, hard_stop = T0 + FORCE_AT_S, T0 + HARD_STOP_S
    sub = Submission(tasks)
    sub.write()
    log(f"{mode}: {len(tasks)} tasks, {sum(len(t['test']) for t in tasks.values())} test inputs"
        + (f"; force at {(force_at - T0) / 3600:.2f}h, hard stop {(hard_stop - T0) / 3600:.2f}h" if force_at else ""))
    if AGENT == "tools" and not setup_harness():
        AGENT = "plain"
        log("falling back to the plain (no-tool) agent")
    PHASES["harness_ready"] = round(time.time() - T0)

    if EXTERNAL:
        # off-Kaggle validation against an already running server: no install, no launch, no watchdog restarts
        class _Alive:
            returncode = None

            def poll(self):
                return None
        mdir, proc, slog = None, _Alive(), os.path.join(LOG_DIR, "external_server.log")
        open(slog, "a").close()
        log(f"external server {BASE} (model {SERVED}); skipping install and launch")
    else:
        global TP
        gpus = gpu_inventory()
        TP = choose_tp(gpus)
        PHASES["gpus"] = [f"{n} {m}MiB sm{c}" for n, m, c in gpus]
        log(f"GPUs present: {gpus}; tensor parallel {TP}")
        if TP == 0:
            log("NO USABLE GPUS for Qwen3.8-27B-FP8 (need SM89+ and enough memory): keeping the fallback submission")
            return
        if TP != 4:
            log(f"WARNING: running with TP={TP} instead of 4; KV capacity and concurrency shrink accordingly")
        mdir = model_dir()
        log("model dir:", mdir)
        install_server()
        PHASES["installed"] = round(time.time() - T0)
        preload(mdir)
        PHASES["preloaded"] = round(time.time() - T0)
        proc, slog = launch_server(mdir)
    if not wait_ready(proc, slog, timeout=1800):
        log("server start failed; killing the process tree and retrying once")
        kill_server(proc)
        proc, slog = launch_server(mdir)
        if not wait_ready(proc, slog, timeout=1800):
            raise RuntimeError("server failed to start (twice)")
    PHASES["server_ready"] = round(time.time() - T0)
    if mode == "local":
        if LOCAL_READY_MIN > 0:
            force_at = time.time() + LOCAL_READY_MIN * 60
        else:
            force_at = min(T0 + LOCAL_FORCE_CAP_MIN * 60, max(T0 + LOCAL_FORCE_MIN * 60, time.time() + 20 * 60))
        hard_stop = force_at + LOCAL_HARD_MIN * 60
        log(f"local time box: force at {(force_at - T0) / 60:.0f} min, hard stop {(hard_stop - T0) / 60:.0f} min")
    kv, max_running = server_capacity(slog)
    conc = CONCURRENCY or min(MAX_SEQS, max_running or MAX_SEQS,
                              max(8, int(kv / TOKENS_PER_SEQ_KV)) if kv else 16)
    log(f"concurrency {conc} (max_seqs {MAX_SEQS}, server max running {max_running}, kv {kv})")
    server = {"proc": proc, "down": False, "restarts": 0}

    async def watchdog(force_at):
        """Restart the server (at most twice) if it dies while there is time left to use it."""
        while True:
            await asyncio.sleep(15)
            if server["proc"].poll() is None:
                continue
            server["down"] = True
            log(f"server died (rc={server['proc'].returncode}):\n" + server_log_tail(slog, 40))
            if server["restarts"] >= 2 or force_at - time.time() < 45 * 60:
                log("not restarting the server")
                return
            server["restarts"] += 1
            kill_server(server["proc"])
            server["proc"], _ = launch_server(mdir)
            ok = await asyncio.to_thread(wait_ready, server["proc"], slog, 1800)
            log(f"server restart {'ok' if ok else 'FAILED'}")
            if not ok:
                return
            server["down"] = False

    async def cpu_monitor():
        while True:
            await asyncio.sleep(120)
            try:
                la = os.getloadavg()
                log(f"cpu load avg {la[0]:.1f}/{la[1]:.1f}/{la[2]:.1f} on {os.cpu_count()} cpus")
            except Exception:
                pass

    async def go():
        from concurrent.futures import ThreadPoolExecutor
        # agent_qwen runs executor calls via asyncio.to_thread: one thread per concurrent attempt (E012)
        asyncio.get_running_loop().set_default_executor(ThreadPoolExecutor(max_workers=conc + 8))
        client = Client()
        sch = Scheduler(tasks, sub, client, force_at, hard_stop, conc, attempts=2, tag=mode)
        sch.server_down = lambda: server["down"] or server["proc"].poll() is not None
        wd = asyncio.create_task(watchdog(sch.force_at))
        cm = asyncio.create_task(cpu_monitor())
        await sch.run(order_jobs(tasks))
        wd.cancel()
        cm.cancel()
        sub.write()
        check_submission(OUT_SUB, tasks)
        recs = sch.records
        summary = {"mode": mode, "server": SERVER, "agent": AGENT, "concurrency": conc, "kv_tokens": kv,
                   "phases_s": PHASES, "server_restarts": server["restarts"],
                   "attempts": len(recs), "tokens": sch.tokens,
                   "gen_minutes": round((time.time() - sch.gen_start) / 60, 1),
                   "answered": sum(r["answered"] for r in recs), "errors": sum("error" in r for r in recs),
                   "stream_errors": sum(r.get("stream_errors") or 0 for r in recs),
                   "server_preemption_lines": sum("preempt" in ln.lower() for ln in open(slog, errors="ignore")),
                   "cache_stats": sch.cache_stats[-1] if sch.cache_stats else None,
                   "cache_hit_overall": cache_line(sch.cache_stats[-1], None) if sch.cache_stats else None,
                   "kv_usage_max_pct": max([float(x) for x in re.findall(r"KV cache usage: ([\d.]+)%",
                                                                          open(slog, errors="ignore").read())] or [0]),
                   "tool_calls": sum(r.get("tool_calls") or 0 for r in recs),
                   "tool_errors": sum(r.get("tool_errors") or 0 for r in recs),
                   "parse_failures": sum(r.get("parse_failures") or 0 for r in recs),
                   "answer_sources": {k: sum(r.get("answer_source") == k for r in recs)
                                      for k in ("submit", "forced", "session", "salvage", "parse", "lenient",
                                                "program_pass", "program_best_fail", "session_solve", None)},
                   "v3": {"answer_mode": TOOLS_ANSWER_MODE, "prompt": TOOLS_PROMPT or "default",
                          "task_summary": TASK_SUMMARY, "helpers": HELPERS, "turn_think_cap": TURN_THINK_CAP,
                          "think_call_rescue": THINK_CALL_RESCUE, "tier": TOOLS_TIER, "feedback": FEEDBACK,
                          "think_rescued": sum(r.get("think_call_rescued") or 0 for r in recs),
                          "free_attempt2": FREE_ATTEMPT2, "attempt2": ATTEMPT2, "session_mem_mb": SESSION_MEM_MB,
                          "gated": sch.gated, "attempts_by_index": {str(i): sum(r["attempt"] == i for r in recs)
                                                                    for i in (0, 1)},
                          "think_nudges": sum(r.get("think_nudges") or 0 for r in recs),
                          "think_forced": sum(r.get("think_forced") or 0 for r in recs),
                          "train_pass": sum(r.get("train_pass") is True for r in recs),
                          "with_alternatives": sum(bool(r.get("alternatives")) for r in recs),
                          "slot2_sources": {k: sum(1 for v in sub.sources.values() if (v[1] or "").startswith(k))
                                            for k in ("a0:alt", "a1:alt", "a0:", "a1:", "fallback", "dup")}}}
        if solutions:
            final = json.load(open(OUT_SUB))
            sc, n = score(final, solutions, tasks)
            summary["score"] = f"{sc:.2f}/{n} tasks"
            only1 = {t: [{"attempt_1": s["attempt_1"], "attempt_2": s["attempt_1"]} for s in v] for t, v in final.items()}
            summary["score_attempt1_only"] = f"{score(only1, solutions, tasks)[0]:.2f}/{n} tasks"
            summary["slots"] = [{"task": t, "pair": p, "src": sub.sources.get((t, p)),
                                 "correct1": final[t][p]["attempt_1"] == solutions[t][p],
                                 "correct2": final[t][p]["attempt_2"] == solutions[t][p]}
                                for t in tasks for p in range(len(tasks[t]["test"]))]
            summary["per_pair"] = [{"task": r["task_id"], "pair": r["pair"], "attempt": r["attempt"],
                                    "correct": r["pred"] == solutions[r["task_id"]][r["pair"]],
                                    "tokens": r["tokens"], "turns": r.get("turns"), "tool_calls": r.get("tool_calls"),
                                    "tool_errors": r.get("tool_errors"), "source": r.get("answer_source"),
                                    "stop": r.get("stop_reason"), "s": r["elapsed_s"], "error": r.get("error"),
                                    "train_pass": r.get("train_pass"), "n_alt": len(r.get("alternatives") or []),
                                    "alt_correct": any(a.get("pred") == solutions[r["task_id"]][r["pair"]]
                                                       for a in r.get("alternatives") or [] if isinstance(a, dict))}
                                   for r in recs]
        json.dump(summary, open(os.path.join(WORK, f"summary_{mode}.json"), "w"), indent=1)
        log("SUMMARY", json.dumps(summary))
    asyncio.run(go())
    log("server log tail:\n" + server_log_tail(slog, 10))


if __name__ == "__main__":
    try:
        main()
    except BaseException as exc:  # keep whatever submission.json exists
        import traceback
        traceback.print_exc()
        log(f"FATAL {type(exc).__name__}: {exc}")
    finally:
        try:
            tasks_path = comp_file("arc-agi_test_challenges.json")
            check_submission(os.path.join(WORK, "submission.json"), json.load(open(tasks_path)))
            log("submission.json valid")
        except Exception as exc:
            log(f"submission check failed ({exc}); rewriting fallback")
            OUT_SUB = os.path.join(WORK, "submission.json")
            Submission(json.load(open(comp_file("arc-agi_test_challenges.json")))).write()
        try:
            kill_children()  # executors / server still alive after a hard-stop cancel (descendants only off Kaggle)
        finally:
            os._exit(0)


## 3. Import check

Every written module must import before any GPU work starts.

In [ ]:
import importlib, sys
sys.path[:0] = ['/kaggle/working/arc_solver', '/kaggle/working/arc_solver/evals']
for m in ['arc_harness.grids', 'arc_harness.arcprize', 'arc_harness.prompts', 'arc_harness.executor', 'arc_harness.audit', 'arc_harness.selection', 'arc_helpers', 'task_summary', 'arc_harness.agent_qwen', 'solver']:
    mod = importlib.import_module(m)
    print('ok', m, getattr(mod, 'HARNESS_VERSION', ''))

## 4. Run

The solver runs in its own process group. A guard thread kills that group (server and tool children included) at T0 + 11h45m.

In [ ]:
import os, signal, subprocess, sys, threading, time
T0 = float(os.environ['ARC_T0'])
GUARD_AT = T0 + float(os.environ.get('ARC_GUARD_S', 11.75 * 3600))
p = subprocess.Popen([sys.executable, '-u', '/kaggle/working/arc_solver/solver.py'], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working', start_new_session=True)
def descendants(root):
    kids = {}
    for d in os.listdir('/proc'):
        if d.isdigit():
            try:
                kids.setdefault(int(open(f'/proc/{d}/stat').read().rsplit(')', 1)[1].split()[1]), []).append(int(d))
            except Exception:
                pass
    out, todo = [], [root]
    while todo:
        for c in kids.get(todo.pop(), []):
            out.append(c)
            todo.append(c)
    return out
def guard():
    while p.poll() is None:
        if time.time() > GUARD_AT:
            print('GUARD: wall-clock limit reached; killing the solver and everything it started', flush=True)
            tree = descendants(p.pid)  # server and tool executors run in their own sessions
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except Exception as exc:
                print('GUARD kill failed', exc, flush=True)
            for pid in tree:
                try:
                    os.kill(pid, signal.SIGKILL)
                except Exception:
                    pass
            if os.path.isdir('/kaggle/input'):  # name patterns only on Kaggle, never on a shared host
                subprocess.run("pkill -9 -f 'VLLM::'; pkill -9 -f vllm.entrypoints; pkill -9 -f sglang", shell=True)
            return
        time.sleep(10)
threading.Thread(target=guard, daemon=True).start()
for line in p.stdout:
    print(line, end='', flush=True)
print('solver exit code', p.wait(), 'at', round((time.time() - T0) / 3600, 2), 'h')


## 5. Validate `submission.json` (and gate the commit run)

Repairs any invalid slot with the fallback. On non-rerun runs it fails the run if the local validation did not work.

In [ ]:
import glob, json, os, shutil
def _valid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) and 1 <= len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
test_path = (glob.glob('/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json')
             or glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True))[0]
tasks = json.load(open(test_path))
try:
    sub = json.load(open('/kaggle/working/submission.json'))
except Exception as exc:
    print('submission.json unreadable:', exc)
    sub = {}
out, repaired = {}, 0
for tid, t in tasks.items():
    slots = sub.get(tid) if isinstance(sub.get(tid), list) else []
    lst = []
    for i, pair in enumerate(t['test']):
        s = slots[i] if i < len(slots) and isinstance(slots[i], dict) else {}
        fb = pair['input'] if _valid(pair['input']) else [[0]]
        slot = {}
        for k in ('attempt_1', 'attempt_2'):
            ok = _valid(s.get(k))
            repaired += not ok
            slot[k] = s[k] if ok else fb
        lst.append(slot)
    out[tid] = lst
json.dump(out, open('/kaggle/working/submission.json', 'w'))
print(len(out), 'tasks;', sum(len(v) for v in out.values()), 'test inputs; slots repaired:', repaired)
shutil.rmtree('/kaggle/working/pkg', ignore_errors=True)
for f in glob.glob('/kaggle/working/*.tmp'):
    os.remove(f)
# Commit-run gate (never on the rerun; the solver also detects the rerun by the test ids and then writes
# summary_submit.json): the local validation must have worked, else this cell fails the commit.
if os.environ.get('ARC_MODE') != 'submit' and not os.path.exists('/kaggle/working/summary_submit.json'):
    problems = []
    try:
        s = json.load(open('/kaggle/working/summary_local.json'))
    except Exception:
        s = None
        problems.append('summary_local.json missing')
    if s is not None:
        if not s.get('answered'):
            problems.append(f"no answered attempt ({s.get('answered')}/{s.get('attempts')})")
        if s.get('agent') == 'tools' and not s.get('tool_calls'):
            problems.append('no tool calls')
        if s.get('agent') != os.environ.get('ARC_AGENT', 'tools'):
            problems.append('tool agent self-test failed (plain fallback in use)')
    print('COMMIT GATE:', 'PASS' if not problems else 'FAIL: ' + '; '.join(problems))
    if problems:
        raise SystemExit('commit gate failed: ' + '; '.join(problems))
